# ARC-AGI-2 — symbolic candidate

CPUとNumPyで動作する独立候補です。任意のC++探索バックエンドを同梱し、コンパイラが利用できない場合はPython探索で実行します。インターネットを無効にして実行できます。

各問題の例示入出力だけから規則を探し、テスト入力ごとに2候補を出力します。途中終了に備え、`submission.json`は最初から全問題を含み、問題ごとに更新します。

既定条件は1問8秒・4000状態・ビーム幅32・深さ3・C++時間配分0.6・C++メモリ768MBです。設定セルまたは`ARC2_`環境変数で変更できます。条件を変更した場合は改めて評価してください。

公式採点への提出はこのNotebookには含まれません。Kaggle上の設定ではCPU、Internet off、出力ファイル`submission.json`を使用してください。


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

# The notebook contains every source file. No download or pip install is needed.
# Bound numerical library threads before importing NumPy in worker processes.
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"
RUNTIME_ROOT = Path("/kaggle/working/arc2_runtime") if Path("/kaggle/working").is_dir() else Path.cwd() / "arc2_runtime"
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE_FILES = {'arc2/__init__.py': '"""ARC2 candidate: bounded, example-validated symbolic grid synthesis."""\n\n__version__ = "0.1.0"\n\n', 'arc2/__main__.py': 'from .runner import main\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'arc2/cpp_backend.py': '"""Optional offline C++ candidate generator with isolated, bounded subprocesses.\n\nCall ``prepare_cpp_backend`` once before task workers are launched.  ``solve_cpp``\nnever compiles or downloads anything, and never forwards IDs or test answers.\n"""\n\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport signal\nimport subprocess\nimport tempfile\nimport time\n\n\nVENDOR_DIR = Path(__file__).resolve().parents[1] / "vendor" / "icecuber"\nVENDOR_COMMIT = "9407072659de1270358c2ba34c527785214dd68b"\nSOURCE_NAMES = (\n    "main.cpp", "read.cpp", "core_functions.cpp", "image_functions.cpp",\n    "image_functions2.cpp", "visu.cpp", "normalize.cpp", "runner.cpp",\n    "brute2.cpp", "deduce_op.cpp", "pieces.cpp", "compose2.cpp",\n    "brute_size.cpp", "efficient.cpp",\n)\nBUILD_FLAGS = ("-std=c++17", "-O2")\n\n\ndef _build_identity() -> tuple[str, Path]:\n    digest = hashlib.sha256(b"arc2-inference-cpp-v1\\0")\n    files = [VENDOR_DIR / "src" / name for name in SOURCE_NAMES]\n    files += sorted((VENDOR_DIR / "headers").glob("*.hpp"))\n    if not files or not (VENDOR_DIR / "LICENSE").is_file():\n        raise FileNotFoundError("Optional vendor source or license is missing")\n    for file in files:\n        digest.update(str(file.relative_to(VENDOR_DIR)).encode())\n        digest.update(b"\\0")\n        digest.update(file.read_bytes())\n    digest.update(" ".join(BUILD_FLAGS).encode())\n    value = digest.hexdigest()\n    return value, Path(tempfile.gettempdir()) / "arc2_cpp_cache" / value[:24] / "run"\n\n\ndef _cache_ready(binary: Path, identity: str) -> bool:\n    if not binary.is_file() or not os.access(binary, os.X_OK):\n        return False\n    try:\n        meta = json.loads(binary.with_name("build.json").read_text())\n        return meta.get("source_sha256") == identity and binary.stat().st_size > 0\n    except (OSError, ValueError):\n        return False\n\n\n@contextmanager\ndef _build_lock(path: Path, deadline: float):\n    import fcntl\n\n    with path.open("a") as handle:\n        while True:\n            try:\n                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n                break\n            except BlockingIOError:\n                if time.monotonic() >= deadline:\n                    raise TimeoutError("Another process is compiling the optional backend")\n                time.sleep(min(0.05, max(0.001, deadline - time.monotonic())))\n        try:\n            yield\n        finally:\n            fcntl.flock(handle, fcntl.LOCK_UN)\n\n\ndef prepare_cpp_backend(compile_seconds: float = 180.0) -> dict:\n    """Compile vendored source once; return ready/unavailable/error diagnostics.\n\n    A zero budget only checks the cache.  Successful preparation sets\n    ``ARC2_CPP_BINARY`` for spawned workers.  No binary is written into the repo.\n    """\n    started = time.monotonic()\n    base = {"vendor_commit": VENDOR_COMMIT}\n    if os.name != "posix":\n        return {**base, "status": "unavailable", "reason": "POSIX resource limits required"}\n    try:\n        identity, binary = _build_identity()\n    except OSError as exc:\n        return {**base, "status": "unavailable", "reason": str(exc)}\n    base["source_sha256"] = identity\n    if _cache_ready(binary, identity):\n        os.environ["ARC2_CPP_BINARY"] = str(binary)\n        return {**base, "status": "ready", "binary": str(binary), "cached": True}\n    if compile_seconds <= 0:\n        return {**base, "status": "not_prepared", "reason": "Call prepare_cpp_backend before task workers"}\n    compiler = shutil.which("g++")\n    if not compiler:\n        return {**base, "status": "unavailable", "reason": "g++ is not installed"}\n    deadline = started + compile_seconds\n    binary.parent.mkdir(parents=True, exist_ok=True)\n    temporary = binary.with_name(f"run.{os.getpid()}.building")\n    try:\n        with _build_lock(binary.with_name("build.lock"), deadline):\n            if _cache_ready(binary, identity):\n                os.environ["ARC2_CPP_BINARY"] = str(binary)\n                return {**base, "status": "ready", "binary": str(binary), "cached": True}\n            command = [compiler, *BUILD_FLAGS, "-I", str(VENDOR_DIR / "headers")]\n            command += [str(VENDOR_DIR / "src" / name) for name in SOURCE_NAMES]\n            command += ["-lstdc++fs", "-o", str(temporary)]\n            log = binary.with_name("build.log")\n            with log.open("wb") as output:\n                proc = subprocess.Popen(command, stdout=output, stderr=subprocess.STDOUT, start_new_session=True)\n                try:\n                    code = proc.wait(timeout=max(0.01, deadline - time.monotonic()))\n                except subprocess.TimeoutExpired:\n                    os.killpg(proc.pid, signal.SIGKILL)\n                    proc.wait()\n                    raise TimeoutError("Optional C++ compilation exceeded its preparation budget")\n            if code != 0 or not temporary.is_file() or temporary.stat().st_size == 0:\n                message = log.read_text(errors="replace")[-5000:]\n                return {**base, "status": "error", "reason": "Compilation failed", "log_tail": message}\n            temporary.replace(binary)\n            meta = {**base, "compiler": compiler, "flags": list(BUILD_FLAGS), "compile_seconds": time.monotonic() - started}\n            binary.with_name("build.json").write_text(json.dumps(meta, indent=2) + "\\n")\n            os.environ["ARC2_CPP_BINARY"] = str(binary)\n            return {**base, "status": "ready", "binary": str(binary), "cached": False, "compile_seconds": meta["compile_seconds"]}\n    except (OSError, TimeoutError) as exc:\n        return {**base, "status": "error", "reason": str(exc)}\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef _grid(value) -> list[list[int]]:\n    if hasattr(value, "tolist"):\n        value = value.tolist()\n    if not isinstance(value, list) or not 1 <= len(value) <= 30:\n        raise ValueError("Grid must contain 1..30 rows")\n    width = len(value[0]) if isinstance(value[0], list) else 0\n    if not 1 <= width <= 30:\n        raise ValueError("Grid must contain 1..30 columns")\n    result = []\n    for row in value:\n        if not isinstance(row, list) or len(row) != width:\n            raise ValueError("Grid must be rectangular")\n        if any(type(c) is not int or not 0 <= c <= 9 for c in row):\n            raise ValueError("Grid colors must be integers in 0..9")\n        result.append(row.copy())\n    return result\n\n\ndef _sanitized_task(task: dict) -> dict:\n    # This allowlist is the only route from the caller\'s task into the backend.\n    # In particular, do not copy whole test dictionaries or inspect their output.\n    train = [{"input": _grid(p["input"]), "output": _grid(p["output"])} for p in task["train"]]\n    test = [{"input": _grid(p["input"])} for p in task["test"]]\n    if not train or not test:\n        raise ValueError("At least one demonstration and test input are required")\n    return {"train": train, "test": test}\n\n\ndef _parse_candidates(path: Path, expected_index: int, training_count: int) -> list[tuple[list, float, int]]:\n    if not path.is_file():\n        return []\n    lines = path.read_text(errors="replace").splitlines()\n    if not lines or lines[0] != f"00000000_{expected_index}":\n        return []\n    result = []\n    for line in lines[1:]:\n        try:\n            encoded, rawscore, match_count = line.split()\n            if not encoded.startswith("|") or not encoded.endswith("|"):\n                continue\n            rows = encoded[1:-1].split("|")\n            if any(not row or not row.isascii() or not row.isdigit() for row in rows):\n                continue\n            grid = _grid([[int(c) for c in row] for row in rows])\n            score, matches = float(rawscore), int(match_count)\n            if not math.isfinite(score) or not 0 <= matches <= training_count:\n                continue\n            result.append((grid, score, matches))\n        except (ValueError, TypeError):\n            continue\n    return result\n\n\ndef _run_limited(binary: str, cwd: Path, index: int, depth: int, seconds: float, memory_mb: int) -> dict:\n    import resource\n\n    def limits():\n        address = int(memory_mb * 1024 * 1024)\n        resource.setrlimit(resource.RLIMIT_AS, (address, address))\n        cpu = max(1, int(math.ceil(seconds)))\n        resource.setrlimit(resource.RLIMIT_CPU, (cpu, cpu + 1))\n        resource.setrlimit(resource.RLIMIT_CORE, (0, 0))\n        resource.setrlimit(resource.RLIMIT_FSIZE, (16 * 1024 * 1024, 16 * 1024 * 1024))\n\n    started = time.monotonic()\n    log = cwd / f"stderr_{index}_{depth}.log"\n    event = {"test_index": index, "depth": depth, "budget_seconds": seconds}\n    try:\n        with log.open("wb") as err:\n            # No new session: the outer task-worker process group owns this\n            # process too. The C++ program itself does not spawn descendants.\n            proc = subprocess.Popen([binary, str(index), str(depth)], cwd=cwd, stdout=subprocess.DEVNULL,\n                                    stderr=err, preexec_fn=limits)\n            try:\n                code = proc.wait(timeout=max(0.001, seconds))\n                event.update(status="complete" if code == 0 else "failed", returncode=code)\n            except subprocess.TimeoutExpired:\n                proc.kill()\n                proc.wait()\n                event.update(status="timeout", returncode=proc.returncode)\n        if event["status"] == "failed":\n            event["stderr_tail"] = log.read_text(errors="replace")[-1500:]\n    except (OSError, subprocess.SubprocessError) as exc:\n        event.update(status="failed", reason=str(exc))\n    event["seconds"] = time.monotonic() - started\n    return event\n\n\ndef solve_cpp(task: dict, seconds: float, max_memory_mb: int = 768) -> dict:\n    """Generate ranked candidates without compiling, networking, or test labels.\n\n    Returns parallel arrays ``predictions``, ``scores``, ``train_matches`` and\n    ``training_exact``, each indexed by test input then candidate.  An empty\n    candidate list requests fallback to the caller\'s symbolic solver.\n    """\n    started = time.monotonic()\n    stop = started + max(0.0, float(seconds))\n    ntests = len(task.get("test", [])) if isinstance(task, dict) else 0\n    result = {"predictions": [[] for _ in range(ntests)], "scores": [[] for _ in range(ntests)],\n              "train_matches": [[] for _ in range(ntests)], "training_exact": [[] for _ in range(ntests)],\n              "diagnostics": {"vendor_commit": VENDOR_COMMIT, "runs": [], "memory_limit_mb": max_memory_mb}}\n    diag = result["diagnostics"]\n    if seconds <= 0:\n        diag.update(status="budget_exhausted", seconds=0.0)\n        return result\n    if max_memory_mb <= 0:\n        diag.update(status="invalid_limit", seconds=0.0)\n        return result\n    prepared = prepare_cpp_backend(compile_seconds=0)\n    diag["source_sha256"] = prepared.get("source_sha256")\n    if prepared["status"] != "ready":\n        diag.update(status=prepared["status"], reason=prepared.get("reason"), seconds=time.monotonic() - started)\n        return result\n    try:\n        safe_task = _sanitized_task(task)\n    except (ValueError, TypeError, KeyError) as exc:\n        diag.update(status="invalid_task", reason=str(exc), seconds=time.monotonic() - started)\n        return result\n    ntrain = len(safe_task["train"])\n    pools: list[dict] = [{} for _ in range(ntests)]\n    with tempfile.TemporaryDirectory(prefix="arc2_icecuber_") as temporary:\n        folder = Path(temporary)\n        (folder / "dataset" / "test").mkdir(parents=True)\n        (folder / "output").mkdir()\n        (folder / "dataset" / "test" / "00000000.json").write_text(json.dumps(safe_task, separators=(",", ":")))\n        for depth in (1, 2, 3):\n            for index in range(ntests):\n                remaining = stop - time.monotonic()\n                if remaining <= 0.02:\n                    break\n                cap = 1.5 if depth == 1 else 5.0 if depth == 2 else math.inf\n                budget = min(cap, remaining / (ntests - index))\n                event = _run_limited(prepared["binary"], folder, index, depth, budget, max_memory_mb)\n                diag["runs"].append(event)\n                # Salvage complete rows even if the process failed after writing\n                # them; earlier completed depth files remain independent.\n                for grid, score, matches in _parse_candidates(folder / "output" / f"answer_{index}_{depth}.csv", index, ntrain):\n                    key = tuple(tuple(row) for row in grid)\n                    current = pools[index].get(key)\n                    if current is None or (matches, score) > (current[2], current[1]):\n                        pools[index][key] = (grid, score, matches)\n            if time.monotonic() >= stop:\n                break\n    for i, pool in enumerate(pools):\n        candidates = sorted(pool.values(), key=lambda p: (-p[2], -p[1], p[0]))\n        result["predictions"][i] = [p[0] for p in candidates]\n        result["scores"][i] = [p[1] for p in candidates]\n        result["train_matches"][i] = [p[2] for p in candidates]\n        result["training_exact"][i] = [p[2] == ntrain for p in candidates]\n    diag.update(status="complete" if all(result["predictions"]) else "partial",\n                seconds=time.monotonic() - started, limits_enforced="RLIMIT_AS, RLIMIT_CPU, RLIMIT_FSIZE, wall-time",\n                completed_depths=sum(e["status"] == "complete" for e in diag["runs"]))\n    return result\n', 'arc2/grid.py': 'from __future__ import annotations\n\nfrom collections import deque\n\nimport numpy as np\n\n\ndef valid_grid(value) -> bool:\n    try:\n        a = np.asarray(value)\n        return bool(\n            a.ndim == 2 and 1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30\n            and a.dtype.kind in "iu" and np.all((a >= 0) & (a <= 9))\n        )\n    except (ValueError, TypeError):\n        return False\n\n\ndef as_grid(value) -> np.ndarray:\n    if not valid_grid(value):\n        raise ValueError("Grid must be a rectangular 1..30 by 1..30 array of integers 0..9")\n    return np.array(value, dtype=np.uint8, copy=True)\n\n\ndef key(grid: np.ndarray) -> bytes:\n    a = np.asarray(grid, dtype=np.uint8)\n    return bytes(a.shape) + a.tobytes(order="C")\n\n\ndef background(grid: np.ndarray) -> int:\n    counts = np.bincount(np.asarray(grid).ravel(), minlength=10)\n    return int(np.argmax(counts))\n\n\ndef crop(grid: np.ndarray, bg: int = 0) -> np.ndarray | None:\n    rows, cols = np.where(grid != bg)\n    if len(rows) == 0:\n        return None\n    return grid[rows.min():rows.max() + 1, cols.min():cols.max() + 1].copy()\n\n\ndef components(grid: np.ndarray, bg: int = 0, diagonal: bool = False,\n               monochrome: bool = True) -> list[dict]:\n    """Connected foreground components with exclusive-end bounding boxes."""\n    h, w = grid.shape\n    seen = np.zeros((h, w), dtype=bool)\n    directions = ((1, 0), (-1, 0), (0, 1), (0, -1))\n    if diagonal:\n        directions += ((1, 1), (1, -1), (-1, 1), (-1, -1))\n    objects = []\n    for r, c in np.argwhere(grid != bg):\n        r, c = int(r), int(c)\n        if seen[r, c]:\n            continue\n        color = int(grid[r, c])\n        todo = deque([(r, c)])\n        seen[r, c] = True\n        cells = []\n        while todo:\n            y, x = todo.popleft()\n            cells.append((y, x))\n            for dy, dx in directions:\n                yy, xx = y + dy, x + dx\n                if 0 <= yy < h and 0 <= xx < w and not seen[yy, xx]:\n                    if grid[yy, xx] != bg and (not monochrome or grid[yy, xx] == color):\n                        seen[yy, xx] = True\n                        todo.append((yy, xx))\n        ys, xs = zip(*cells)\n        if not monochrome:\n            color = int(np.bincount([int(grid[y, x]) for y, x in cells], minlength=10).argmax())\n        objects.append({"cells": cells, "color": color, "size": len(cells),\n                        "bbox": (min(ys), min(xs), max(ys) + 1, max(xs) + 1)})\n    return objects\n', 'arc2/objects.py': '"""Bounded, interpretable object transforms for grid-to-grid induction.\n\nThe factory observes training examples only.  None means that a rule\'s structural\npreconditions fail; it never means an all-zero prediction.  Learned recoloring\nrules are retained only if they reproduce every training output exactly.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter, deque\nfrom functools import lru_cache\nimport time\nfrom typing import Callable\n\nimport numpy as np\n\nfrom .grid import background, components\nfrom .types import Rule\n\n\ndef _expired(deadline: float | None) -> bool:\n    return deadline is not None and time.monotonic() >= deadline\n\n\ndef _mode_background(grid: np.ndarray) -> int:\n    return int(background(grid))\n\n\ndef _zero_background(grid: np.ndarray) -> int:\n    return 0\n\n\n@lru_cache(maxsize=192)\ndef _cached_components(shape: tuple[int, int], data: bytes, bg: int,\n                       diagonal: bool, monochrome: bool) -> tuple[dict, ...]:\n    grid = np.frombuffer(data, dtype=np.int8).reshape(shape)\n    # The shared grid implementation owns connectivity conventions.  Copy the\n    # returned dictionaries so callers cannot mutate a shared grid cache.\n    return tuple(dict(o) for o in components(\n        grid, bg=bg, diagonal=diagonal, monochrome=monochrome))\n\n\ndef _objects(grid: np.ndarray, bg: int, diagonal: bool = False,\n             monochrome: bool = True) -> tuple[dict, ...]:\n    arr = np.asarray(grid, dtype=np.int8)\n    return _cached_components(arr.shape, arr.tobytes(), int(bg),\n                              diagonal, monochrome)\n\n\ndef _shape(obj: dict, dihedral: bool = False) -> tuple:\n    r0, c0, r1, c1 = obj[\'bbox\']\n    mask = np.zeros((r1-r0, c1-c0), dtype=np.uint8)\n    for r, c in obj[\'cells\']:\n        mask[r-r0, c-c0] = 1\n    return _canonical_shape(mask.shape, mask.tobytes(), dihedral)\n\n\n@lru_cache(maxsize=4096)\ndef _canonical_shape(shape: tuple[int, int], data: bytes, dihedral: bool) -> tuple:\n    if not dihedral:\n        return (shape, data)\n    mask = np.frombuffer(data, dtype=np.uint8).reshape(shape)\n    variants = []\n    for k in range(4):\n        a = np.rot90(mask, k)\n        variants.append((a.shape, a.tobytes()))\n        b = np.fliplr(a)\n        variants.append((b.shape, b.tobytes()))\n    return min(variants)\n\n\ndef _holes(obj: dict) -> int:\n    """Number of bounded 4-connected regions in an object\'s binary mask."""\n    r0, c0, r1, c1 = obj[\'bbox\']\n    cells = tuple(sorted((r-r0+1, c-c0+1) for r, c in obj[\'cells\']))\n    return _hole_count(r1-r0+2, c1-c0+2, cells)\n\n\n@lru_cache(maxsize=2048)\ndef _hole_count(h: int, w: int, cell_tuple: tuple[tuple[int, int], ...]) -> int:\n    cells = set(cell_tuple)\n    seen: set[tuple[int, int]] = set()\n    bounded = 0\n    for r in range(h):\n        for c in range(w):\n            if (r, c) in cells or (r, c) in seen:\n                continue\n            q = [(r, c)]\n            seen.add((r, c))\n            outside = False\n            while q:\n                u, v = q.pop()\n                outside |= u in (0, h-1) or v in (0, w-1)\n                for a, b in ((u-1, v), (u+1, v), (u, v-1), (u, v+1)):\n                    if (0 <= a < h and 0 <= b < w and (a, b) not in cells\n                            and (a, b) not in seen):\n                        seen.add((a, b))\n                        q.append((a, b))\n            bounded += not outside\n    return bounded\n\n\ndef _crop_objects(grid: np.ndarray, objs: list[dict] | tuple[dict, ...],\n                  bg: int, masked: bool = True) -> np.ndarray | None:\n    if not objs:\n        return None\n    r0 = min(o[\'bbox\'][0] for o in objs)\n    c0 = min(o[\'bbox\'][1] for o in objs)\n    r1 = max(o[\'bbox\'][2] for o in objs)\n    c1 = max(o[\'bbox\'][3] for o in objs)\n    if not masked:\n        return grid[r0:r1, c0:c1].copy()\n    out = np.full((r1-r0, c1-c0), bg, dtype=grid.dtype)\n    for obj in objs:\n        for r, c in obj[\'cells\']:\n            out[r-r0, c-c0] = grid[r, c]\n    return out\n\n\ndef _select(grid: np.ndarray, objs: tuple[dict, ...], criterion: str) -> list[dict]:\n    if not objs:\n        return []\n    if criterion in (\'unique_shape\', \'repeated_shape\'):\n        counts = Counter(_shape(o, True) for o in objs)\n        if criterion == \'unique_shape\':\n            chosen = [o for o in objs if counts[_shape(o, True)] == 1]\n            return chosen if len(chosen) == 1 else []\n        repeated = {shape for shape, count in counts.items() if count > 1}\n        # A single repeated equivalence class is an unambiguous object group;\n        # crop its union, preserving relative positions and background gaps.\n        return ([o for o in objs if _shape(o, True) in repeated]\n                if len(repeated) == 1 else [])\n    values: list[int] = []\n    for o in objs:\n        r0, c0, r1, c1 = o[\'bbox\']\n        if criterion in (\'largest\', \'smallest\'):\n            value = o[\'size\']\n        elif criterion in (\'largest_box\', \'smallest_box\'):\n            value = (r1-r0)*(c1-c0)\n        elif criterion == \'widest\':\n            value = c1-c0\n        elif criterion == \'tallest\':\n            value = r1-r0\n        elif criterion == \'most_colors\':\n            value = len({int(grid[r, c]) for r, c in o[\'cells\']})\n        elif criterion == \'most_holes\':\n            value = _holes(o)\n        else:\n            raise ValueError(criterion)\n        values.append(value)\n    target = min(values) if criterion in (\'smallest\', \'smallest_box\') else max(values)\n    selected = [o for o, v in zip(objs, values) if v == target]\n    return selected if len(selected) == 1 else []\n\n\ndef _selection(grid: np.ndarray, bg_fn: Callable, diagonal: bool,\n               monochrome: bool, criterion: str, masked: bool = True) -> np.ndarray | None:\n    bg = bg_fn(grid)\n    objs = _objects(grid, bg, diagonal, monochrome)\n    return _crop_objects(grid, _select(grid, objs, criterion), bg, masked)\n\n\ndef _filter(grid: np.ndarray, bg_fn: Callable, diagonal: bool,\n            monochrome: bool, criterion: str) -> np.ndarray | None:\n    bg = bg_fn(grid)\n    objs = _objects(grid, bg, diagonal, monochrome)\n    if not objs:\n        return None\n    sizes = Counter(o[\'size\'] for o in objs)\n    shapes = (Counter(_shape(o, True) for o in objs)\n              if criterion in (\'repeated_shape\', \'unique_shape\') else None)\n    h, w = grid.shape\n    largest = max(o[\'size\'] for o in objs)\n    smallest = min(o[\'size\'] for o in objs)\n    chosen = []\n    for o in objs:\n        r0, c0, r1, c1 = o[\'bbox\']\n        border = r0 == 0 or c0 == 0 or r1 == h or c1 == w\n        if criterion == \'largest\':\n            keep = o[\'size\'] == largest\n        elif criterion == \'smallest\':\n            keep = o[\'size\'] == smallest\n        elif criterion == \'non_singletons\':\n            keep = o[\'size\'] > 1\n        elif criterion == \'unique_size\':\n            keep = sizes[o[\'size\']] == 1\n        elif criterion == \'repeated_shape\':\n            keep = shapes[_shape(o, True)] > 1\n        elif criterion == \'unique_shape\':\n            keep = shapes[_shape(o, True)] == 1\n        elif criterion == \'border\':\n            keep = border\n        elif criterion == \'interior\':\n            keep = not border\n        else:\n            raise ValueError(criterion)\n        if keep:\n            chosen.append(o)\n    if not chosen:\n        return None\n    out = np.full_like(grid, bg)\n    for obj in chosen:\n        for r, c in obj[\'cells\']:\n            out[r, c] = grid[r, c]\n    return out\n\n\ndef _paint(grid: np.ndarray, bg_fn: Callable, diagonal: bool, mode: str) -> np.ndarray | None:\n    bg = bg_fn(grid)\n    objs = _objects(grid, bg, diagonal, True)\n    out = grid.copy()\n    if not objs:\n        return None\n    for o in objs:\n        r0, c0, r1, c1 = o[\'bbox\']\n        color = int(o[\'color\'])\n        cells = set(o[\'cells\'])\n        if mode == \'outline\':\n            for r, c in cells:\n                if all(p in cells for p in ((r-1, c), (r+1, c), (r, c-1), (r, c+1))):\n                    out[r, c] = bg\n        elif mode in (\'rectangle\', \'box_outline\'):\n            for r in range(r0, r1):\n                for c in range(c0, c1):\n                    if mode == \'box_outline\' and r not in (r0, r1-1) and c not in (c0, c1-1):\n                        continue\n                    if grid[r, c] == bg:\n                        # Conflicting paints are an ambiguous transformation.\n                        if out[r, c] not in (bg, color):\n                            return None\n                        out[r, c] = color\n        elif mode == \'fill_holes\':\n            # The object\'s own pixels form the boundary. Other colors are\n            # preserved, including markers inside the bounded region.\n            outside = set()\n            q = deque()\n            for r in range(r0-1, r1+1):\n                for c in (c0-1, c1):\n                    outside.add((r, c)); q.append((r, c))\n            for c in range(c0-1, c1+1):\n                for r in (r0-1, r1):\n                    if (r, c) not in outside:\n                        outside.add((r, c)); q.append((r, c))\n            while q:\n                r, c = q.popleft()\n                for p in ((r-1, c), (r+1, c), (r, c-1), (r, c+1)):\n                    if (r0-1 <= p[0] <= r1 and c0-1 <= p[1] <= c1\n                            and p not in cells and p not in outside):\n                        outside.add(p); q.append(p)\n            for r in range(r0, r1):\n                for c in range(c0, c1):\n                    if (r, c) not in outside and grid[r, c] == bg:\n                        if out[r, c] not in (bg, color):\n                            return None\n                        out[r, c] = color\n        else:\n            raise ValueError(mode)\n    return out\n\n\ndef _pixel_gravity(grid: np.ndarray, bg_fn: Callable, direction: str) -> np.ndarray:\n    bg = bg_fn(grid)\n    out = np.full_like(grid, bg)\n    if direction in (\'down\', \'up\'):\n        for c in range(grid.shape[1]):\n            values = grid[:, c][grid[:, c] != bg]\n            start = grid.shape[0]-len(values) if direction == \'down\' else 0\n            out[start:start+len(values), c] = values\n    else:\n        for r in range(grid.shape[0]):\n            values = grid[r][grid[r] != bg]\n            start = grid.shape[1]-len(values) if direction == \'right\' else 0\n            out[r, start:start+len(values)] = values\n    return out\n\n\ndef _object_gravity(grid: np.ndarray, bg_fn: Callable,\n                    direction: str, monochrome: bool) -> np.ndarray:\n    bg = bg_fn(grid)\n    objs = _objects(grid, bg, False, monochrome)\n    dr, dc = {\'down\': (1, 0), \'up\': (-1, 0), \'right\': (0, 1), \'left\': (0, -1)}[direction]\n    blocks = [[(r, c, int(grid[r, c])) for r, c in o[\'cells\']] for o in objs]\n    blocks.sort(key=lambda cells: max(dr*r+dc*c for r, c, _ in cells), reverse=True)\n    out = grid.copy()\n    h, w = grid.shape\n    # Repeated sweeps handle interlocking outlines deterministically. Each\n    # successful sweep strictly advances at least one finite object.\n    for _ in range(h+w):\n        moved = False\n        for i, cells in enumerate(blocks):\n            own = {(r, c) for r, c, _ in cells}\n            distance = 0\n            while True:\n                d = distance+1\n                if any(not (0 <= r+dr*d < h and 0 <= c+dc*d < w)\n                       or ((r+dr*d, c+dc*d) not in own and out[r+dr*d, c+dc*d] != bg)\n                       for r, c, _ in cells):\n                    break\n                distance = d\n            if distance:\n                for r, c, _ in cells:\n                    out[r, c] = bg\n                cells = [(r+dr*distance, c+dc*distance, color) for r, c, color in cells]\n                for r, c, color in cells:\n                    out[r, c] = color\n                blocks[i] = cells\n                moved = True\n        if not moved:\n            break\n    return out\n\n\ndef _stamp_template(grid: np.ndarray, bg_fn: Callable, diagonal: bool) -> np.ndarray | None:\n    """Copy a single multicolor template to isolated, uniquely colored anchors."""\n    bg = bg_fn(grid)\n    objs = _objects(grid, bg, diagonal, False)\n    templates = [o for o in objs if o[\'size\'] > 1]\n    markers = [o for o in objs if o[\'size\'] == 1]\n    if len(templates) != 1 or not markers:\n        return None\n    template = templates[0]\n    anchors: dict[int, list[tuple[int, int]]] = {}\n    for r, c in template[\'cells\']:\n        anchors.setdefault(int(grid[r, c]), []).append((r, c))\n    if len(anchors) < 2:\n        return None\n    out = grid.copy()\n    h, w = grid.shape\n    for marker in markers:\n        mr, mc = marker[\'cells\'][0]\n        points = anchors.get(int(grid[mr, mc]), [])\n        if len(points) != 1:\n            return None\n        ar, ac = points[0]\n        for r, c in template[\'cells\']:\n            nr, nc = r+mr-ar, c+mc-ac\n            if not (0 <= nr < h and 0 <= nc < w):\n                return None\n            color = int(grid[r, c])\n            if out[nr, nc] not in (bg, color):\n                return None\n            out[nr, nc] = color\n    return out\n\n\ndef _features(grid: np.ndarray, bg: int, diagonal: bool) -> tuple[dict, ...]:\n    a = np.asarray(grid, dtype=np.int8)\n    return _cached_features(a.shape, a.tobytes(), bg, diagonal)\n\n\n@lru_cache(maxsize=192)\ndef _cached_features(shape: tuple[int, int], data: bytes, bg: int,\n                     diagonal: bool) -> tuple[dict, ...]:\n    grid = np.frombuffer(data, dtype=np.int8).reshape(shape)\n    objs = _objects(grid, bg, diagonal, True)\n    sizes = sorted({o[\'size\'] for o in objs})\n    widths = sorted({o[\'bbox\'][3]-o[\'bbox\'][1] for o in objs})\n    heights = sorted({o[\'bbox\'][2]-o[\'bbox\'][0] for o in objs})\n    shapes = [_shape(o, True) for o in objs]\n    shape_counts = Counter(shapes)\n    size_counts = Counter(o[\'size\'] for o in objs)\n    top_values = sorted({o[\'bbox\'][0] for o in objs})\n    left_values = sorted({o[\'bbox\'][1] for o in objs})\n    h, w = grid.shape\n    result = []\n    for obj, shape in zip(objs, shapes):\n        r0, c0, r1, c1 = obj[\'bbox\']\n        cell_set = set(obj[\'cells\'])\n        adjacent = set()\n        for r, c in obj[\'cells\']:\n            for a, b in ((r-1, c), (r+1, c), (r, c-1), (r, c+1)):\n                if (0 <= a < h and 0 <= b < w and (a, b) not in cell_set\n                        and grid[a, b] != bg):\n                    adjacent.add(int(grid[a, b]))\n        result.append({\n            \'color\': int(obj[\'color\']), \'area\': obj[\'size\'],\n            \'width\': c1-c0, \'height\': r1-r0,\n            \'bbox_area\': (r1-r0)*(c1-c0),\n            \'solid\': obj[\'size\'] == (r1-r0)*(c1-c0),\n            \'area_rank\': sizes.index(obj[\'size\']),\n            \'area_rank_desc\': len(sizes)-1-sizes.index(obj[\'size\']),\n            \'width_rank\': widths.index(c1-c0),\n            \'height_rank\': heights.index(r1-r0),\n            \'top_rank\': top_values.index(r0), \'left_rank\': left_values.index(c0),\n            \'shape\': shape, \'shape_count\': shape_counts[shape],\n            \'size_count\': size_counts[obj[\'size\']],\n            \'holes\': _holes(obj),\n            \'border\': r0 == 0 or c0 == 0 or r1 == h or c1 == w,\n            \'adjacent_colors\': tuple(sorted(adjacent)),\n            \'adjacent_count\': len(adjacent),\n        })\n    return tuple(result)\n\n\ndef _fit_recolor(train: list[dict], bg_fn: Callable, diagonal: bool,\n                  name_prefix: str, deadline: float | None) -> list[Rule]:\n    observations = []\n    changed = False\n    for example in train:\n        x, y = np.asarray(example[\'input\']), np.asarray(example[\'output\'])\n        if x.shape != y.shape:\n            return []\n        bg = bg_fn(x)\n        if np.any(y[x == bg] != bg):\n            return []\n        objs = _objects(x, bg, diagonal, True)\n        features = _features(x, bg, diagonal)\n        for obj, values in zip(objs, features):\n            colors = {int(y[r, c]) for r, c in obj[\'cells\']}\n            if len(colors) != 1:\n                return []\n            target = next(iter(colors))\n            changed |= target != int(obj[\'color\'])\n            observations.append((values, target))\n    if not changed or not observations:\n        return []\n    names = tuple(observations[0][0])\n    feature_sets = [(name,) for name in names]\n    feature_sets += [(\'color\', name) for name in names if name != \'color\']\n    feature_sets += [(\'border\', \'solid\'), (\'area_rank\', \'shape_count\'),\n                     (\'holes\', \'border\'), (\'width_rank\', \'height_rank\')]\n    rules = []\n    for fields in feature_sets:\n        if _expired(deadline):\n            break\n        mapping = {}\n        conflict = False\n        for values, target in observations:\n            key = tuple(values[n] for n in fields)\n            if key in mapping and mapping[key] != target:\n                conflict = True\n                break\n            mapping[key] = target\n        if conflict:\n            continue\n\n        def repaint(grid, fields=fields, mapping=dict(mapping), bg_fn=bg_fn,\n                    diagonal=diagonal):\n            bg = bg_fn(grid)\n            objs = _objects(grid, bg, diagonal, True)\n            feats = _features(grid, bg, diagonal)\n            out = grid.copy()\n            for obj, values in zip(objs, feats):\n                target = mapping.get(tuple(values[n] for n in fields), int(obj[\'color\']))\n                for r, c in obj[\'cells\']:\n                    out[r, c] = target\n            return out\n\n        # The fitter never sees held-out test outputs. Validate its own entire\n        # training transformation, including cells outside the chosen objects.\n        if all(np.array_equal(repaint(np.asarray(e[\'input\'])), np.asarray(e[\'output\']))\n               for e in train):\n            cost = 1.7+0.2*len(fields)+0.12*len(mapping)\n            if \'shape\' in fields:\n                cost += 1.0  # literal shape tables are less general than ranks\n            rules.append(Rule(f\'{name_prefix}:recolor:{"+".join(fields)}\', repaint,\n                              cost=cost, family=\'object_recolor\'))\n    return rules\n\n\ndef object_rules(train: list[dict], deadline: float | None = None) -> list[Rule]:\n    """Return a bounded family of pure object transformations.\n\n    Static operations are candidates for composition, so they need not themselves\n    fit the examples. Object recoloring rules, in contrast, are fitted from all\n    examples and validated here. A missed deadline returns the completed prefix.\n    """\n    if not train or _expired(deadline):\n        return []\n    bg_options = [(\'zero\', _zero_background), (\'mode\', _mode_background)]\n    rules: list[Rule] = []\n    for bg_name, bg_fn in bg_options:\n        if _expired(deadline):\n            break\n        for direction in (\'down\', \'up\', \'left\', \'right\'):\n            rules.append(Rule(\n                f\'objects:{bg_name}:pixel_gravity:{direction}\',\n                lambda g, bf=bg_fn, d=direction: _pixel_gravity(g, bf, d),\n                cost=1.3, family=\'gravity\'))\n            for mono in (True, False):\n                rules.append(Rule(\n                    f\'objects:{bg_name}:object_gravity:{direction}:mono{int(mono)}\',\n                    lambda g, bf=bg_fn, d=direction, m=mono: _object_gravity(g, bf, d, m),\n                    cost=1.7, family=\'gravity\'))\n        for diagonal in (False, True):\n            if _expired(deadline):\n                break\n            prefix = f\'objects:{bg_name}:{8 if diagonal else 4}\'\n            for mode in (\'rectangle\', \'box_outline\', \'outline\', \'fill_holes\'):\n                rules.append(Rule(\n                    f\'{prefix}:{mode}\',\n                    lambda g, bf=bg_fn, d=diagonal, m=mode: _paint(g, bf, d, m),\n                    cost=1.5, family=\'object_paint\'))\n            rules.append(Rule(\n                f\'{prefix}:stamp_template\',\n                lambda g, bf=bg_fn, d=diagonal: _stamp_template(g, bf, d),\n                cost=2.0, family=\'object_copy\'))\n            rules.extend(_fit_recolor(train, bg_fn, diagonal, prefix, deadline))\n            for monochrome in (True, False):\n                if _expired(deadline):\n                    break\n                label = f\'{prefix}:mono{int(monochrome)}\'\n                criteria = [\'largest\', \'smallest\', \'largest_box\', \'smallest_box\',\n                            \'widest\', \'tallest\', \'unique_shape\', \'repeated_shape\']\n                if monochrome:\n                    criteria.append(\'most_holes\')\n                else:\n                    criteria.append(\'most_colors\')\n                for criterion in criteria:\n                    rules.append(Rule(\n                        f\'{label}:crop:{criterion}\',\n                        lambda g, bf=bg_fn, d=diagonal, m=monochrome, c=criterion:\n                            _selection(g, bf, d, m, c),\n                        cost=1.4, family=\'object_select\'))\n                # Bounding-box extraction keeps all enclosed colored cells;\n                # masked extraction above keeps only the selected component.\n                rules.append(Rule(\n                    f\'{label}:crop_bbox:largest\',\n                    lambda g, bf=bg_fn, d=diagonal, m=monochrome:\n                        _selection(g, bf, d, m, \'largest\', masked=False),\n                    cost=1.4, family=\'object_select\'))\n                for criterion in (\'largest\', \'smallest\', \'non_singletons\', \'unique_size\',\n                                  \'repeated_shape\', \'unique_shape\', \'border\', \'interior\'):\n                    rules.append(Rule(\n                        f\'{label}:keep:{criterion}\',\n                        lambda g, bf=bg_fn, d=diagonal, m=monochrome, c=criterion:\n                            _filter(g, bf, d, m, c),\n                        cost=1.5, family=\'object_filter\'))\n    return rules\n', 'arc2/operators.py': '"""Small, composable grid operators; parameters come only from demonstrations."""\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .grid import background, crop\nfrom .types import Rule\n\n\ndef _bg(a, color):\n    return background(a) if color == -1 else color\n\n\ndef _crop_color(a, color):\n    yy, xx = np.where(a == color)\n    return None if not len(yy) else a[yy.min():yy.max()+1, xx.min():xx.max()+1].copy()\n\n\ndef _remove_lines(a, color, axis):\n    if axis == 0:\n        return a[~np.all(a == color, axis=1)].copy()\n    return a[:, ~np.all(a == color, axis=0)].copy()\n\n\ndef _compress(a, axis):\n    if axis == 0:\n        keep = np.r_[True, np.any(a[1:] != a[:-1], axis=1)]\n        return a[keep].copy()\n    keep = np.r_[True, np.any(a[:, 1:] != a[:, :-1], axis=0)]\n    return a[:, keep].copy()\n\n\ndef _downscale(a, fy, fx, mode):\n    h, w = a.shape\n    if h % fy or w % fx:\n        return None\n    blocks = a.reshape(h // fy, fy, w // fx, fx).transpose(0, 2, 1, 3)\n    if mode == "first":\n        return blocks[:, :, 0, 0].copy()\n    if mode == "uniform":\n        first = blocks[:, :, :1, :1]\n        return blocks[:, :, 0, 0].copy() if np.all(blocks == first) else None\n    out = np.zeros((h // fy, w // fx), dtype=np.uint8)\n    for y in range(out.shape[0]):\n        for x in range(out.shape[1]):\n            counts = np.bincount(blocks[y, x].ravel(), minlength=10)\n            if mode == "nonzero":\n                counts[0] = 0\n            out[y, x] = int(counts.argmax())\n    return out\n\n\ndef _shift(a, dy, dx, color):\n    h, w = a.shape\n    out = np.full_like(a, _bg(a, color))\n    sy0, sy1 = max(0, -dy), min(h, h-dy)\n    sx0, sx1 = max(0, -dx), min(w, w-dx)\n    if sy1 > sy0 and sx1 > sx0:\n        out[sy0+dy:sy1+dy, sx0+dx:sx1+dx] = a[sy0:sy1, sx0:sx1]\n    return out\n\n\ndef _window(a, h, w, anchor):\n    if h > a.shape[0] or w > a.shape[1]:\n        return None\n    ay, ax = anchor\n    r = (a.shape[0]-h) * ay // 2\n    c = (a.shape[1]-w) * ax // 2\n    return a[r:r+h, c:c+w].copy()\n\n\ndef _resize_canvas(a, dy, dx, anchor, color):\n    h, w = a.shape[0]+dy, a.shape[1]+dx\n    if not (1 <= h <= 30 and 1 <= w <= 30):\n        return None\n    if dy < 0 or dx < 0:\n        return _window(a, h, w, anchor) if dy <= 0 and dx <= 0 else None\n    out = np.full((h, w), _bg(a, color), dtype=np.uint8)\n    y, x = dy * anchor[0]//2, dx * anchor[1]//2\n    out[y:y+a.shape[0], x:x+a.shape[1]] = a\n    return out\n\n\ndef _minimal_tile(a):\n    h, w = a.shape\n    for hh in range(1, h+1):\n        if h % hh:\n            continue\n        for ww in range(1, w+1):\n            if w % ww:\n                continue\n            b = a[:hh, :ww]\n            if np.array_equal(np.tile(b, (h//hh, w//ww)), a):\n                return b.copy()\n    return a.copy()\n\n\ndef _color_rank(a, reverse=False):\n    counts = np.bincount(a.ravel(), minlength=10)\n    present = [i for i in range(10) if counts[i]]\n    present.sort(key=lambda x: ((-1 if reverse else 1)*int(counts[x]), x))\n    lut = np.arange(10, dtype=np.uint8)\n    for rank, color in enumerate(present):\n        lut[color] = rank\n    return lut[a]\n\n\ndef _summary(a, axis, bg):\n    b = a if axis == 0 else a.T\n    colors = []\n    for row in b:\n        vals = np.unique(row[row != _bg(a, bg)])\n        if len(vals) > 1:\n            return None\n        colors.append(int(vals[0]) if len(vals) else _bg(a, bg))\n    out = np.asarray(colors, dtype=np.uint8).reshape(-1, 1)\n    return out if axis == 0 else out.T\n\n\ndef primitive_rules(train):\n    ins = [np.asarray(p["input"], dtype=np.uint8) for p in train]\n    outs = [np.asarray(p["output"], dtype=np.uint8) for p in train]\n    palette = sorted(set(int(c) for a in ins for c in np.unique(a)))\n    rules = [\n        Rule("rotate90", lambda a: np.rot90(a).copy(), 1, "geometry"),\n        Rule("rotate180", lambda a: np.rot90(a, 2).copy(), 1, "geometry"),\n        Rule("rotate270", lambda a: np.rot90(a, 3).copy(), 1, "geometry"),\n        Rule("flip_vertical", lambda a: a[::-1].copy(), 1, "geometry"),\n        Rule("flip_horizontal", lambda a: a[:, ::-1].copy(), 1, "geometry"),\n        Rule("transpose", lambda a: a.T.copy(), 1, "geometry"),\n        Rule("anti_transpose", lambda a: a.T[::-1, ::-1].copy(), 1, "geometry"),\n        Rule("compress_rows", lambda a: _compress(a, 0), 1.4, "compression"),\n        Rule("compress_columns", lambda a: _compress(a, 1), 1.4, "compression"),\n        Rule("compress_rows_columns", lambda a: _compress(_compress(a, 0), 1), 2, "compression"),\n        Rule("minimal_tile", _minimal_tile, 2, "periodicity"),\n        Rule("color_rank_ascending", lambda a: _color_rank(a, False), 2.5, "color_rank"),\n        Rule("color_rank_descending", lambda a: _color_rank(a, True), 2.5, "color_rank"),\n    ]\n    for color in [-1] + palette:\n        label = "dominant" if color == -1 else str(color)\n        rules.append(Rule(f"crop_background_{label}", lambda a, c=color: crop(a, _bg(a,c)), 1.2, "crop"))\n        if color >= 0:\n            rules.append(Rule(f"crop_color_{color}", lambda a, c=color: _crop_color(a,c), 1.4, "crop"))\n            rules.append(Rule(f"mask_color_{color}", lambda a, c=color: (a == c).astype(np.uint8), 1.8, "mask"))\n            rules.append(Rule(f"erase_color_{color}", lambda a, c=color: np.where(a == c,0,a), 1.5, "color"))\n        for axis in (0,1):\n            rules.append(Rule(f"remove_uniform_{axis}_{label}", lambda a,c=color,ax=axis: _remove_lines(a,_bg(a,c),ax), 1.5, "compression"))\n            rules.append(Rule(f"summarize_{axis}_{label}", lambda a,c=color,ax=axis: _summary(a,ax,c), 2, "summary"))\n\n    # Spatial operations include generic factors, plus ratios supported by all pairs.\n    factors = {(2,2), (3,3), (2,1), (1,2), (3,1), (1,3)}\n    if ins:\n        for a,b in zip(ins,outs):\n            if b.shape[0] % a.shape[0] == 0 and b.shape[1] % a.shape[1] == 0:\n                fy,fx = b.shape[0]//a.shape[0], b.shape[1]//a.shape[1]\n                if 1 <= fy <= 10 and 1 <= fx <= 10 and (fy,fx) != (1,1): factors.add((fy,fx))\n    for fy,fx in sorted(factors):\n        rules.append(Rule(f"scale_{fy}_{fx}", lambda a,y=fy,x=fx: np.repeat(np.repeat(a,y,axis=0),x,axis=1) if a.shape[0]*y <= 30 and a.shape[1]*x <= 30 else None, 1.5 + .05*(fy+fx), "scale"))\n        rules.append(Rule(f"tile_{fy}_{fx}", lambda a,y=fy,x=fx: np.tile(a,(y,x)) if a.shape[0]*y <= 30 and a.shape[1]*x <= 30 else None, 1.8, "tile"))\n    down = {(2,2),(3,3),(2,1),(1,2)}\n    for a,b in zip(ins,outs):\n        if a.shape[0] % b.shape[0] == 0 and a.shape[1] % b.shape[1] == 0:\n            y,x = a.shape[0]//b.shape[0],a.shape[1]//b.shape[1]\n            if (y,x) != (1,1): down.add((y,x))\n    for fy,fx in sorted(down):\n        for mode in ("uniform","first","mode","nonzero"):\n            rules.append(Rule(f"downscale_{fy}_{fx}_{mode}", lambda a,y=fy,x=fx,m=mode: _downscale(a,y,x,m), 1.8 if mode == "uniform" else 2.4, "scale"))\n    for axis in (0,1):\n        for reverse in (False,True):\n            for flip_axis in (0,1):\n                def mirror(a,ax=axis,rev=reverse,fl=flip_axis):\n                    if a.shape[ax]*2 > 30: return None\n                    b=np.flip(a,axis=fl)\n                    return np.concatenate((b,a) if rev else (a,b),axis=ax)\n                rules.append(Rule(f"mirror_tile_{axis}_{int(reverse)}_{flip_axis}",mirror,1.8,"symmetry"))\n    for bg in (0,-1):\n        def kron(a,c=bg,reverse=False):\n            if a.shape[0]**2 > 30 or a.shape[1]**2 > 30: return None\n            mask=(a != _bg(a,c)).astype(np.uint8)\n            return np.kron(a,mask) if reverse else np.kron(mask,a)\n        rules.append(Rule(f"self_tile_{bg}",kron,2.8,"fractal"))\n        rules.append(Rule(f"self_tile_reverse_{bg}",lambda a,c=bg: kron(a,c,True),2.8,"fractal"))\n\n    deltas = set()\n    if ins:\n        ds = {(b.shape[0]-a.shape[0],b.shape[1]-a.shape[1]) for a,b in zip(ins,outs)}\n        if len(ds) == 1 and ds != {(0,0)}: deltas |= ds\n    for dy,dx in sorted(deltas):\n        for anchor in ((0,0),(0,2),(2,0),(2,2),(1,1)):\n            for bg in (0,-1):\n                rules.append(Rule(f"canvas_{dy}_{dx}_{anchor}_{bg}",lambda a,y=dy,x=dx,an=anchor,c=bg: _resize_canvas(a,y,x,an,c),2.2,"canvas"))\n    shapes = set(b.shape for b in outs)\n    if len(shapes)==1:\n        h,w=next(iter(shapes))\n        for anchor in ((0,0),(0,2),(2,0),(2,2),(1,1)):\n            rules.append(Rule(f"window_{h}_{w}_{anchor}",lambda a,hh=h,ww=w,an=anchor:_window(a,hh,ww,an),2.4,"crop"))\n    if ins and all(a.shape==b.shape for a,b in zip(ins,outs)):\n        shifts={(dy,dx) for dy,dx in ((-1,0),(1,0),(0,-1),(0,1),(-2,0),(2,0),(0,-2),(0,2))}\n        for a,b in zip(ins,outs):\n            for bg in (0,background(a)):\n                y,x=np.where(a!=bg); yy,xx=np.where(b!=bg)\n                if len(y) and len(yy):\n                    shifts.add((int(yy.min()-y.min()),int(xx.min()-x.min())))\n        for dy,dx in sorted(shifts-{(0,0)}):\n            if abs(dy)<30 and abs(dx)<30:\n                rules.append(Rule(f"shift_{dy}_{dx}",lambda a,y=dy,x=dx:_shift(a,y,x,0),2+0.04*(abs(dy)+abs(dx)),"translation"))\n    return rules\n\n\ndef constant_rule(train):\n    outputs = [np.asarray(p["output"],dtype=np.uint8) for p in train]\n    if not outputs or not all(np.array_equal(outputs[0], a) for a in outputs):\n        return None\n    fixed=outputs[0].copy()\n    return Rule("constant_output",lambda a:fixed.copy(),4.0+fixed.size*.04,"constant")\n', 'arc2/patterns.py': '"""Small, train-only families for local, periodic, and panel reasoning.\n\nThese are hypothesis generators, not a catalogue of ARC task IDs.  Every emitted\nrule reproduces every demonstration exactly.  This is an empirical consistency\ncheck, not a claim that the rule will generalize to the test input.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom itertools import product\nimport math\nimport time\nfrom typing import Callable\n\nimport numpy as np\n\nfrom .types import Rule\n\n\ndef _bg(g: np.ndarray, selector: int | str) -> int:\n    if selector == "mode":\n        return int(np.bincount(g.ravel(), minlength=10).argmax())\n    return int(selector)\n\n\ndef _valid(g: object) -> bool:\n    return (\n        isinstance(g, np.ndarray)\n        and g.ndim == 2\n        and 0 < g.shape[0] <= 30\n        and 0 < g.shape[1] <= 30\n        and np.issubdtype(g.dtype, np.integer)\n        and bool(np.all((g >= 0) & (g <= 9)))\n    )\n\n\ndef _symmetry_complete(g: np.ndarray, mask: int, group: str) -> np.ndarray | None:\n    transforms: list[np.ndarray] = [g]\n    if group == "h":\n        transforms += [np.fliplr(g)]\n    elif group == "v":\n        transforms += [np.flipud(g)]\n    elif group == "r2":\n        transforms += [np.rot90(g, 2)]\n    elif group == "hv":\n        transforms += [np.fliplr(g), np.flipud(g), np.rot90(g, 2)]\n    elif group in {"d", "a", "r4", "d4"}:\n        if g.shape[0] != g.shape[1]:\n            return None\n        if group == "d":\n            transforms += [g.T]\n        elif group == "a":\n            transforms += [np.rot90(g.T, 2)]\n        else:\n            transforms = [np.rot90(g, k) for k in range(4)]\n            if group == "d4":\n                transforms += [np.rot90(g.T, k) for k in range(4)]\n    out = g.copy()\n    known = out != mask\n    for other in transforms[1:]:\n        usable = other != mask\n        if np.any(known & usable & (out != other)):\n            return None\n        take = ~known & usable\n        out[take] = other[take]\n        known |= usable\n    return out\n\n\ndef _period_complete(\n    g: np.ndarray, mask: int, period_r: int | None, period_c: int | None\n) -> np.ndarray | None:\n    h, w = g.shape\n    pr, pc = period_r or h, period_c or w\n    if pr > h or pc > w:\n        return None\n    rr, cc = np.indices(g.shape)\n    codes = (rr % pr) * pc + cc % pc\n    tile = np.full(pr * pc, -1, dtype=np.int16)\n    known = g != mask\n    for color in np.unique(g[known]):\n        indices = np.unique(codes[known & (g == color)])\n        if np.any((tile[indices] >= 0) & (tile[indices] != color)):\n            return None\n        tile[indices] = color\n    fill = tile[codes]\n    if np.any((~known) & (fill < 0)):\n        return None\n    return np.where(known, g, fill).astype(np.int16)\n\n\ndef _split_equal(\n    g: np.ndarray, axis: int, n: int, gap: int\n) -> list[np.ndarray] | None:\n    length = g.shape[axis]\n    usable = length - gap * (n - 1)\n    if usable <= 0 or usable % n:\n        return None\n    size = usable // n\n    result = []\n    for i in range(n):\n        start = i * (size + gap)\n        panel = g[start : start + size, :] if axis == 0 else g[:, start : start + size]\n        result.append(panel)\n        if i < n - 1 and gap:\n            line = g[start + size : start + size + gap, :] if axis == 0 else g[:, start + size : start + size + gap]\n            if not np.all(line == line.flat[0]):\n                return None\n    return result\n\n\ndef _split_separators(g: np.ndarray, color: int) -> list[np.ndarray] | None:\n    row_lines = np.flatnonzero(np.all(g == color, axis=1))\n    col_lines = np.flatnonzero(np.all(g == color, axis=0))\n    if not len(row_lines) and not len(col_lines):\n        return None\n\n    def spans(length: int, lines: np.ndarray) -> list[tuple[int, int]]:\n        blocked = set(map(int, lines))\n        out = []\n        start = 0\n        for i in range(length + 1):\n            if i == length or i in blocked:\n                if i > start:\n                    out.append((start, i))\n                start = i + 1\n        return out\n\n    rows, cols = spans(g.shape[0], row_lines), spans(g.shape[1], col_lines)\n    panels = [g[a:b, c:d] for a, b in rows for c, d in cols]\n    if not 2 <= len(panels) <= 16 or len({x.shape for x in panels}) != 1:\n        return None\n    return panels\n\n\ndef _panel_overlay(\n    panels: list[np.ndarray], bg: int, operation: str\n) -> np.ndarray:\n    stack = np.stack(panels)\n    if operation in {"first", "last"}:\n        out = np.full(panels[0].shape, bg, dtype=np.int16)\n        sequence = reversed(panels) if operation == "first" else panels\n        for panel in sequence:\n            np.copyto(out, panel, where=panel != bg)\n        return out\n    nonbg = stack != bg\n    if operation == "consensus":\n        present = np.any(nonbg, axis=0)\n        lo = np.min(np.where(nonbg, stack, 10), axis=0)\n        hi = np.max(np.where(nonbg, stack, -1), axis=0)\n        return np.where(present & (lo == hi), lo, bg).astype(np.int16)\n    raise ValueError(operation)\n\n\ndef _panel_codes(panels: list[np.ndarray], bg: int, mode: str) -> np.ndarray:\n    stack = np.stack(panels)\n    if mode == "count":\n        return np.count_nonzero(stack != bg, axis=0)\n    if mode == "bits":\n        codes = np.zeros(panels[0].shape, dtype=np.int64)\n        for i, p in enumerate(panels):\n            codes += (p != bg).astype(np.int64) << i\n        return codes\n    # Color tuple coding distinguishes overlapping source colors without using\n    # task identifiers or memorizing spatial coordinates.\n    codes = np.zeros(panels[0].shape, dtype=np.int64)\n    for p in panels:\n        codes = codes * 10 + p\n    return codes\n\n\ndef _learn_map(xs: list[np.ndarray], ys: list[np.ndarray]) -> dict[int, int] | None:\n    mapping: dict[int, int] = {}\n    for x, y in zip(xs, ys):\n        for code in np.unique(x):\n            values = np.unique(y[x == code])\n            if len(values) != 1:\n                return None\n            key, value = int(code), int(values[0])\n            if key in mapping and mapping[key] != value:\n                return None\n            mapping[key] = value\n    return mapping\n\n\ndef _paint_lines(\n    g: np.ndarray,\n    bg_selector: int | str,\n    marker: int | str,\n    paint: int | str,\n    directions: tuple[tuple[int, int], ...],\n    extend: bool,\n) -> np.ndarray:\n    bg = _bg(g, bg_selector)\n    colors = [int(x) for x in np.unique(g) if x != bg] if marker == "each" else [int(marker)]\n    out = g.copy()\n    h, w = g.shape\n    for color in colors:\n        draw = color if paint == "marker" else int(paint)\n        positions = np.argwhere(g == color)\n        for dr, dc in directions:\n            groups: dict[int, list[tuple[int, int]]] = {}\n            for r, c in positions:\n                invariant = int(c * dr - r * dc)\n                groups.setdefault(invariant, []).append((int(r), int(c)))\n            for points in groups.values():\n                if not extend and len(points) < 2:\n                    continue\n                points.sort(key=lambda p: p[0] * dr + p[1] * dc)\n                r, c = points[0]\n                er, ec = points[-1]\n                if extend:\n                    while 0 <= r - dr < h and 0 <= c - dc < w:\n                        r, c = r - dr, c - dc\n                    while 0 <= er + dr < h and 0 <= ec + dc < w:\n                        er, ec = er + dr, ec + dc\n                while True:\n                    if g[r, c] == bg:\n                        out[r, c] = draw\n                    if r == er and c == ec:\n                        break\n                    r, c = r + dr, c + dc\n    return out\n\n\ndef _features(g: np.ndarray, selector: int | str) -> dict[str, tuple[np.ndarray, ...]]:\n    """Vectorized context features; no absolute coordinates or output labels."""\n    bg = _bg(g, selector)\n    h, w = g.shape\n    pad = np.pad(g, 1, constant_values=10)\n    shifts = [\n        pad[1 + dr : h + 1 + dr, 1 + dc : w + 1 + dc]\n        for dr, dc in [(-1, 0), (0, 1), (1, 0), (0, -1), (-1, -1), (-1, 1), (1, 1), (1, -1)]\n    ]\n    count4 = sum(((x != bg) & (x != 10)).astype(np.int16) for x in shifts[:4])\n    count8 = sum(((x != bg) & (x != 10)).astype(np.int16) for x in shifts)\n    same4 = sum((x == g).astype(np.int16) for x in shifts[:4])\n    same8 = sum((x == g).astype(np.int16) for x in shifts)\n    row_n = np.broadcast_to(np.count_nonzero(g != bg, axis=1)[:, None], g.shape)\n    col_n = np.broadcast_to(np.count_nonzero(g != bg, axis=0)[None, :], g.shape)\n    row_colors = np.zeros(g.shape, dtype=np.int64)\n    col_colors = np.zeros(g.shape, dtype=np.int64)\n    row_same = np.zeros(g.shape, dtype=np.int16)\n    col_same = np.zeros(g.shape, dtype=np.int16)\n    freq = np.bincount(g.ravel(), minlength=10)\n    rank = np.array([np.count_nonzero(freq > freq[c]) for c in range(10)], dtype=np.int16)\n    for color in np.unique(g):\n        has = g == color\n        row_colors |= np.any(has, axis=1)[:, None].astype(np.int64) << int(color)\n        col_colors |= np.any(has, axis=0)[None, :].astype(np.int64) << int(color)\n        row_same[has] = np.broadcast_to(np.sum(has, axis=1)[:, None], g.shape)[has]\n        col_same[has] = np.broadcast_to(np.sum(has, axis=0)[None, :], g.shape)[has]\n    result = {\n        "color": (g,),\n        "count4": (g, count4),\n        "count8": (g, count8),\n        "same4": (g, same4),\n        "same8": (g, same8),\n        "counts": (g, count4, count8),\n        "same_counts": (g, same4, same8),\n        "row_count": (g, row_n),\n        "col_count": (g, col_n),\n        "cross_counts": (g, row_n, col_n),\n        "row_same": (g, row_same),\n        "col_same": (g, col_same),\n        "cross_same": (g, row_same, col_same),\n        "row_colors": (g, row_colors),\n        "col_colors": (g, col_colors),\n        "cross_colors": (g, row_colors, col_colors),\n        "color_rank": (g, rank[g]),\n        "vertical": (g, shifts[0], shifts[2]),\n        "horizontal": (g, shifts[1], shifts[3]),\n        "cross": (g, *shifts[:4]),\n        "patch": (g, *shifts),\n    }\n    # Rotation-invariant neighbor color counts are deliberately separate from\n    # ordered patches, since their inductive bias and complexity differ.\n    present = [int(c) for c in np.unique(g)]\n    # Always use all ten color bins so feature identities remain stable when a\n    # color is absent from one demonstration or the unseen input.\n    del present\n    result["hist4"] = (g, *(sum((x == c).astype(np.int16) for x in shifts[:4]) for c in range(10)))\n    result["hist8"] = (g, *(sum((x == c).astype(np.int16) for x in shifts) for c in range(10)))\n    return result\n\n\ndef _feature_rows(arrays: tuple[np.ndarray, ...]) -> np.ndarray:\n    return np.stack([a.ravel() for a in arrays], axis=1)\n\n\ndef _local_map(\n    g: np.ndarray, selector: int | str, feature: str, mapping: dict[tuple[int, ...], int]\n) -> np.ndarray:\n    rows = _feature_rows(_features(g, selector)[feature])\n    values = np.fromiter(\n        (mapping.get(tuple(map(int, row)), int(center)) for row, center in zip(rows, g.ravel())),\n        dtype=np.int16,\n        count=g.size,\n    )\n    return values.reshape(g.shape)\n\n\ndef pattern_rules(train: list[dict], deadline: float | None = None) -> list[Rule]:\n    """Infer bounded rule families from demonstrations, with no test-label access.\n\n    ``deadline`` is an absolute ``time.monotonic()`` value.  This component uses\n    at most roughly 0.75 seconds even if the outer search has a larger budget;\n    one NumPy operation already in progress can finish after the deadline.\n    """\n    if not train:\n        return []\n    started = time.monotonic()\n    stop = min(deadline if deadline is not None else math.inf, started + 0.75)\n    pairs = [(np.asarray(p["input"], dtype=np.int16), np.asarray(p["output"], dtype=np.int16)) for p in train]\n    if not all(_valid(x) and _valid(y) for x, y in pairs):\n        return []\n    xs, ys = [x for x, _ in pairs], [y for _, y in pairs]\n    same_shape = all(x.shape == y.shape for x, y in pairs)\n    rules: list[Rule] = []\n    names: set[str] = set()\n\n    def expired() -> bool:\n        return time.monotonic() >= stop\n\n    def add(name: str, fn: Callable, cost: float, family: str) -> None:\n        if expired() or len(rules) >= 96 or name in names:\n            return\n        try:\n            for x, y in pairs:\n                predicted = fn(x)\n                if not _valid(predicted) or not np.array_equal(predicted, y):\n                    return\n        except (ValueError, IndexError, TypeError, OverflowError):\n            return\n        names.add(name)\n        rules.append(Rule(name=name, fn=fn, cost=cost, family=family))\n\n    common_input_colors = sorted(set.intersection(*(set(map(int, np.unique(x))) for x in xs)))\n    backgrounds: list[int | str] = [0, "mode"]\n    for c in common_input_colors:\n        if c != 0 and all(np.mean(x == c) >= 0.25 for x in xs):\n            backgrounds.append(c)\n\n    # Panels are considered first because a dimension reduction is informative\n    # and makes expensive same-size local models irrelevant.\n    splitters: list[tuple[str, Callable]] = []\n    if not same_shape:\n        for axis, n, gap in product((0, 1), range(2, 5), (0, 1)):\n            fn = lambda g, a=axis, n=n, gap=gap: _split_equal(g, a, n, gap)\n            panels = [fn(x) for x in xs]\n            if all(p is not None and p[0].shape == y.shape for p, y in zip(panels, ys)):\n                splitters.append((f"equal{axis}:{n}:{gap}", fn))\n        for color in common_input_colors:\n            fn = lambda g, c=color: _split_separators(g, c)\n            panels = [fn(x) for x in xs]\n            if all(p is not None and p[0].shape == y.shape for p, y in zip(panels, ys)):\n                splitters.append((f"separator:{color}", fn))\n    for label, split in splitters:\n        if expired():\n            return rules\n        training_panels = [split(x) for x in xs]\n        for selector in backgrounds:\n            for operation in ("first", "last", "consensus"):\n                def overlay(g, sp=split, s=selector, op=operation):\n                    ps = sp(g)\n                    return None if ps is None else _panel_overlay(ps, _bg(g, s), op)\n                add(f"panel:{label}:bg{selector}:{operation}", overlay, 2.0, "panels")\n            for mode in ("count", "bits", "colors"):\n                if mode == "colors" and max(len(p) for p in training_panels) > 8:\n                    continue\n                codes = [_panel_codes(p, _bg(x, selector), mode) for p, x in zip(training_panels, xs)]\n                mapping = _learn_map(codes, ys)\n                if mapping is None:\n                    continue\n                def mapped(g, sp=split, s=selector, mode=mode, mapping=mapping):\n                    ps = sp(g)\n                    if ps is None or (mode == "colors" and len(ps) > 8):\n                        return None\n                    bg = _bg(g, s)\n                    code = _panel_codes(ps, bg, mode)\n                    out = np.full(code.shape, bg, dtype=np.int16)\n                    for k, value in mapping.items():\n                        out[code == k] = value\n                    return out\n                cost = 2.1 + 0.12 * len(mapping) + (0.8 if mode == "colors" else 0)\n                add(f"panel:{label}:bg{selector}:{mode}", mapped, cost, "panels")\n\n    if not same_shape or expired():\n        return rules\n\n    mask_colors = [c for c in common_input_colors if any(np.any((x == c) & (x != y)) for x, y in pairs)]\n    for mask in mask_colors:\n        for group in ("h", "v", "r2", "hv", "d", "a", "r4", "d4"):\n            add(f"complete:{group}:mask{mask}", lambda g, m=mask, s=group: _symmetry_complete(g, m, s), 1.6 + 0.1 * len(group), "symmetry")\n        if expired():\n            return rules\n        row_periods: list[int | None] = [None]\n        col_periods: list[int | None] = [None]\n        for p in range(1, min(12, min(y.shape[0] for y in ys)) + 1):\n            if all(np.array_equal(y[p:], y[:-p]) for y in ys) and any(p < y.shape[0] for y in ys):\n                row_periods.append(p)\n        for p in range(1, min(12, min(y.shape[1] for y in ys)) + 1):\n            if all(np.array_equal(y[:, p:], y[:, :-p]) for y in ys) and any(p < y.shape[1] for y in ys):\n                col_periods.append(p)\n        periods = [(pr, pc) for pr, pc in product(row_periods, col_periods) if pr is not None or pc is not None]\n        periods.sort(key=lambda pair: ((pair[0] or 31) * (pair[1] or 31), str(pair)))\n        for pr, pc in periods[:16]:\n            add(f"period:{pr}:{pc}:mask{mask}", lambda g, m=mask, r=pr, c=pc: _period_complete(g, m, r, c), 2.0 + 0.03 * ((pr or 1) + (pc or 1)), "periodic")\n\n    # Draw through markers while preserving all non-background input pixels.\n    changed_colors = sorted(set.intersection(*(set(map(int, np.unique(y[x != y]))) for x, y in pairs)))\n    markers: list[int | str] = ["each", *common_input_colors]\n    directions = [((0, 1),), ((1, 0),), ((0, 1), (1, 0)), ((1, 1),), ((1, -1),), ((1, 1), (1, -1))]\n    for selector in backgrounds[:2]:\n        # A line painter cannot recolor an occupied input cell.\n        if any(np.any((x != _bg(x, selector)) & (x != y)) for x, y in pairs):\n            continue\n        for marker in markers:\n            if marker != "each" and all(marker == _bg(x, selector) for x in xs):\n                continue\n            for paint in ["marker", *changed_colors[:3]]:\n                for ds, extend in product(directions, (False, True)):\n                    if expired():\n                        return rules\n                    add(\n                        f"line:bg{selector}:mark{marker}:paint{paint}:{ds}:extend{extend}",\n                        lambda g, s=selector, m=marker, p=paint, d=ds, e=extend: _paint_lines(g, s, m, p, d, e),\n                        2.3 + 0.2 * len(ds) + 0.3 * extend,\n                        "lines",\n                    )\n\n    # Learn context-to-color maps.  Unknown contexts retain their input color.\n    # All unchanged pixels also constrain the map, preventing a changed-pixel\n    # rule from being accepted if it damages an unchanged demonstration cell.\n    for selector in backgrounds[:2]:\n        if expired():\n            return rules\n        prepared = [_features(x, selector) for x in xs]\n        for feature in prepared[0]:\n            if expired():\n                return rules\n            mapping: dict[tuple[int, ...], int] = {}\n            counts: Counter = Counter()\n            consistent = True\n            for f, y in zip(prepared, ys):\n                for row, value in zip(_feature_rows(f[feature]), y.ravel()):\n                    key, value = tuple(map(int, row)), int(value)\n                    if key in mapping and mapping[key] != value:\n                        consistent = False\n                        break\n                    mapping[key] = value\n                    counts[key] += 1\n                if not consistent:\n                    break\n            if not consistent:\n                continue\n            changes = {k: v for k, v in mapping.items() if k[0] != v}\n            if not changes:\n                continue\n            sparse = sum(counts[k] == 1 for k in changes)\n            cost = 2.0 + 0.35 * len(prepared[0][feature]) + 0.08 * len(changes) + 0.15 * sparse\n            add(\n                f"local:bg{selector}:{feature}",\n                lambda g, s=selector, f=feature, m=changes: _local_map(g, s, f, m),\n                cost,\n                "local",\n            )\n    return rules\n', 'arc2/portfolio.py': '"""A fixed, label-free two-attempt policy for the independent search engines."""\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .grid import key, valid_grid\n\n\ndef merge_predictions(symbolic, cpp, training_count):\n    """Prefer exact demonstration fits and preserve independent predictions.\n\n    Complexity numbers from different engines are not calibrated probabilities.\n    The policy is declared explicitly: very simple Python programs (cost<=2.8),\n    exact C++ candidates, other exact Python programs, then partial fallbacks.\n    For the second slot, prefer a different engine at the same consistency tier,\n    unless doing so would substitute a complex Python lookup (cost>8) for an\n    exact C++ candidate. No held-out labels enter this function.\n    """\n    attempts=[]; selections=[]\n    for i, own in enumerate(symbolic["attempts"]):\n        candidates=[]\n        own_meta=symbolic.get("diagnostics",{}).get("selection",[])\n        for j in range(2):\n            meta=own_meta[i][j] if i<len(own_meta) and j<len(own_meta[i]) else {}\n            exact=bool(meta.get("training_exact",False))\n            cost=float(meta.get("cost",100))\n            candidates.append({"grid":own[f"attempt_{j+1}"],"origin":"python",\n                               "training_exact":exact,"cost":cost,"rank":j,\n                               "program":meta.get("program",[]),\n                               "quality":3 if exact and cost<=2.8 else 1 if exact else 0})\n        cpp_grids=cpp.get("predictions",[])\n        for j,g in enumerate(cpp_grids[i] if i<len(cpp_grids) else []):\n            if not valid_grid(g): continue\n            matches=cpp.get("train_matches",[])\n            count=matches[i][j] if i<len(matches) and j<len(matches[i]) else -1\n            exact=count==training_count\n            raw=cpp.get("scores",[])\n            raw_score=raw[i][j] if i<len(raw) and j<len(raw[i]) else None\n            candidates.append({"grid":g,"origin":"cpp","training_exact":exact,\n                               "train_matches":count,"raw_score":raw_score,"rank":j,\n                               "quality":2 if exact else 0})\n        candidates.sort(key=lambda c:(-int(c["training_exact"]),-c["quality"],c["rank"],c["origin"]))\n        first=candidates[0]\n        seen={key(np.asarray(first["grid"],dtype=np.uint8))}\n        alternatives=[c for c in candidates if key(np.asarray(c["grid"],dtype=np.uint8)) not in seen]\n        if alternatives:\n            best=alternatives[0]\n            cross=[c for c in alternatives if c["origin"]!=first["origin"]\n                   and c["training_exact"]==best["training_exact"]\n                   and not (c["origin"]=="python" and c.get("cost",100)>8 and best["training_exact"])]\n            second=cross[0] if cross else best\n        else:\n            second=first\n        chosen=[first,second]\n        attempts.append({f"attempt_{j+1}":np.asarray(c["grid"],dtype=np.uint8).tolist()\n                         for j,c in enumerate(chosen)})\n        selections.append([{k:v for k,v in c.items() if k not in {"grid","quality"}} for c in chosen])\n    return attempts,selections\n\n', 'arc2/relations.py': '"""Small exact-fit relation families learned exclusively from demonstrations."""\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom functools import lru_cache\nimport time\n\nimport numpy as np\n\nfrom .grid import background\nfrom .types import Rule\n\n\ndef _bg_zero(grid):\n    return 0\n\n\ndef _bg_mode(grid):\n    return int(background(grid))\n\n\n@lru_cache(maxsize=128)\ndef _enclosed_cached(shape: tuple[int, int], data: bytes, bg: int,\n                     diagonal: bool) -> tuple:\n    grid = np.frombuffer(data, dtype=np.uint8).reshape(shape)\n    h, w = shape\n    directions = [(1,0),(-1,0),(0,1),(0,-1)]\n    if diagonal:\n        directions += [(1,1),(1,-1),(-1,1),(-1,-1)]\n    visited = np.zeros(shape, dtype=bool)\n    result = []\n    for r, c in np.argwhere(grid == bg):\n        r, c = int(r), int(c)\n        if visited[r,c]:\n            continue\n        visited[r,c] = True\n        queue = deque([(r,c)])\n        cells = []\n        boundary = set()\n        outside = False\n        while queue:\n            a, b = queue.popleft()\n            cells.append((a,b))\n            outside |= a in (0,h-1) or b in (0,w-1)\n            for dr, dc in directions:\n                rr, cc = a+dr, b+dc\n                if 0 <= rr < h and 0 <= cc < w:\n                    if grid[rr,cc] != bg:\n                        boundary.add(int(grid[rr,cc]))\n                    elif not visited[rr,cc]:\n                        visited[rr,cc] = True\n                        queue.append((rr,cc))\n        if not outside:\n            result.append((tuple(cells), tuple(sorted(boundary))))\n    return tuple(result)\n\n\ndef _enclosed(grid, bg, diagonal):\n    a = np.asarray(grid,dtype=np.uint8)\n    return _enclosed_cached(a.shape,a.tobytes(),bg,diagonal)\n\n\ndef _fill(grid, bg_fn, diagonal, target, selector=\'all\', boundary_color=None):\n    bg = bg_fn(grid)\n    regions = _enclosed(grid,bg,diagonal)\n    if boundary_color is not None:\n        regions = tuple(r for r in regions if r[1] == (boundary_color,))\n    if not regions:\n        return None\n    if selector == \'largest\':\n        size = max(len(cells) for cells,_ in regions)\n        regions = tuple(r for r in regions if len(r[0]) == size)\n    elif selector == \'smallest\':\n        size = min(len(cells) for cells,_ in regions)\n        regions = tuple(r for r in regions if len(r[0]) == size)\n    out = grid.copy()\n    for cells,_ in regions:\n        for r,c in cells:\n            out[r,c] = target\n    return out\n\n\ndef _bbox_color(grid, color):\n    rows,cols = np.where(grid == color)\n    if not len(rows):\n        return None\n    return (int(rows.min()),int(cols.min()),int(rows.max())+1,int(cols.max())+1)\n\n\ndef _axis_ref(bbox, axis, ref):\n    low,high = bbox[axis],bbox[axis+2]-1\n    # Twice the coordinate represents half-integer centers without rounding.\n    return {\'min\':2*low,\'max\':2*high,\'center\':low+high}[ref]\n\n\ndef _axis_delta(moving_bbox, anchor_bbox, axis, formula):\n    if formula == (\'unchanged\',):\n        return 0\n    anchor_ref,moving_ref,offset = formula\n    twice = (_axis_ref(anchor_bbox,axis,anchor_ref)\n             -_axis_ref(moving_bbox,axis,moving_ref)+2*offset)\n    return twice//2 if twice % 2 == 0 else None\n\n\ndef _translate(grid,bg_fn,moving_color,anchor_color,row_formula,col_formula):\n    bg = bg_fn(grid)\n    if moving_color in (bg,anchor_color):\n        return None\n    moving_bbox,anchor_bbox = _bbox_color(grid,moving_color),_bbox_color(grid,anchor_color)\n    if moving_bbox is None or anchor_bbox is None:\n        return None\n    dr = _axis_delta(moving_bbox,anchor_bbox,0,row_formula)\n    dc = _axis_delta(moving_bbox,anchor_bbox,1,col_formula)\n    if dr is None or dc is None:\n        return None\n    rows,cols = np.where(grid == moving_color)\n    nr,nc = rows+dr,cols+dc\n    h,w = grid.shape\n    if np.any(nr<0) or np.any(nr>=h) or np.any(nc<0) or np.any(nc>=w):\n        return None\n    if np.any((grid[nr,nc] != bg) & (grid[nr,nc] != moving_color)):\n        return None\n    out = grid.copy()\n    out[rows,cols] = bg\n    out[nr,nc] = moving_color\n    return out\n\n\ndef _observed_translation(x,y,color):\n    source = np.argwhere(x == color)\n    target = np.argwhere(y == color)\n    if not len(source) or source.shape != target.shape:\n        return None\n    delta = target.min(axis=0)-source.min(axis=0)\n    if not np.array_equal(source+delta,target):\n        return None\n    return tuple(int(v) for v in delta)\n\n\ndef _fit_axis(records,axis):\n    formulas = [(\'unchanged\',)]\n    # Equality of any edge or center; offsets +-1 express edge adjacency.\n    formulas += [(a,m,d) for d in (0,-1,1)\n                 for a in (\'min\',\'max\',\'center\') for m in (\'min\',\'max\',\'center\')]\n    found = []\n    for formula in formulas:\n        if all(_axis_delta(m,a,axis,formula) == delta[axis] for m,a,delta in records):\n            found.append(formula)\n    # Favor simpler edge formulas; retain enough ambiguity for unseen layouts.\n    return found[:6]\n\n\ndef _exact(fn,train):\n    for pair in train:\n        result = fn(pair[\'input\'])\n        if result is None or not np.array_equal(result,pair[\'output\']):\n            return False\n    return True\n\n\ndef relation_rules(train: list[dict], deadline: float | None = None) -> list[Rule]:\n    """Learn exact hole-fill and shape-translation rules in at most ~0.2 s.\n\n    Only training input/output arrays are inspected. Every returned rule is pure,\n    with parameters frozen from demonstrations; its name contains no task ID.\n    The limit is cooperative and checked between bounded grid operations.\n    """\n    finish = time.monotonic()+.2\n    if deadline is not None:\n        finish = min(finish,deadline)\n    if not train or time.monotonic() >= finish:\n        return []\n    pairs = [{\'input\':np.asarray(e[\'input\'],dtype=np.uint8),\n              \'output\':np.asarray(e[\'output\'],dtype=np.uint8)} for e in train]\n    if any(p[\'input\'].shape != p[\'output\'].shape for p in pairs):\n        return []\n    changed_colors = sorted({int(c) for p in pairs\n                             for c in p[\'output\'][p[\'input\'] != p[\'output\']]})\n    if not changed_colors:\n        return []\n    rules = []\n    bg_options = [(\'zero\',_bg_zero),(\'mode\',_bg_mode)]\n    for bg_name,bg_fn in bg_options:\n        for diagonal in (False,True):\n            if time.monotonic() >= finish:\n                return rules\n            boundaries = set()\n            for pair in pairs:\n                for _,colors in _enclosed(pair[\'input\'],bg_fn(pair[\'input\']),diagonal):\n                    if len(colors) == 1:\n                        boundaries.add(colors[0])\n            selectors = [(\'all\',None),(\'largest\',None),(\'smallest\',None)]\n            selectors += [(\'all\',c) for c in sorted(boundaries)]\n            for target in changed_colors:\n                for selector,boundary_color in selectors:\n                    if time.monotonic() >= finish:\n                        return rules\n                    def fill(grid,bf=bg_fn,d=diagonal,t=target,s=selector,b=boundary_color):\n                        return _fill(grid,bf,d,t,s,b)\n                    if _exact(fill,pairs):\n                        suffix = \'\' if boundary_color is None else f\':boundary{boundary_color}\'\n                        name = f\'relation:enclosed:{bg_name}:{8 if diagonal else 4}:{selector}:to{target}{suffix}\'\n                        cost = 1.6 + .2*(selector != \'all\') + .2*(boundary_color is not None)\n                        rules.append(Rule(name,fill,cost,\'learned_relation_fill\'))\n\n    common_colors = set(int(c) for c in np.unique(pairs[0][\'input\']))\n    for pair in pairs[1:]:\n        common_colors.intersection_update(int(c) for c in np.unique(pair[\'input\']))\n    for moving_color in sorted(common_colors):\n        if time.monotonic() >= finish:\n            break\n        deltas = [_observed_translation(p[\'input\'],p[\'output\'],moving_color) for p in pairs]\n        if any(d is None for d in deltas) or all(d == (0,0) for d in deltas):\n            continue\n        for anchor_color in sorted(common_colors-{moving_color}):\n            if time.monotonic() >= finish:\n                return rules\n            # The anchor itself must remain unchanged in every demonstration.\n            if any(not np.array_equal(p[\'input\']==anchor_color,p[\'output\']==anchor_color) for p in pairs):\n                continue\n            records = [(_bbox_color(p[\'input\'],moving_color),\n                        _bbox_color(p[\'input\'],anchor_color),delta)\n                       for p,delta in zip(pairs,deltas)]\n            row_formulas,col_formulas = _fit_axis(records,0),_fit_axis(records,1)\n            for bg_name,bg_fn in bg_options:\n                if any(bg_fn(p[\'input\']) in (moving_color,anchor_color) for p in pairs):\n                    continue\n                for rf in row_formulas:\n                    for cf in col_formulas:\n                        if time.monotonic() >= finish:\n                            return rules\n                        def translate(grid,bf=bg_fn,m=moving_color,a=anchor_color,r=rf,c=cf):\n                            return _translate(grid,bf,m,a,r,c)\n                        if _exact(translate,pairs):\n                            name = f\'relation:translate:{bg_name}:{moving_color}:anchor{anchor_color}:r{rf}:c{cf}\'\n                            complexity = sum(f != (\'unchanged\',) for f in (rf,cf))\n                            rules.append(Rule(name,translate,2.0+.3*complexity,\'learned_relation_move\'))\n    return rules\n', 'arc2/runner.py': '"""Offline ARC runner: isolated inference, atomic checkpoints, honest exact scoring.\n\nThe evaluator is deliberately outside the solver.  ``test[*].output`` never\ncrosses the worker boundary.  The initial submission already covers every test\ninput; a killed, failed, or budget-skipped worker leaves a valid fallback behind.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom copy import deepcopy\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport math\nimport multiprocessing as mp\nimport os\nfrom pathlib import Path\nimport signal\nimport tempfile\nimport time\nimport traceback\nfrom typing import Any, Callable\n\n\nATTEMPT_FIELDS = ("attempt_1", "attempt_2")\n\n\ndef validate_grid(grid: Any) -> bool:\n    """Validate the serialized competition grid, rejecting bools and floats."""\n    return (\n        isinstance(grid, list)\n        and 1 <= len(grid) <= 30\n        and isinstance(grid[0], list)\n        and 1 <= len(grid[0]) <= 30\n        and all(\n            isinstance(row, list)\n            and len(row) == len(grid[0])\n            and all(type(cell) is int and 0 <= cell <= 9 for cell in row)\n            for row in grid\n        )\n    )\n\n\ndef challenge_for_solver(task: dict[str, Any]) -> dict[str, Any]:\n    """Copy only authorized inference fields; ignore all test-answer fields."""\n    if not isinstance(task, dict) or not isinstance(task.get("train"), list):\n        raise ValueError("Task must contain a train list")\n    if not task["train"]:\n        raise ValueError("Task must contain at least one training example")\n    if not isinstance(task.get("test"), list) or not task["test"]:\n        raise ValueError("Task must contain at least one test input")\n    result: dict[str, Any] = {"train": [], "test": []}\n    for pair in task["train"]:\n        if not isinstance(pair, dict) or not all(validate_grid(pair.get(k)) for k in ("input", "output")):\n            raise ValueError("Invalid training grid")\n        result["train"].append({"input": deepcopy(pair["input"]), "output": deepcopy(pair["output"])})\n    for pair in task["test"]:\n        if not isinstance(pair, dict) or not validate_grid(pair.get("input")):\n            raise ValueError("Invalid test input grid")\n        result["test"].append({"input": deepcopy(pair["input"])})\n    return result\n\n\ndef fallback_attempts(task: dict[str, Any]) -> list[dict[str, Any]]:\n    return [{key: deepcopy(pair["input"]) for key in ATTEMPT_FIELDS} for pair in task["test"]]\n\n\ndef validate_attempts(attempts: Any, expected_count: int) -> bool:\n    return (\n        isinstance(attempts, list)\n        and len(attempts) == expected_count\n        and all(\n            isinstance(item, dict)\n            and set(item) == set(ATTEMPT_FIELDS)\n            and all(validate_grid(item[k]) for k in ATTEMPT_FIELDS)\n            for item in attempts\n        )\n    )\n\n\ndef validate_submission(submission: Any, challenges: dict[str, Any]) -> bool:\n    return (\n        isinstance(submission, dict)\n        and set(submission) == set(challenges)\n        and all(validate_attempts(submission[k], len(challenges[k]["test"])) for k in challenges)\n    )\n\n\ndef atomic_json(path: str | Path, value: Any) -> None:\n    """Replace only a complete, flushed JSON document on the same filesystem."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temp_name = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    try:\n        with os.fdopen(fd, "w", encoding="utf-8") as stream:\n            json.dump(value, stream, ensure_ascii=False, separators=(",", ":"), allow_nan=False)\n            stream.write("\\n")\n            stream.flush()\n            os.fsync(stream.fileno())\n        os.replace(temp_name, path)\n    finally:\n        try:\n            os.unlink(temp_name)\n        except FileNotFoundError:\n            pass\n\n\ndef sha256_file(path: str | Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for part in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef code_manifest() -> dict[str, str]:\n    root = Path(__file__).resolve().parents[1]\n    paths = list((root / "arc2").rglob("*.py"))\n    if (root / "LICENSE").is_file():\n        paths.append(root / "LICENSE")\n    vendor = root / "vendor" / "icecuber"\n    if vendor.is_dir():\n        paths.extend(p for p in vendor.rglob("*") if p.is_file() and (\n            p.suffix in {".cpp", ".hpp", ".h", ".cc"}\n            or p.name in {"LICENSE", "PROVENANCE.md", "SOURCE_MANIFEST.json"}\n        ))\n    return {str(p.relative_to(root)): sha256_file(p) for p in sorted(paths)}\n\n\ndef prepare_requested_backend(backend: str, compile_seconds: float = 180.0) -> dict[str, Any]:\n    """Prepare the optional compiler once, outside all per-task time budgets."""\n    if backend == "python":\n        return {"requested": "python", "effective": "python", "status": "disabled"}\n    if backend != "hybrid":\n        raise ValueError("backend must be python or hybrid")\n    try:\n        from .cpp_backend import prepare_cpp_backend\n        result = prepare_cpp_backend(compile_seconds=compile_seconds)\n        if not isinstance(result, dict):\n            raise ValueError("Invalid C++ preparation result")\n    except Exception as exc:\n        result = {"status": "error", "error_type": type(exc).__name__, "error": str(exc)}\n    return {**result, "requested": "hybrid", "effective": "hybrid" if result.get("status") == "ready" else "python"}\n\n\ndef _solve_worker(task: dict[str, Any], config: dict[str, Any], result_path: str) -> None:\n    # Imported here so runner validation and score computation need no solver.\n    try:\n        from .solver import SolverConfig, solve_task\n        result = solve_task(task, config=SolverConfig(**config))\n        atomic_json(result_path, {"ok": True, "result": result})\n    except BaseException as exc:\n        atomic_json(result_path, {\n            "ok": False,\n            "error_type": type(exc).__name__,\n            "error": str(exc),\n            "traceback": traceback.format_exc(limit=8),\n        })\n\n\ndef _worker_entry(\n    worker_fn: Callable[..., None], task: dict[str, Any], config: dict[str, Any],\n    result_path: str, session_path: str,\n) -> None:\n    """Isolate a POSIX worker and every compiler/solver child in one session."""\n    if os.name == "posix":\n        try:\n            os.setsid()\n            identity = {"pid": os.getpid(), "pgid": os.getpgrp(), "sid": os.getsid(0)}\n            if not identity["pid"] == identity["pgid"] == identity["sid"]:\n                raise RuntimeError("Worker did not become its own process-group leader")\n            atomic_json(session_path, identity)\n        except Exception as exc:\n            # Do not launch external programs without an isolated process group.\n            atomic_json(result_path, {"ok": False, "error_type": "SessionIsolationError", "error": str(exc)})\n            return\n    worker_fn(task, config, result_path)\n\n\ndef _verified_worker_group(process: Any, session_path: Path) -> int | None:\n    """Return only a group established by our wrapper with pgid == worker pid.\n\n    The wrapper records the verified identity before invoking solver code.  The\n    record also permits cleanup after the leader exits but a descendant remains.\n    A currently living PID must still have that same group and session identity.\n    """\n    if os.name != "posix" or process.pid is None or not session_path.is_file():\n        return None\n    try:\n        identity = json.loads(session_path.read_text(encoding="utf-8"))\n        pid = process.pid\n        if not identity.get("pid") == identity.get("pgid") == identity.get("sid") == pid:\n            return None\n        if pid <= 1 or pid == os.getpgrp() or pid == os.getpid():\n            return None\n        try:\n            if os.getpgid(pid) != pid or os.getsid(pid) != pid:\n                return None\n        except ProcessLookupError:\n            # A process group outlives its leader while any child remains in it.\n            pass\n        return pid\n    except (OSError, ValueError, TypeError):\n        return None\n\n\ndef _terminate_worker_tree(process: Any, session_path: Path) -> None:\n    """Terminate a verified worker group, including SIGTERM-resistant children."""\n    group = _verified_worker_group(process, session_path)\n    if group is not None:\n        try:\n            os.killpg(group, signal.SIGTERM)\n        except ProcessLookupError:\n            return\n        process.join(timeout=0.25)\n        # The leader may have exited while a compiler/solver child ignored TERM.\n        # Always send KILL to this already verified group, not only if the leader\n        # is alive. A missing group simply means all descendants already exited.\n        try:\n            os.killpg(group, signal.SIGKILL)\n        except ProcessLookupError:\n            pass\n        process.join(timeout=0.5)\n        return\n    if process.is_alive():\n        process.terminate()\n        process.join(timeout=0.25)\n        if process.is_alive():\n            process.kill()\n            process.join(timeout=0.5)\n\n\ndef run_task_isolated(\n    task: dict[str, Any],\n    config: dict[str, Any],\n    hard_timeout: float,\n    *,\n    worker_fn: Callable[..., None] | None = None,\n) -> dict[str, Any]:\n    """Run one task in a spawn worker with a hard wall-clock cap.\n\n    A file rather than a multiprocessing Queue avoids pipe-buffer deadlocks for\n    large diagnostics. ``worker_fn`` is an injection point for boundary tests.\n    """\n    clean_task = challenge_for_solver(task)\n    fallback = {"attempts": fallback_attempts(clean_task), "diagnostics": {}}\n    if hard_timeout <= 0:\n        fallback["diagnostics"] = {"runner_status": "budget_skipped", "wall_seconds": 0.0}\n        return fallback\n    started = time.monotonic()\n    context = mp.get_context("spawn")\n    process = None\n    tree_cleaned = False\n    with tempfile.TemporaryDirectory(prefix="arc2-worker-") as folder:\n        result_path = Path(folder) / "result.json"\n        session_path = Path(folder) / "worker-session.json"\n        try:\n            process = context.Process(target=_worker_entry, args=(worker_fn or _solve_worker, clean_task, dict(config), str(result_path), str(session_path)))\n            process.start()\n            process.join(timeout=max(0.0, hard_timeout - (time.monotonic() - started)))\n            if process.is_alive():\n                _terminate_worker_tree(process, session_path)\n                tree_cleaned = True\n                fallback["diagnostics"] = {"runner_status": "hard_timeout"}\n                return fallback\n            if process.exitcode != 0 or not result_path.is_file():\n                fallback["diagnostics"] = {"runner_status": "worker_failed", "exit_code": process.exitcode}\n                return fallback\n            packet = json.loads(result_path.read_text(encoding="utf-8"))\n            if not packet.get("ok"):\n                fallback["diagnostics"] = {\n                    "runner_status": "solver_error",\n                    "error_type": packet.get("error_type"),\n                    "error": packet.get("error"),\n                }\n                return fallback\n            result = packet.get("result", {})\n            if not isinstance(result, dict) or not validate_attempts(result.get("attempts"), len(clean_task["test"])):\n                fallback["diagnostics"] = {"runner_status": "invalid_prediction"}\n                return fallback\n            diagnostic = result.get("diagnostics", {})\n            if not isinstance(diagnostic, dict):\n                diagnostic = {}\n            result["diagnostics"] = {**diagnostic, "runner_status": "ok", "wall_seconds": time.monotonic() - started}\n            return result\n        except Exception as exc:\n            fallback["diagnostics"] = {"runner_status": "runner_error", "error_type": type(exc).__name__, "error": str(exc)}\n            return fallback\n        finally:\n            if process is not None and process.pid is not None and not tree_cleaned:\n                _terminate_worker_tree(process, session_path)\n            fallback["diagnostics"]["wall_seconds"] = time.monotonic() - started\n\n\ndef score_submission(submission: dict[str, Any], solutions: dict[str, list[Any]]) -> dict[str, Any]:\n    """Official-style per-test-output exact pass@2 plus stricter task metric."""\n    if set(submission) != set(solutions):\n        raise ValueError("Prediction and solution task IDs differ")\n    task_results: dict[str, Any] = {}\n    correct_outputs = 0\n    total_outputs = 0\n    solved_tasks = 0\n    for task_id, answers in solutions.items():\n        guesses = submission[task_id]\n        if not answers or not all(validate_grid(answer) for answer in answers):\n            raise ValueError(f"Invalid or missing solution for {task_id}")\n        if not validate_attempts(guesses, len(answers)):\n            raise ValueError(f"Invalid prediction for {task_id}")\n        hits = [any(item[field] == answer for field in ATTEMPT_FIELDS) for item, answer in zip(guesses, answers)]\n        first_hits = [item["attempt_1"] == answer for item, answer in zip(guesses, answers)]\n        correct_outputs += sum(hits)\n        total_outputs += len(hits)\n        solved_tasks += int(all(hits))\n        task_results[task_id] = {\n            "correct_outputs": sum(hits), "total_outputs": len(hits),\n            "pass_at_2_by_output": hits, "pass_at_1_by_output": first_hits,\n            "all_outputs_solved": all(hits),\n        }\n    return {\n        "correct_outputs": correct_outputs,\n        "total_outputs": total_outputs,\n        "output_pass_at_2": correct_outputs / total_outputs if total_outputs else 0.0,\n        "correct_outputs_attempt_1": sum(sum(t["pass_at_1_by_output"]) for t in task_results.values()),\n        "solved_tasks": solved_tasks,\n        "total_tasks": len(solutions),\n        "task_pass_at_2": solved_tasks / len(solutions) if solutions else 0.0,\n        "tasks": task_results,\n    }\n\n\ndef run_challenges(\n    challenges: dict[str, Any],\n    output_path: str | Path,\n    *,\n    seconds_per_task: float = 5.0,\n    total_seconds: float = 39600.0,\n    diagnostics_path: str | Path | None = None,\n    solver_config: dict[str, Any] | None = None,\n    backend_info: dict[str, Any] | None = None,\n    hard_timeout_slack: float = 2.0,\n    task_runner: Callable[..., dict[str, Any]] = run_task_isolated,\n    progress: bool = True,\n) -> tuple[dict[str, Any], dict[str, Any]]:\n    if not isinstance(challenges, dict) or not challenges or not all(isinstance(k, str) for k in challenges):\n        raise ValueError("Challenges must be a nonempty task-ID dictionary")\n    if not math.isfinite(seconds_per_task) or seconds_per_task <= 0:\n        raise ValueError("seconds_per_task must be positive and finite")\n    if not math.isfinite(total_seconds) or total_seconds < 0:\n        raise ValueError("total_seconds must be nonnegative and finite")\n    if not math.isfinite(hard_timeout_slack) or hard_timeout_slack < 0:\n        raise ValueError("hard_timeout_slack must be nonnegative and finite")\n    clean = {task_id: challenge_for_solver(task) for task_id, task in challenges.items()}\n    submission = {task_id: fallback_attempts(task) for task_id, task in clean.items()}\n    config = {"time_limit": seconds_per_task, **(solver_config or {})}\n    # The explicit CLI/task budget wins over an accidental config override.\n    config["time_limit"] = seconds_per_task\n    diagnostic: dict[str, Any] = {\n        "created_utc": datetime.now(timezone.utc).isoformat(),\n        "solver_config": config,\n        "backend": backend_info or {"requested": "python-api", "effective": "unverified" if config.get("use_cpp") else "python", "status": "not_prepared_by_runner" if config.get("use_cpp") else "disabled"},\n        "total_seconds_budget": total_seconds,\n        "hard_timeout_slack": hard_timeout_slack,\n        "code_sha256": code_manifest(),\n        "answer_visibility": "test outputs stripped before solver worker",\n        "fallback": "identity input copied to both attempts",\n        "tasks": {},\n    }\n    atomic_json(output_path, submission)\n    if diagnostics_path is not None:\n        atomic_json(diagnostics_path, diagnostic)\n    started = time.monotonic()\n    deadline = started + total_seconds\n    task_ids = sorted(clean)\n    for index, task_id in enumerate(task_ids):\n        remaining = deadline - time.monotonic()\n        if remaining <= 0:\n            result = {"attempts": submission[task_id], "diagnostics": {"runner_status": "budget_skipped", "wall_seconds": 0.0}}\n        else:\n            hard_cap = min(seconds_per_task + hard_timeout_slack, remaining)\n            task_config = dict(config)\n            task_config["time_limit"] = min(seconds_per_task, max(0.01, hard_cap - min(hard_timeout_slack, hard_cap * 0.25)))\n            try:\n                result = task_runner(clean[task_id], task_config, hard_cap)\n                if not validate_attempts(result.get("attempts"), len(clean[task_id]["test"])):\n                    result = {"attempts": submission[task_id], "diagnostics": {"runner_status": "invalid_prediction"}}\n            except Exception as exc:\n                result = {"attempts": submission[task_id], "diagnostics": {"runner_status": "runner_error", "error": str(exc)}}\n        submission[task_id] = result["attempts"]\n        diagnostic["tasks"][task_id] = result.get("diagnostics", {})\n        diagnostic["completed_tasks"] = index + 1\n        diagnostic["wall_seconds"] = time.monotonic() - started\n        atomic_json(output_path, submission)\n        if diagnostics_path is not None:\n            atomic_json(diagnostics_path, diagnostic)\n        if progress:\n            state = diagnostic["tasks"][task_id].get("runner_status", "unknown")\n            print(f"[{index + 1}/{len(task_ids)}] {task_id}: {state}", flush=True)\n    if not validate_submission(submission, clean):\n        raise RuntimeError("Internal submission schema invariant failed")\n    diagnostic["submission_sha256"] = sha256_file(output_path)\n    diagnostic["wall_seconds"] = time.monotonic() - started\n    if diagnostics_path is not None:\n        atomic_json(diagnostics_path, diagnostic)\n    return submission, diagnostic\n\n\ndef evaluate_directory(\n    tasks_directory: str | Path,\n    report_path: str | Path,\n    *,\n    limit: int | None = None,\n    offset: int = 0,\n    seconds_per_task: float = 5.0,\n    total_seconds: float = 39600.0,\n    solver_config: dict[str, Any] | None = None,\n    backend_info: dict[str, Any] | None = None,\n    progress: bool = True,\n) -> dict[str, Any]:\n    if offset < 0 or (limit is not None and limit <= 0):\n        raise ValueError("offset must be nonnegative; limit must be positive")\n    root = Path(tasks_directory)\n    paths = sorted(root.glob("*.json"))[offset:]\n    if limit is not None:\n        paths = paths[:limit]\n    if not paths:\n        raise ValueError("No task JSON files selected")\n    challenges: dict[str, Any] = {}\n    solutions: dict[str, list[Any]] = {}\n    data_manifest: dict[str, str] = {}\n    for path in paths:\n        task = json.loads(path.read_text(encoding="utf-8"))\n        challenges[path.stem] = challenge_for_solver(task)\n        answers = [pair.get("output") for pair in task["test"]]\n        if not all(validate_grid(answer) for answer in answers):\n            raise ValueError(f"Missing or invalid test answer in {path.name}")\n        solutions[path.stem] = answers\n        data_manifest[path.name] = sha256_file(path)\n    report_path = Path(report_path)\n    submission_path = report_path.with_name(report_path.stem + ".submission.json")\n    diagnostics_path = report_path.with_name(report_path.stem + ".diagnostics.json")\n    submission, diagnostics = run_challenges(\n        challenges, submission_path, seconds_per_task=seconds_per_task,\n        total_seconds=total_seconds, diagnostics_path=diagnostics_path,\n        solver_config=solver_config, backend_info=backend_info, progress=progress,\n    )\n    report = {\n        "created_utc": datetime.now(timezone.utc).isoformat(),\n        "dataset_directory": str(root.resolve()),\n        "selection": {"ordering": "lexicographic task filename", "offset": offset, "limit": limit},\n        "dataset_sha256": data_manifest,\n        "solver_config": diagnostics["solver_config"],\n        "backend": diagnostics["backend"],\n        "code_sha256": diagnostics["code_sha256"],\n        "answer_visibility": diagnostics["answer_visibility"],\n        "submission_file": submission_path.name,\n        "submission_sha256": diagnostics["submission_sha256"],\n        "diagnostics_file": diagnostics_path.name,\n        "wall_seconds": diagnostics["wall_seconds"],\n        "metrics": score_submission(submission, solutions),\n    }\n    atomic_json(report_path, report)\n    return report\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Offline ARC-AGI-2 symbolic candidate")\n    commands = parser.add_subparsers(dest="command", required=True)\n    for name in ("solve", "evaluate"):\n        command = commands.add_parser(name)\n        command.add_argument("--output", required=True, type=Path)\n        command.add_argument("--seconds-per-task", type=float, default=8.0)\n        command.add_argument("--total-seconds", type=float, default=39600.0)\n        command.add_argument("--max-states", type=int, default=4000)\n        command.add_argument("--beam-width", type=int, default=32)\n        command.add_argument("--max-depth", type=int, default=3)\n        command.add_argument("--backend", choices=("hybrid", "python"), default="hybrid")\n        command.add_argument("--cpp-fraction", type=float, default=0.6)\n        command.add_argument("--cpp-memory-mb", type=int, default=768)\n        command.add_argument("--compile-seconds", type=float, default=180.0)\n        command.add_argument("--quiet", action="store_true")\n        if name == "solve":\n            command.add_argument("--challenges", required=True, type=Path)\n            command.add_argument("--diagnostics", type=Path)\n        else:\n            command.add_argument("--tasks", required=True, type=Path)\n            command.add_argument("--limit", type=int)\n            command.add_argument("--offset", type=int, default=0)\n    args = parser.parse_args(argv)\n    if args.max_states <= 0 or args.beam_width <= 0 or args.max_depth < 0:\n        parser.error("max-states and beam-width must be positive; max-depth must be nonnegative")\n    if not 0 < args.cpp_fraction < 1 or args.cpp_memory_mb < 128 or not math.isfinite(args.compile_seconds) or args.compile_seconds <= 0:\n        parser.error("cpp-fraction must be in (0,1), cpp-memory-mb >=128, and compile-seconds positive and finite")\n    challenges = None\n    if args.command == "solve":\n        challenges = json.loads(args.challenges.read_text(encoding="utf-8"))\n        if not isinstance(challenges, dict) or not challenges or not all(isinstance(k, str) for k in challenges):\n            parser.error("Challenges must be a nonempty task-ID dictionary")\n        clean = {task_id: challenge_for_solver(task) for task_id, task in challenges.items()}\n        # Preserve a full valid submission even if the job is stopped in compile.\n        atomic_json(args.output, {task_id: fallback_attempts(task) for task_id, task in clean.items()})\n    backend_info = prepare_requested_backend(args.backend, args.compile_seconds)\n    if not args.quiet or backend_info["effective"] != args.backend:\n        print("Backend: " + json.dumps(backend_info, ensure_ascii=False), flush=True)\n    config = {\n        "max_states": args.max_states, "beam_width": args.beam_width, "max_depth": args.max_depth,\n        "use_cpp": backend_info["effective"] == "hybrid",\n        "cpp_fraction": args.cpp_fraction, "cpp_memory_mb": args.cpp_memory_mb,\n    }\n    if args.command == "solve":\n        run_challenges(\n            challenges, args.output, seconds_per_task=args.seconds_per_task,\n            total_seconds=args.total_seconds, diagnostics_path=args.diagnostics,\n            solver_config=config, backend_info=backend_info, progress=not args.quiet,\n        )\n    else:\n        report = evaluate_directory(\n            args.tasks, args.output, limit=args.limit, offset=args.offset,\n            seconds_per_task=args.seconds_per_task, total_seconds=args.total_seconds,\n            solver_config=config, backend_info=backend_info, progress=not args.quiet,\n        )\n        metrics = {key: value for key, value in report["metrics"].items() if key != "tasks"}\n        print(json.dumps(metrics, ensure_ascii=False, indent=2))\n    return 0\n\n', 'arc2/solver.py': '"""Bounded symbolic synthesis with exact observational-state deduplication.\n\nEvery fitted rule sees demonstration pairs only.  A search state includes ALL\ndemonstration and test inputs: equality on demonstrations alone is not a sound\nreason to merge programs that disagree on a test input.  Complete grid bytes\nare compared, so there is no lossy embedding or hash-collision assumption.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass, replace\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\n\nfrom .grid import as_grid, key, valid_grid\nfrom .operators import primitive_rules, constant_rule\nfrom .types import Rule\n\n\n@dataclass(frozen=True)\nclass SolverConfig:\n    time_limit: float = 5.0\n    max_states: int = 2000\n    beam_width: int = 32\n    max_depth: int = 3\n    max_candidates: int = 96\n    use_cpp: bool = False\n    cpp_fraction: float = 0.6\n    cpp_memory_mb: int = 768\n\n\n@dataclass\nclass State:\n    grids: tuple[np.ndarray, ...]\n    path: tuple[str, ...]\n    families: tuple[str, ...]\n    cost: float\n    score: float\n    exact: bool\n\n\ndef state_key(grids) -> tuple[bytes, ...]:\n    return tuple(key(a) for a in grids)\n\n\ndef _agreement(pred, target):\n    if pred.shape != target.shape:\n        # Shape receives a small, nonnegative heuristic; never exact credit.\n        h,w=pred.shape; hh,ww=target.shape\n        return .16*(min(h,hh)/max(h,hh)+min(w,ww)/max(w,ww))/2\n    same=pred==target\n    if bool(np.all(same)):\n        return 1.0\n    colors=np.unique(target)\n    balanced=sum(float(same[target==c].mean()) for c in colors)/len(colors)\n    return .2+.4*float(same.mean())+.4*balanced\n\n\ndef _fitness(grids, targets):\n    scores=[_agreement(a,b) for a,b in zip(grids,targets)]\n    return sum(scores)/len(scores), all(x==1.0 for x in scores)\n\n\ndef _apply(rule, grids, errors):\n    values=[]\n    try:\n        for a in grids:\n            b=rule.fn(a)\n            if b is None or not valid_grid(b):\n                return None\n            b=np.array(b,dtype=np.uint8,copy=True)\n            b.flags.writeable=False\n            values.append(b)\n    except Exception as exc:\n        # A failed rule cannot poison the whole submission. Unexpected failures\n        # are visible in the report instead of being silently counted as fits.\n        k=f"{rule.name}: {type(exc).__name__}: {str(exc)[:100]}"\n        if len(errors)<12 or k in errors:\n            errors[k]=errors.get(k,0)+1\n        return None\n    return tuple(values)\n\n\ndef _color_maps(grids, targets):\n    if any(a.shape!=b.shape for a,b in zip(grids,targets)):\n        return []\n    mapping={}\n    for a,b in zip(grids,targets):\n        for color in np.unique(a):\n            vals=np.unique(b[a==color])\n            if len(vals)!=1:\n                return []\n            old=mapping.setdefault(int(color),int(vals[0]))\n            if old!=int(vals[0]):\n                return []\n    if all(k==v for k,v in mapping.items()):\n        return []\n    lut=np.arange(10,dtype=np.uint8)\n    for k,v in mapping.items():\n        lut[k]=v\n    desc=",".join(f"{k}:{v}" for k,v in sorted(mapping.items()) if k!=v)\n    cost=.7+.12*sum(k!=v for k,v in mapping.items())\n    rules=[Rule(f"recolor[{desc}]",lambda a,m=lut:m[a],cost,"recolor")]\n    # When demonstrations establish a binary mask, unknown colors may mean\n    # foreground. Retain this as a separate, more expensive hypothesis.\n    if 0 in mapping and len(set(v for k,v in mapping.items() if k!=0))==1:\n        fg=next(v for k,v in mapping.items() if k!=0)\n        m=lut.copy()\n        for c in range(1,10):\n            if c not in mapping: m[c]=fg\n        if not np.array_equal(m,lut):\n            rules.append(Rule(f"recolor_foreground[{desc}]",lambda a,m=m:m[a],cost+.4,"recolor"))\n    return rules\n\n\ndef _beam(states, width):\n    ranked=sorted(states,key=lambda s:(-s.score+.012*s.cost,s.cost,s.path))\n    # Reserve part of the beam for structurally different intermediates.\n    result=[]; chosen=set(); groups=set()\n    for s in ranked:\n        tag=(tuple(a.shape for a in s.grids),s.families[-1] if s.families else "identity")\n        if tag not in groups:\n            result.append(s); chosen.add(id(s)); groups.add(tag)\n            if len(result)>=max(1,width//2): break\n    for s in ranked:\n        if len(result)>=width: break\n        if id(s) not in chosen:\n            result.append(s); chosen.add(id(s))\n    return result\n\n\ndef _rank_outputs(states, test_index):\n    """Keep separate output events; aliases within a rule family do not vote twice."""\n    groups={}\n    for s in states:\n        k=key(s.grids[test_index])\n        group=groups.setdefault(k,{"grid":s.grids[test_index],"states":[],"votes":{}})\n        group["states"].append(s)\n    ranked=[]\n    for g in groups.values():\n        best=min(g["states"],key=lambda s:(not s.exact,-s.score,s.cost,s.path))\n        for s in g["states"]:\n            if best.exact and not s.exact:\n                continue\n            family=tuple(sorted(set(s.families)))\n            weight=math.exp(-.55*s.cost)\n            g["votes"][family]=max(weight,g["votes"].get(family,0))\n        support=sum(sorted(g["votes"].values(),reverse=True)[:4])\n        # Training agreement dominates approximate candidates. Among exact\n        # hypotheses, simple independent derivations provide limited support.\n        rank=(int(best.exact),best.score,support,-best.cost)\n        ranked.append((rank,best,g["grid"]))\n    ranked.sort(key=lambda t:(t[0],key(t[2])),reverse=True)\n    return ranked\n\n\ndef _solve_symbolic(task, config=None):\n    config=config or SolverConfig()\n    if isinstance(config,dict): config=SolverConfig(**config)\n    if (not math.isfinite(config.time_limit) or config.time_limit<=0 or config.max_states<1\n            or config.beam_width<1 or config.max_depth<0 or config.max_candidates<1):\n        raise ValueError("Positive finite time/state/beam/candidate budgets and nonnegative depth are required")\n    start=time.monotonic(); deadline=start+config.time_limit\n    # Reconstruct the accepted input explicitly. Extra keys, including test\n    # answers in public files, are never forwarded to a hypothesis factory.\n    train=[{"input":as_grid(p["input"]),"output":as_grid(p["output"])} for p in task["train"]]\n    tests=[as_grid(p["input"]) for p in task["test"]]\n    if not train or not tests: raise ValueError("At least one train and test input is required")\n    targets=tuple(p["output"] for p in train)\n    grids=tuple([p["input"] for p in train]+tests)\n    for a in grids: a.flags.writeable=False\n    score,exact=_fitness(grids,targets)\n    initial=State(grids,(),(),0.0,score,exact)\n    seen={state_key(grids):0.0}\n    exact_states=[]; partial_states=[initial]; errors={}\n    stats={"operator_applications":0,"duplicate_states":0,"generated_states":1,\n           "exact_training_programs":0,"peak_frontier":1}\n    if exact: exact_states.append(initial)\n\n    base_rules=primitive_rules(train)\n    object_rules=[]; direct_rules=[]\n    stages={}\n    stage_start=time.monotonic()\n    from .objects import object_rules as make_objects\n    object_rules=make_objects(train,deadline=min(deadline,start+config.time_limit*.25))\n    stages["object_rule_generation_seconds"]=time.monotonic()-stage_start\n    stage_start=time.monotonic()\n    from .patterns import pattern_rules\n    direct_rules=pattern_rules(train,deadline=min(deadline,start+config.time_limit*.50))\n    stages["pattern_rule_generation_seconds"]=time.monotonic()-stage_start\n    stage_start=time.monotonic()\n    from .relations import relation_rules\n    direct_rules+=relation_rules(train,deadline=min(deadline,start+config.time_limit*.60))\n    stages["relation_rule_generation_seconds"]=time.monotonic()-stage_start\n    const=constant_rule(train)\n    if const: direct_rules.append(const)\n    # Direct fitted rules are assessed on every demonstration again here.\n    all_first=direct_rules+base_rules+object_rules\n    composed=base_rules+[r for r in object_rules if r.family!="object_recolor"]\n\n    def remember(state):\n        stats["generated_states"]+=1\n        if state.exact:\n            stats["exact_training_programs"]+=1\n            exact_states.append(state)\n        else:\n            partial_states.append(state)\n\n    def recolor_terminal(state):\n        for rule in _color_maps(state.grids,targets):\n            if time.monotonic()>=deadline or len(seen)>=config.max_states: break\n            result=_apply(rule,state.grids,errors)\n            stats["operator_applications"]+=1\n            if result is None: continue\n            k=state_key(result); cost=state.cost+rule.cost\n            if k in seen and seen[k]<=cost:\n                stats["duplicate_states"]+=1; continue\n            seen[k]=cost\n            fit,ex=_fitness(result,targets)\n            remember(State(result,state.path+(rule.name,),state.families+(rule.family,),cost,fit,ex))\n\n    recolor_terminal(initial)\n    frontier=[initial]\n    for depth in range(1,config.max_depth+1):\n        next_states=[]\n        for parent in frontier:\n            rules=all_first if depth==1 else composed\n            for rule in rules:\n                if time.monotonic()>=deadline or len(seen)>=config.max_states: break\n                # Avoid immediate repeated involutions without reducing the\n                # language\'s useful non-involutive repetitions.\n                if parent.path and parent.path[-1]==rule.name and rule.family=="geometry": continue\n                result=_apply(rule,parent.grids,errors)\n                stats["operator_applications"]+=1\n                if result is None: continue\n                k=state_key(result); cost=parent.cost+rule.cost\n                if k in seen and seen[k]<=cost:\n                    stats["duplicate_states"]+=1; continue\n                seen[k]=cost\n                fit,ex=_fitness(result,targets)\n                state=State(result,parent.path+(rule.name,),parent.families+(rule.family,),cost,fit,ex)\n                remember(state)\n                recolor_terminal(state)\n                if not ex:\n                    next_states.append(state)\n            if time.monotonic()>=deadline or len(seen)>=config.max_states: break\n        stats["peak_frontier"]=max(stats["peak_frontier"],len(next_states))\n        frontier=_beam(next_states,config.beam_width)\n        if not frontier or time.monotonic()>=deadline or len(seen)>=config.max_states: break\n    # Removing duplicate final predictions does not change the transition\n    # system. A bounded pool avoids spending unbounded time on vote ranking.\n    exact_states.sort(key=lambda s:(s.cost,s.path))\n    unique_exact={}\n    for s in exact_states:\n        unique_exact.setdefault(state_key(s.grids[len(train):]),s)\n    exact_pool=list(unique_exact.values())[:config.max_candidates]\n    partial_states.sort(key=lambda s:(-s.score,s.cost,s.path))\n    partial_pool=partial_states[:max(24,config.max_candidates//2)]\n    attempts=[]; selections=[]\n    for index in range(len(tests)):\n        ranking=_rank_outputs(exact_pool+partial_pool,len(train)+index)\n        chosen=ranking[:2]\n        if len(chosen)==1: chosen=chosen*2\n        attempts.append({f"attempt_{j+1}":entry[2].tolist() for j,entry in enumerate(chosen)})\n        selections.append([{"program":list(entry[1].path) or ["identity"],\n                            "training_exact":entry[1].exact,"training_agreement":round(entry[1].score,6),\n                            "cost":round(entry[1].cost,4)} for entry in chosen])\n    elapsed=time.monotonic()-start\n    return {"attempts":attempts,"diagnostics":{\n        "solver":"arc2-symbolic-candidate-v1","config":asdict(config),"elapsed_seconds":elapsed,\n        "stopped_on_time":time.monotonic()>=deadline,"stopped_on_state_budget":len(seen)>=config.max_states,\n        "unique_exact_training_predictions":len(unique_exact),\n        "rules":{"primitive":len(base_rules),"object":len(object_rules),"pattern":len(direct_rules)},\n        "selection":selections,"rule_errors":errors,**stats,**stages,\n        "equivalence_scope":"full grids on every train input AND every test input",\n        "test_output_access":False,\n    }}\n\n\ndef solve_task(task, config=None):\n    """Public API. C++ is optional and must be prepared outside task budgets."""\n    config=config or SolverConfig()\n    if isinstance(config,dict): config=SolverConfig(**config)\n    if not 0<config.cpp_fraction<1 or config.cpp_memory_mb<128:\n        raise ValueError("cpp_fraction must be in (0,1) and cpp_memory_mb >= 128")\n    if not config.use_cpp:\n        return _solve_symbolic(task,config)\n    start=time.monotonic()\n    path=os.environ.get("ARC2_CPP_BINARY","")\n    if not path or not Path(path).is_file():\n        result=_solve_symbolic(task,replace(config,use_cpp=False))\n        result["diagnostics"]["cpp"]={"status":"not_prepared","used":False}\n        result["diagnostics"]["requested_config"]=asdict(config)\n        return result\n    symbolic=_solve_symbolic(task,replace(config,use_cpp=False,time_limit=config.time_limit*(1-config.cpp_fraction)))\n    remaining=max(0.0,config.time_limit-(time.monotonic()-start))\n    from .cpp_backend import solve_cpp\n    # Rebuild the payload here as well; direct API callers may pass full public\n    # tasks with answers. Only the demonstrations and test inputs go to C++.\n    clean={"train":[{"input":p["input"],"output":p["output"]} for p in task["train"]],\n           "test":[{"input":p["input"]} for p in task["test"]]}\n    try:\n        cpp=solve_cpp(clean,seconds=remaining,max_memory_mb=config.cpp_memory_mb)\n    except Exception as exc:\n        cpp={"predictions":[],"diagnostics":{"status":"error","error":f"{type(exc).__name__}: {exc}"}}\n    from .portfolio import merge_predictions\n    attempts,selections=merge_predictions(symbolic,cpp,len(task["train"]))\n    return {"attempts":attempts,"diagnostics":{\n        "solver":"arc2-portfolio-candidate-v1","config":asdict(config),\n        "elapsed_seconds":time.monotonic()-start,"selection":selections,\n        "symbolic":symbolic["diagnostics"],"symbolic_attempts":symbolic["attempts"],\n        "cpp":cpp.get("diagnostics",{}),"cpp_predictions":cpp.get("predictions",[]),\n        "cpp_training_exact":cpp.get("training_exact",[]),\n        "test_output_access":False,\n    }}\n', 'arc2/types.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Callable\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Rule:\n    """A pure grid transformation; None means the rule is inapplicable."""\n\n    name: str\n    fn: Callable[[np.ndarray], np.ndarray | None] = field(compare=False, repr=False)\n    cost: float = 1.0\n    family: str = "generic"\n\n', 'LICENSE': 'MIT No Attribution\n\nCopyright 2026\n\nPermission is hereby granted, free of charge, to any person obtaining a copy of this\nsoftware and associated documentation files (the "Software"), to deal in the\nSoftware without restriction, including without limitation the rights to use,\ncopy, modify, merge, publish, distribute, sublicense, and/or sell copies of the\nSoftware, and to permit persons to whom the Software is furnished to do so.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS\nFOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR\nCOPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER\nIN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION\nWITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.\n', 'vendor/icecuber/LICENSE': 'MIT License\n\nCopyright (c) 2020 top-quarks\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'vendor/icecuber/PROVENANCE.md': "# Optional C++ DSL backend provenance\n\nAuthor: Johan S. Wind (`top-quarks`), *DSL solution to the ARC challenge* (2020).\n\n- Upstream: https://github.com/top-quarks/ARC-solution\n- Pinned commit: `9407072659de1270358c2ba34c527785214dd68b`\n- License: MIT; upstream copyright and permission notice are in `LICENSE`.\n- `SOURCE_MANIFEST.json` records the SHA-256 of each original file before the\n  modifications below. It is provenance information, not a claim that the\n  modified files retain the original hashes.\n\nOnly generic C++ source and headers are included. No dataset, output answers,\nmodel weights, compiled executable, precompiled header, or historical Python\nrunner is included. In particular, upstream `src/tasks.cpp`, `src/evals.cpp`,\nand their headers, which contain historical task-index-specific examples, are\nexcluded. Their evaluation entry points are not reachable or linked.\n\n## Changes from the pinned source\n\n1. Replaced `src/runner.cpp` with a compact inference-only adaptation of the\n   upstream generic pipeline: normalize demonstration colors, infer sizes,\n   enumerate pieces, compose pieces, deduce outer products, rank candidates.\n   It reads only the test input. It removes evaluation/verdict routines, hidden\n   output comparisons, visualization plumbing, and historical alternative\n   entry points. Search depths 1, 2, and 3 preserve upstream cost thresholds\n   10, 20, and 30. The three best distinct candidates are emitted per depth.\n2. The inference runner additionally exports an explicit demonstration-match\n   count for each candidate, by reconstructing that candidate's predictions on\n   the original demonstration inputs and comparing their original outputs.\n   This is empirical fit, not a generalization guarantee. The upstream raw\n   score is also retained.\n3. Restricted `src/read.cpp` to `./dataset/` within the wrapper's isolated\n   temporary working directory. Removed its implicit legacy Kaggle path and\n   sorted filenames. Capped a positive file-count limit safely.\n4. The Python adapter supplies one generated `00000000.json`, retaining only\n   demonstration input/output pairs and test inputs. No caller task ID or test\n   output is passed. Upstream's reader already supports absent test outputs;\n   its internal invalid-image placeholder is never passed into the search.\n5. Removed the unused `cheatSize` helper and `evalNormalizeRigid` debug\n   entry point and their declarations. Removed stale commented entry points\n   from `main.cpp`. These functions were unreachable from inference.\n6. Omitted unused progress and evaluation-scoring translation units. Compiler\n   flags are C++17 and `-O2`, with filesystem linkage, and compilation occurs\n   in a temporary cache. No architecture-specific flags or external downloads\n   are required at inference time.\n\nThe Python adapter runs increasing depths and preserves files from completed\ndepths when a later process times out. Subprocess wall-time and POSIX memory/CPU\nlimits are enforced. Its C++ subprocess remains in the caller's process group\nso an outer worker can terminate the complete group. Compilation is a separate\npreparation step; this backend reports unavailable if preparation was not done.\n\n## Intended use and limits\n\nThis is an optional independent candidate generator for a two-attempt ensemble.\nThe upstream README's historical ARC results are not ARC-AGI-2 results. No\naccuracy on the ARC-AGI-2 public evaluation or Kaggle private tasks is implied\nby including this backend. Fixed source revisions, offline execution, and\ndemonstration fit establish reproducibility, not competitive performance.\n", 'vendor/icecuber/SOURCE_MANIFEST.json': '{\n  "repository": "https://github.com/top-quarks/ARC-solution",\n  "commit": "9407072659de1270358c2ba34c527785214dd68b",\n  "license": "MIT",\n  "original_files_sha256": {\n    "src/main.cpp": "b5a6eacfef80f74e687388ced558a89f3118e6bb9bd51e02b7fe1faa3a9681d7",\n    "src/read.cpp": "1ab1854de355f802d8f1d1e029e875fd24a7f36b8aa0028929c6eccc51e5ed27",\n    "src/core_functions.cpp": "7e49137de5a631359659431c4f936428d4b1861dd09d66ca1ea3e93d3bc3f9d0",\n    "src/image_functions.cpp": "af7987d0beeb9ecd68f021de2507a8578f06e743c5825446c201f202407ad006",\n    "src/image_functions2.cpp": "992395d07723147f96760e5af62e0438dae77ce68758e7a874ba79bac170cc1f",\n    "src/visu.cpp": "b1c97aa69dfce677563a5a4c0d302b68d21d9fadf5ba6ecb2bd1e8275c70ca77",\n    "src/normalize.cpp": "d3314d9f38f13ccde54ee53b3c1a263e332dc0e8b4576d77713dd88feaa05403",\n    "src/runner.cpp": "bbf7df86444cdf7c0dac5b123158996e33f7bb0e337fac3a4069fa797888a193",\n    "src/brute2.cpp": "6fa3bfa35e75e1863162d46514fdc8b4a97491f7ad8df599534c0b0873beca63",\n    "src/deduce_op.cpp": "bff9c1ac75753f45ae0b27fdf32f7d7b2c0e84db2f0be8c5b4683ebf5acb833d",\n    "src/pieces.cpp": "a158f8b270be4d23c2991968e0c90c808e0fcbe48a9af1208dc74d299998669f",\n    "src/compose2.cpp": "444630213f756932430e54b9bfc441d20bbc3a5bfcee336ce87b10f32f340885",\n    "src/brute_size.cpp": "7faba5035a4e0960201602deeea4f0b97ec69754ff15098b06e26ef12c29c908",\n    "src/efficient.cpp": "4e6578fb19c8e2f18a5387c8ca9a7e794ce1c0ec97459f9a9171e66a9c279ada",\n    "headers/brute2.hpp": "ad0685484e4c7b79595b09ac3b8845155e2022621dcd84e1cec6dc954670987a",\n    "headers/brute_size.hpp": "3b715a17a4f13004e0d6635ba0f1f541e68d834e072d7c6564abbd1eebd80110",\n    "headers/compose2.hpp": "c6cfb87d5a6ffdedb08f3b658a9bbf0225d8e38cf9ecfd3df3fbd489ac3aac13",\n    "headers/core_functions.hpp": "063c3740f2f77663460dfc2e0378d154dc1c0845acb6fc214b9076a849894bed",\n    "headers/deduce_op.hpp": "64de804365cfee0ae68373b56421ff76987f01538c0b6b5808e1c8f4fd0d6ec0",\n    "headers/efficient.hpp": "7b616a16bebad98a028668ddce697ace7d69e72ae10c411c2940b311d6f628f7",\n    "headers/image_functions.hpp": "34da8c28fd433855bdd9e67846068fa7f0af9b7afe7f8c6f0606d1021f50f75d",\n    "headers/image_functions2.hpp": "5cf1215715a7bba3eec078e70b39fbdb3e5e9569cb25f8abad22169022010e98",\n    "headers/normalize.hpp": "23c28099dc264fc1a4576f176ae5bab3609cd8407a102ef810f7942c7fd69aab",\n    "headers/pieces.hpp": "795bb846b274c77c9738598f48445f64b60841241c5a398aedd8f15c449760da",\n    "headers/precompiled_stl.hpp": "f18c6a0ba94ab95e534221209a45086c1caa612aa48e4da23d61066abd6694a5",\n    "headers/read.hpp": "956b869f3484fcc192841d8436a6006e41e7f9331eed66f4ff2c87edf1983e42",\n    "headers/runner.hpp": "a0bfc8f32e8521605b3ea844ad161fc5376a4bac515a122cde2343dc6ce3ad0b",\n    "headers/spec.hpp": "8bfbe9109ac28f36c7b5792aa217975e6f09fed4803ec6fb8aff713be316d723",\n    "headers/store.hpp": "7264a93dbcc6548f164291a53ea70211ccabecc862e4a4b132c804d82be12224",\n    "headers/timer.hpp": "bdff9329afd807cbb729edb181fb872b3bd1075ff3771821fb10bc8add405305",\n    "headers/utils.hpp": "0ea4c63cacaeff8d3c77e80b6a88e4a76ce5b4b6cfb9ecfb33c3a8e6c787128f",\n    "headers/visu.hpp": "db797783e9745f44a07388a9363a4000d46936d9dfd9e68559613178853be991",\n    "LICENSE": "c1bce530447da29e46a0b9055dab7c1ac04692889144530efbd0edbcbfff36e0"\n  }\n}\n', 'vendor/icecuber/headers/brute2.hpp': '//#include <ext/pb_ds/assoc_container.hpp>\n//using namespace __gnu_pbds;\n\ndouble now();\n\nstruct State {\n  vImage vimg;\n  int depth;\n  bool isvec;\n  State() {}\n  State(vImage_ vimg_, bool isvec_, int depth_) : vimg(vimg_), isvec(isvec_), depth(depth_) {}\n  ull hash() const {\n    ull r = isvec;\n    for (Image_ img : vimg) {\n      r += hashImage(img)*123413491;\n    }\n    return r;\n  }\n};\n\n\n#include "efficient.hpp"\n\n\n\nstruct Node {\n  State state;\n  //vImage vimg;\n  //bool isvec;\n  vector<pair<int,int>> child;\n  //int depth;\n  int par, pfi;\n  bool freed, ispiece;\n  Node() {\n    freed = ispiece = false;\n  }\n  Node(vImage_ vimg_, bool isvec_, int depth_, int par_ = -1, int pfi_ = -1) : par(par_), pfi(pfi_) {\n    freed = ispiece = false;\n    state.vimg = vimg_;\n    state.isvec = isvec_;\n    state.depth = depth_;\n  }\n};\n\n\nstruct Functions3 {\n  vector<int> listed, cost;\n  vector<string> names;\n  vector<function<bool(const State&,State&)>> f_list;\n\n  void add(const string& name, int cost, const function<bool(const State&,State&)>&func, int list);\n  void add(string name, int cost, const function<Image(Image_)>&f, int list = 1);\n  void add(string name, int cost, const function<vImage(Image_)>&f, int list = 1);\n  void add(string name, int cost, const function<Image(vImage_)>&f, int list = 1);\n  void add(string name, int cost, const function<vImage(vImage_)>&f, int list = 1);\n  void add(const vector<point>&sizes, string name, int cost, const function<Image(Image_,Image_)>&f, int list = 1);\n  string getName(int fi);\n  int findfi(string name);\n};\n\n\n\nstruct DAG {\n  Functions3 funcs;\n  //vector<Node> node;\n  TinyNodeBank tiny_node;\n  int givens;\n  point target_size;\n  //gp_hash_table<ull, int> hashi;\n  TinyHashMap hashi;\n  vector<int> binary;\n  int add(const State&nxt, bool force = false);\n  Image getImg(int nodei);\n  void build();\n  void buildBinary();\n  void initial(Image_ test_in, const vector<pair<Image,Image>>&train, vector<point> sizes, int ti);\n  void benchmark();\n  int applyFunc(int curi, int fi, const State&state);\n  int applyFunc(int curi, int fi);\n  void applyFunc(string name, bool vec);\n  void applyFuncs(vector<pair<string,int>> names, bool vec);\n};\n\n\nstruct Pieces;\nvector<DAG> brutePieces2(Image_ test_in, const vector<pair<Image,Image>>&train, vector<point> out_sizes);\n', 'vendor/icecuber/headers/brute_size.hpp': 'vector<point> bruteSize(Image_ test_in, vector<pair<Image,Image>> train);\n', 'vendor/icecuber/headers/compose2.hpp': 'struct Candidate {\n  vImage imgs;\n  double score;\n  int cnt_pieces, sum_depth, max_depth;\n  Candidate(vImage_ imgs_, double score_) : imgs(imgs_), score(score_) {\n    cnt_pieces = sum_depth = max_depth = -1;\n  }\n  Candidate(vImage_ imgs_, int cnt_pieces_, int sum_depth_, int max_depth_) :\n    imgs(imgs_), cnt_pieces(cnt_pieces_), sum_depth(sum_depth_), max_depth(max_depth_) {\n    score = -1;\n  }\n};\n\ninline bool operator<(const Candidate& a, const Candidate& b) {\n  return a.score > b.score;\n}\n\nvector<Candidate> composePieces2(Pieces&pieces, vector<pair<Image, Image>> train, vector<point> out_sizes);\nvector<Candidate> evaluateCands(const vector<Candidate>&cand, vector<pair<Image,Image>> train);\n', 'vendor/icecuber/headers/core_functions.hpp': 'namespace core {\n  int colMask(Image_ img);\n  int countCols(Image_ img, int include0 = 0);\n  int count(Image_ img);\n  Image full(point p, point sz, int filling = 1);\n  Image empty(point p, point sz);\n  Image full(point sz, int filling = 1);\n  Image empty(point sz);\n  bool isRectangle(Image_ a);\n  int countComponents(Image img);\n  char majorityCol(Image_ img, int include0 = 0);\n  Image subImage(Image_ img, point p, point sz);\n  vector<pair<Image,int>> splitCols(Image_ img, int include0 = 0);\n};\n', 'vendor/icecuber/headers/deduce_op.hpp': '\nstruct deduceOuterProduct {\n  vector<pair<Image,Image>> train_targets;\n  int rec_funci;\n  deduceOuterProduct(vector<pair<Image,Image>> train);\n  Image reconstruct(Image_ a, Image_ b);\n};\nvoid addDeduceOuterProduct(Pieces&pieces, vector<pair<Image,Image>> train, vector<Candidate>&cands);\n', 'vendor/icecuber/headers/efficient.hpp': 'typedef unsigned long long ull;\nstruct hashEntry {\n  ull key;\n  int value, next;\n};\nstruct TinyHashMap {\n  static constexpr double sparse_factor = 1.5, resize_step = 2;\n  static constexpr int minsz = 1<<20;\n  vector<hashEntry> data;\n  vector<int> table;\n  ull mask;\n  pair<int,bool> insert(ull key, int value);\n  unsigned int size() {\n    return data.size();\n  }\n  void clear() {\n    data.clear();\n    table.clear();\n  }\n};\n\n\n\n\n\nstruct TinyChildren {\n  static constexpr int dense_thres = 10;\n  static constexpr int None = -2;\n  union {\n    int*dense;\n    pair<int,int>*sparse;\n  };\n  short sz = 0, cap = 0;\n  TinyChildren() {\n    dense = NULL;\n  }\n  void add(int fi, int to);\n  int get(int fi);\n  ~TinyChildren() {\n    delete[]dense;\n    dense = NULL;\n  }\n  TinyChildren(const TinyChildren&) = delete;\n  TinyChildren(TinyChildren&&o) {\n    dense = o.dense;\n    sz = o.sz, cap = o.cap;\n    o.dense = NULL;\n  }\n\n  void legacy(vector<pair<int,int>>&child);\n  void clear() {\n    delete[]dense;\n    dense = NULL;\n    sz = cap = 0;\n  }\n};\n\n\n\n\nstruct TinyBank {\n  vector<unsigned int> mem;\n  ll curi = 0;\n  void alloc() {\n    if (curi/32+2000 >= mem.size())\n      mem.resize(curi/32+2000);\n  }\n  inline void set(ll bi, unsigned int v) {\n    int i = bi>>5, j = bi&31;\n    if (i >= mem.size()) mem.resize(max(i+1,1024));\n    mem[i] |= v<<j;\n    //if (v) mem[i] |= 1u<<j;\n    //else mem[i] &= ~(1u<<j);\n  }\n  inline void set(ll bi, unsigned int v, int len) {\n    int i = bi>>5, j = bi&31;\n\n    mem[i] |= v << j;\n    if (j+len > 32)\n      mem[i+1] |= v >> 32-j;\n  }\n  inline int get(ll bi) {\n    return mem[bi>>5]>>(bi&31)&1;\n  }\n};\n\nstruct TinyImage {\n  static constexpr int align = 16;\n  uint32_t memi;\n  short sz;\n  char x, y, w, h;\n  uint8_t tree[9];\n  TinyImage(Image_ img, TinyBank&bank);\n  Image decompress(TinyBank&bank);\n};\n\nstruct TinyNode {\n  int*vimg = NULL;\n  bool isvec, ispiece;\n  short imgs; //TODO change to unsigned char\n  char depth;\n  TinyChildren child;\n  TinyNode() {}\n  TinyNode(TinyNode&&o) = default;\n};\n\nstruct TinyNodeBank {\n  TinyBank bank;\n  vector<TinyImage> imgs;\n  vector<TinyNode> node;\n  Image read(int i) {\n    assert(i >= 0 && i < imgs.size());\n    return imgs[i].decompress(bank);\n  }\n  Image getImg(int ni) {\n    return imgs[node[ni].vimg[0]].decompress(bank);\n  }\n  State getState(int ni) {\n    State ret;\n    ret.vimg.resize(node[ni].imgs);\n    for (int i = 0; i < node[ni].imgs; i++)\n      ret.vimg[i] = imgs[node[ni].vimg[i]].decompress(bank);\n    ret.depth = node[ni].depth;\n    ret.isvec = node[ni].isvec;\n    return ret;\n  }\n  void append(const State&state, bool ispiece) {\n    assert(state.depth >= 0 && state.depth < 128);\n    TinyNode v;\n    //assert(state.vimg.size() < 1<<16); //TODO: change to unsigned char\n\n    v.imgs = min((int)state.vimg.size(),100000);\n    v.vimg = new int[v.imgs];\n    for (int i = 0; i < v.imgs; i++) {\n      v.vimg[i] = imgs.size();\n      imgs.emplace_back(state.vimg[i], bank);\n    }\n    v.isvec = state.isvec;\n    v.ispiece = ispiece;\n    v.depth = state.depth;\n    node.push_back(move(v));\n  }\n  void addChild(int ni, int fi, int to) {\n    node[ni].child.add(fi, to);\n  }\n  int getChild(int ni, int fi) {\n    return node[ni].child.get(fi);\n  }\n  ~TinyNodeBank() {\n    for (TinyNode&n : node)\n      delete[]n.vimg;\n  }\n  TinyNode& operator[](int i) {\n    return node[i];\n  }\n  int size() { return node.size(); }\n};\n', 'vendor/icecuber/headers/image_functions.hpp': 'Image Col(int id);\nImage Pos(int dx, int dy);\nImage Square(int id);\nImage Line(int orient, int id);\nImage getPos(Image_ img);\nImage getSize(Image_ img);\nImage hull(Image_ img);\nImage toOrigin(Image img);\nImage getW(Image_ img, int id);\nImage getH(Image_ img, int id);\nImage filterCol(Image_ img, int col);\nImage colShape(Image_ shape, int col);\nImage compress(Image_ img, Image_ bg = Col(0));\nImage Fill(Image_ a);\nImage interior(Image_ a);\nImage border(Image_ a);\nImage center(Image_ img);\n//Image transform(Image_ img, int A00, int A01, int A10, int A11);\nImage rigid(Image_ img, int id);\nImage invert(Image img);\nImage interior2(Image_ a);\nImage count(Image_ img, int id, int outType);\n//Image smear(Image_ base, int id);\n\nImage hull0(Image_ img);\nImage getSize0(Image_ img);\nImage getRegular(Image_ img);\nvImage cut(Image_ img);\n\n//One and a bit of the other\nImage Move(Image img, Image_ p);\nImage embed(Image_ img, Image_ shape);\nImage extend(Image_ img, Image_ room);\nImage wrap(Image_ line, Image_ area);\n\n//Two inputs\nImage filterCol(Image_ img, Image_ palette);\nImage colShape(Image_ col, Image_ shape);\nImage broadcast(Image_ col, Image_ shape, int include0 = 1);\nImage replaceCols(Image_ base, Image_ cols);\nImage smear(Image_ base, Image_ room, int id);\n//Image alignx(Image_ a, Image_ b, int id);\n//Image aligny(Image_ a, Image_ b, int id);\nImage align(Image_ a, Image_ b, int idx, int idy);\nImage align(Image_ a, Image_ b);\n//Image compose(Image_ a, Image_ b, const function<int(int,int)>&f, int overlap_only);\nImage compose(Image_ a, Image_ b, int id = 0);\nImage outerProductIS(Image_ a, Image_ b);\nImage outerProductSI(Image_ a, Image_ b);\nImage myStack(Image_ a, Image b, int orient);\n\n\n//Image pickMax(const vector<Image>& v, function<int(Image_)> f);\nImage pickMax(vImage_ v, int id);\n//vector<Image> cut(Image_ img, Image_ a);\n//vector<Image> splitCols(Image_ img, int include0 = 0);\n//Image compose(const vector<Image>&imgs, int id);\n//void getRegular(vector<int>&col);\n//Image getRegular(Image_ img);\nImage cutPickMax(Image_ a, int id);\n//Image regularCutPickMax(Image_ a, int id);\nImage splitPickMax(Image_ a, int id, int include0 = 0);\n//Image cutCompose(Image_ a, Image_ b, int id);\n//Image regularCutCompose(Image_ a, int id);\n//Image splitCompose(Image_ a, int id, int include0 = 0);\n//Image cutIndex(Image_ a, Image_ b, int ind);\nImage splitPickMaxes(Image_ a, int id);\nImage cutIndex(Image_ a, int ind);\nImage cutPickMaxes(Image_ a, int id);\nImage heuristicCut(Image_ a);\n\nvImage pickNotMaxes(vImage_ v, int id);\n\nImage cutPickMax(Image_ a, Image_ b, int id);\nImage cutPickMaxes(Image_ a, Image_ b, int id);\nvImage pickMaxes(vImage_ v, int id);\nImage compose(vImage_ imgs, int id);\nvImage cut(Image_ img, Image_ a);\nvImage splitCols(Image_ img, int include0 = 0);\n\nImage repeat(Image_ a, Image_ b, int pad = 0);\nImage mirror(Image_ a, Image_ b, int pad = 0);\nImage majCol(Image_ img);\n', 'vendor/icecuber/headers/image_functions2.hpp': 'vImage splitAll(Image_ img);\nImage eraseCol(Image img, int col);\nvImage insideMarked(Image_ in);\nImage makeBorder(Image_ img, int bcol = 1);\nImage makeBorder2(Image_ img, int usemaj = 1);\nImage makeBorder2(Image_ img, Image_ bord);\nImage compress2(Image_ img);\nImage compress3(Image_ img);\nImage greedyFillBlack(Image_ img, int N = 3);\nImage greedyFillBlack2(Image_ img, int N = 3);\nImage extend2(Image_ img, Image_ room);\nImage connect(Image_ img, int id);\nImage replaceTemplate(Image_ in, Image_ need_, Image_ marked_, int overlapping = 0, int rigids = 0);\nImage swapTemplate(Image_ in, Image_ a, Image_ b, int rigids = 0);\nImage spreadCols(Image img, int skipmaj = 0);\nvImage splitColumns(Image_ img);\nvImage splitRows(Image_ img);\nImage half(Image_ img, int id);\nImage smear(Image_ img, int id);\nImage mirror2(Image_ a, Image_ line);\nvImage gravity(Image_ in, int d);\n\nImage myStack(vImage_ lens, int id);\nImage stackLine(vImage_ shapes);\nImage composeGrowing(vImage_ imgs);\n\nImage pickUnique(vImage_ imgs, int id);\n', 'vendor/icecuber/headers/normalize.hpp': 'struct Simplifier {\n  function<Image(Image_)> in;\n  function<Image(Image_,Image_)> out, rec;\n  pair<Image,Image> operator()(Image_ a, Image_ b);\n};\n\nSimplifier normalizeCols(const vector<pair<Image,Image>>&train);\nSimplifier normalizeDummy(const vector<pair<Image,Image>>&train);\n\n//inline vector<double> shapeFeatures(Image_ img, int col);\n//inline Image remapCols(Image_ img, int cols[10]);\nvoid remapCols(const vector<pair<Image,Image>>&train, vector<Simplifier>&sims);\n//inline Image listCols(Image_ img, int extra);\nvoid normalizeCols(vector<Sample>&sample);\n\nvoid normalizeRigid(const vector<pair<Image,Image>>&train, vector<Simplifier>&sims);\n', 'vendor/icecuber/headers/pieces.hpp': '/*struct Piece2 {\n  vector<int> ind;\n  int depth;\n  };*/\nstruct Piece3 {\n  int memi, depth;\n};\n\nstruct Pieces {\n  vector<DAG> dag;\n  vector<Piece3> piece;\n  //vector<vector<int>> seen;\n  vector<int> mem;\n};\n\nPieces makePieces2(vector<DAG>&dag, vector<pair<Image,Image>> train, vector<point> out_sizes);\n', 'vendor/icecuber/headers/precompiled_stl.hpp': '#include <vector>\n#include <unordered_map>\n#include <cassert>\n#include <iostream>\n#include <set>\n#include <random>\n#include <algorithm>\n#include <tuple>\n#include <functional>\n#include <regex>\n#include <queue>\n\n//#include <bits/stdc++.h>\n', 'vendor/icecuber/headers/read.hpp': '\nstruct Sample {\n  vector<pair<Image,Image>> train, test;\n  Image test_in, test_out;\n  string id;\n  int id_ind;\n  FILE*fp;\n  Sample(string filename);\n  char mygetc();\n  int end(char endc);\n  void expect(char c);\n  string getQuote();\n  vector<int> readRow();\n  Image readImage();\n  vector<Sample> split();\n};\n\nvector<Sample> readAll(string path, int maxn);\n\nstruct Writer {\n  FILE*fp;\n  map<string,int> seen;\n  Writer(string filename = "submission_part.csv");\n  void operator()(const Sample& s, vector<Image> imgs);\n  ~Writer();\n};\n\nvoid writeAnswersWithScores(const Sample&s, string fn, vector<Image> imgs, vector<double> scores);\n', 'vendor/icecuber/headers/runner.hpp': 'void run(int only_sid = -1, int maxdepth = -1);\n', 'vendor/icecuber/headers/spec.hpp': '\nstruct Spec {\n  bool bad = 0, anypos = 0;\n  union {\n    point p;\n    struct {\n      int x, y;\n    };\n  };\n  union {\n    point sz;\n    struct {\n      int w, h;\n    };\n  };\n  vector<int> mask;\n  inline int& operator()(int i, int j) {\n#if defined CHECK_BOUNDS\n    assert(i >= 0 && i < h && j >= 0 && j < w);\n#endif\n    return mask[i*w+j];\n  }\n  inline int operator()(int i, int j) const {\n#if defined CHECK_BOUNDS\n    assert(i >= 0 && i < h && j >= 0 && j < w);\n#endif\n    return mask[i*w+j];\n  }\n  bool check(Image_ img) const {\n    if (bad) return 0;\n    if (img.sz != sz || (!anypos && img.p != p)) return 0;\n    for (int i = 0; i < mask.size(); i++)\n      if ((mask[i]>>img.mask[i]&1) == 0) return 0;\n    return 1;\n  }\n};\n\ntypedef const Spec& Spec_;\nconst int allCols = (1<<10)-1;\n', 'vendor/icecuber/headers/store.hpp': 'void storePieces(vector<vector<Piece>>&pieces);\nvector<vector<Piece>> loadPieces();\nvoid storeCands(vector<vector<Image>>&cands);\nvector<vector<Image>> loadCands();\n\nvoid storePieces(vector<Piece>&vp, string id, int id_ind);\nvector<Piece> loadPieces(string id, int id_ind);\n', 'vendor/icecuber/headers/timer.hpp': '#include <chrono>\n\nstruct Timer {\n  const int skips = 100, brute_thres = 1000;\n  bool running = false;\n  double cnt = 0, brute_sum = 0;\n  int sample_counter = 0, target = 0;\n  double samples = 0, sample_sum = 0, sample_var = 0;\n  mt19937 mrand;\n  double now() {\n    return chrono::high_resolution_clock::now().time_since_epoch().count()*1e-9;\n  }\n  double start_time;\n  void start() {\n    assert(!running);\n    running = true;\n    if (cnt < brute_thres || sample_counter == target)\n      start_time = now();\n  }\n  void stop() {\n    assert(running);\n    running = false;\n\n    double dt;\n    if (cnt < brute_thres || sample_counter == target)\n      dt = now()-start_time;\n\n    if (cnt < brute_thres) {\n      brute_sum += dt;\n    }\n    if (sample_counter == target) {\n      samples++;\n      sample_sum += dt;\n      sample_var += dt*dt;\n      target = mrand()%skips;\n    }\n\n    if (++sample_counter == skips)\n      sample_counter = 0;\n    cnt++;\n  }\n  tuple<double,double,double> read() {\n    if (cnt <= brute_thres)\n      return {cnt, brute_sum, 0};\n    double mean = sample_sum/samples;\n    double var = (sample_var/samples-mean*mean)/(samples-1);\n    return {cnt, mean*cnt, sqrt(var)*cnt};\n  }\n  void print(const string& label) {\n    auto [cnt,sum,std] = read();\n    printf("%5.1f ± %4.1f ms - %s\\n", sum*1e3, std*1e3, label.c_str());\n  }\n};\n', 'vendor/icecuber/headers/utils.hpp': '//const int MAXSIDE = 100, MAXAREA = 40*40, MAXPIXELS = 40*40*5;\nextern int MAXSIDE, MAXAREA, MAXPIXELS;\n\n//#define CHECK_BOUNDS\n\nstruct point {\n  int x, y;\n};\ninline point operator+(const point& a, const point& b) {\n  return {a.x+b.x, a.y+b.y};\n}\ninline point operator-(const point& a, const point& b) {\n  return {a.x-b.x, a.y-b.y};\n}\ninline point operator*(const point& a, int f) {\n  return {a.x*f, a.y*f};\n}\ninline point operator/(const point& a, int f) {\n  assert(a.x%f == 0 && a.y%f == 0);\n  return {a.x/f, a.y/f};\n}\ninline int operator*(const point& a, const point& b) {\n  return a.x*b.x+a.y*b.y;\n}\ninline int operator%(const point& a, const point& b) {\n  return a.x*b.y-a.y*b.x;\n}\ninline bool operator==(const point& a, const point& b) {\n  return a.x == b.x && a.y == b.y;\n}\ninline bool operator!=(const point& a, const point& b) {\n  return a.x != b.x || a.y != b.y;\n}\n\n\n\nstruct Image {\n  union {\n    point p;\n    struct {\n      int x, y;\n    };\n  };\n  union {\n    point sz;\n    struct {\n      int w, h;\n    };\n  };\n  vector<char> mask;\n  inline char& operator()(int i, int j) {\n#if defined CHECK_BOUNDS\n    assert(i >= 0 && i < h && j >= 0 && j < w);\n#endif\n    return mask[i*w+j];\n  }\n  inline char operator()(int i, int j) const {\n#if defined CHECK_BOUNDS\n    assert(i >= 0 && i < h && j >= 0 && j < w);\n#endif\n    return mask[i*w+j];\n  }\n  inline char safe(int i, int j) const {\n    return (i<0 || j<0 || i>=h || j>=w ? (char)0 : mask[i*w+j]);\n  }\n};\n\nusing Image_ = const Image&;\nusing vImage = vector<Image>;\nusing vImage_ = const vector<Image>&;\n\ninline bool operator==(Image_ a, Image_ b) {\n  return tie(a.p,a.sz,a.mask) == tie(b.p,b.sz,b.mask);\n}\ninline bool operator!=(Image_ a, Image_ b) {\n  return tie(a.p,a.sz,a.mask) != tie(b.p,b.sz,b.mask);\n}\ninline bool operator<(Image_ a, Image_ b) {\n  if (a.sz != b.sz) return tie(a.w,a.h) < tie(b.w,b.h);\n  return a.mask < b.mask;\n}\n\nusing ll = long long;\nusing ull = unsigned long long;\ninline ull hashImage(Image_ img) {\n  const ull base = 137;\n  ull r = 1543;\n  r = r*base+img.w;\n  r = r*base+img.h;\n  r = r*base+img.x;\n  r = r*base+img.y;\n  for(char c : img.mask) {\n    //assert(c >= 0 && c < 10);\n    r = r*base+c;\n  }\n  return r;\n}\n\nstruct Piece {\n  vector<Image> imgs;\n  double node_prob;\n  int keepi, knowi;\n};\n\nconst Image badImg = {{0,0},{0,0},{}};\nconst Image dummyImg = {{0,0},{1,1},{0}};\n\nusing Piece_ = const Piece&;\n\ntemplate<class T, class F>\nint checkAll(const vector<T>&v, F f) {\n  int all = 1;\n  for (const T& it : v)\n    all &= f(it);\n  return all;\n}\ntemplate<class T, class F>\nint allEqual(const vector<T>&v, F f) {\n  int need = f(v[0]);\n  for (const T& it : v) {\n    if (f(it) != need) return 0;\n  }\n  return 1;\n}\n', 'vendor/icecuber/headers/visu.hpp': 'struct Visu {\n  FILE*fp;\n  Visu();\n  ~Visu();\n  void next(string s);\n  void add(Image in, Image out);\n};\n\nvoid plot(const vector<vector<int>>&inp, const char*filename = "out.ppm");\nvoid print(Image img);\n', 'vendor/icecuber/src/brute2.cpp': '#include "precompiled_stl.hpp"\n#include <chrono>\nusing namespace std;\n#include "utils.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "image_functions2.hpp"\n\n#include "visu.hpp"\n\n#include "brute2.hpp"\n#include "pieces.hpp"\n\n#include "timer.hpp"\n\nextern int MAXDEPTH, print_nodes;\n\n//double build_f_time = 0, apply_f_time = 0;\n//double real_f_time = 0;\n\ndouble now() {\n  ll t = chrono::high_resolution_clock::now().time_since_epoch().count();\n  static ll time0 = 0;\n  if (time0 == 0) time0 = t;\n  return (t-time0)*1e-9;\n}\n//double now() { return chrono::steady_clock::now().time_since_epoch().count()*1e-9;}\n\nTimer build_f_time, apply_f_time, real_f_time, add_time, find_child_time, add_child_time, hash_time, map_time, total_time;\nTimer state_time;\n\nvoid Functions3::add(const string& name, int cost_, const function<bool(const State&,State&)>&func, int list) {\n  //if (cost_ != 10) cout << name << endl;\n  //assert(cost_ == 10);\n  if (list) listed.push_back(names.size());\n  names.push_back(name);\n  f_list.push_back(func);\n  cost.push_back(cost_);\n}\n\nvoid Functions3::add(string name, int cost, const function<Image(Image_)>&f, int list) { //list = 1\n  auto func = [f](const State& cur, State& nxt) {\n\n    //if (cur.isvec) return false;\n\n    nxt.vimg.resize(cur.vimg.size());\n    nxt.isvec = cur.isvec;\n\n    int area = 0;\n    for (int i = 0; i < cur.vimg.size(); i++) {\n      real_f_time.start();\n      nxt.vimg[i] = f(cur.vimg[i]);\n      real_f_time.stop();\n\n      area += nxt.vimg[i].w*nxt.vimg[i].h;\n      if (area > MAXPIXELS) return false;\n    }\n    return true;\n  };\n  add(name, cost, func, list);\n}\n\nvoid Functions3::add(string name, int cost, const function<vImage(Image_)>&f, int list) { //list = 1\n  const int buffer = 5;\n  auto func = [f,cost](const State& cur, State& nxt) {\n    if (cur.isvec || cur.depth+cost+buffer > MAXDEPTH) return false;\n    real_f_time.start();\n    nxt.vimg = f(cur.vimg[0]);\n    real_f_time.stop();\n    nxt.isvec = true;\n    return true;\n  };\n  add(name, cost, func, list);\n}\n\nvoid Functions3::add(string name, int cost, const function<Image(vImage_)>&f, int list) { //list = 1\n  auto func = [f](const State& cur, State& nxt) {\n    if (!cur.isvec) return false;\n    nxt.vimg.resize(1);\n    real_f_time.start();\n    nxt.vimg[0] = f(cur.vimg);\n    real_f_time.stop();\n    nxt.isvec = false;\n    return true;\n  };\n  add(name, cost, func, list);\n}\n\nvoid Functions3::add(string name, int cost, const function<vImage(vImage_)>&f, int list) { //list = 1\n  auto func = [f](const State& cur, State& nxt) {\n    if (!cur.isvec) return false;\n    real_f_time.start();\n    nxt.vimg = f(cur.vimg);\n    real_f_time.stop();\n    nxt.isvec = true;\n    return true;\n  };\n  add(name, cost, func, list);\n}\n\nvoid Functions3::add(const vector<point>&sizes, string name, int cost, const function<Image(Image_,Image_)>&f, int list) { //list = 1\n  int szi = 0;\n  for (point sz : sizes) {\n    Image arg2 = core::empty(sz);\n    auto func = [f,arg2](const State& cur, State& nxt) {\n\n      if (cur.isvec) return false;\n\n      nxt.vimg.resize(cur.vimg.size());\n\n      int area = 0;\n      for (int i = 0; i < cur.vimg.size(); i++) {\n\treal_f_time.start();\n\tnxt.vimg[i] = f(cur.vimg[i], arg2);\n\treal_f_time.stop();\n\n\tarea += nxt.vimg[i].w*nxt.vimg[i].h;\n\tif (area > MAXPIXELS) return false;\n      }\n      nxt.isvec = cur.isvec;\n      return true;\n    };\n    add(name+" "+to_string(szi++), cost, func, list);\n  }\n}\n\nstring Functions3::getName(int fi) {\n  assert(fi >= 0 && fi < names.size());\n  return names[fi];\n}\nint Functions3::findfi(string name) {\n  int fi = find(names.begin(), names.end(), name)-names.begin();\n  if (fi == names.size()) {\n    cerr << name << " is not a known function" << endl;\n    assert(0);\n  }\n  return fi;\n}\n\n\nFunctions3 initFuncs3(const vector<point>&sizes) {\n  Functions3 funcs;\n\n  // Unary\n\n  //invert is filterCol(img, 0)\n  for (int c = 0; c < 10; c++)\n    funcs.add("filterCol "+to_string(c), 10, [c](Image_ img) {return filterCol(img, c);});\n  for (int c = 1; c < 10; c++)\n    funcs.add("eraseCol "+to_string(c), 10,\n\t      [c](Image_ img) {return eraseCol(img, c);});\n\n  for (int c = 1; c < 10; c++)\n    funcs.add("colShape "+to_string(c), 10,\n\t      [c](Image_ img) {return colShape(img, c);}, 0);\n\n  funcs.add("compress", 10, [](Image_ img) {return compress(img);});\n  funcs.add("getPos", 10, getPos);\n  funcs.add("getSize0", 10, getSize0);\n  funcs.add("getSize", 10, getSize);\n  funcs.add("hull0", 10, hull0);\n  funcs.add("hull", 10, hull);\n  funcs.add("toOrigin", 10, toOrigin);\n  funcs.add("Fill", 10, Fill);\n  funcs.add("interior", 10, interior);\n  funcs.add("interior2", 10, interior2);\n  funcs.add("border", 10, border);\n  funcs.add("center", 10, center);\n  funcs.add("majCol", 10, majCol);\n\n  //funcs.add("greedyFillBlack", 10, [](Image_ img) {return greedyFillBlack(img);});\n  //funcs.add("greedyFillBlack2", 10, [](Image_ img) {return greedyFillBlack2(img);});\n\n  for (int i = 1; i < 9; i++)\n    funcs.add("rigid "+to_string(i), 10,\n\t      [i](Image_ img) {return rigid(img, i);});\n  for (int a = 0; a < 3; a++)\n    for (int b = 0; b < 3; b++)\n      funcs.add("count "+to_string(a)+" "+to_string(b), 10,\n\t\t[a,b](Image_ img) {return count(img, a, b);});\n  for (int i = 0; i < 15; i++)\n    funcs.add("smear "+to_string(i), 10,\n\t      [i](Image_ img) {return smear(img, i);});\n\n\n  funcs.add("makeBorder", 10,\n\t    [](Image_ img) {return makeBorder(img, 1);});\n\n  for (int id : {0,1})\n    funcs.add("makeBorder2 "+to_string(id), 10,\n\t      [id](Image_ img) {return makeBorder2(img, id);});\n  funcs.add("compress2", 10, compress2);\n  funcs.add("compress3", 10, compress3);\n\n  for (int id = 0; id < 3; id++)\n    funcs.add("connect "+to_string(id), 10,\n\t      [id](Image_ img) {return connect(img,id);});\n\n  for (int id : {0,1})\n    funcs.add("spreadCols "+to_string(id), 10,\n\t      [id](Image_ img) {return spreadCols(img, id);});\n\n  for (int id = 0; id < 4; id++)\n    funcs.add("half "+to_string(id), 10,\n\t      [id](Image_ img) {return half(img, id);});\n\n\n  for (int dy = -2; dy <= 2; dy++) {\n    for (int dx = -2; dx <= 2; dx++) {\n      funcs.add("Move "+to_string(dx)+" "+to_string(dy), 10,\n\t\t[dx,dy](Image_ img) {return Move(img, Pos(dx,dy));}, 0);\n    }\n  }\n\n  // Binary\n  funcs.add(sizes, "embed", 10, embed);\n  funcs.add(sizes, "wrap", 10, wrap);\n  funcs.add(sizes, "broadcast", 10, [](Image_ a, Image_ b) {return broadcast(a,b);});\n  funcs.add(sizes, "repeat 0",  10, [](Image_ a, Image_ b) {return repeat(a,b);});\n  funcs.add(sizes, "repeat 1",  10, [](Image_ a, Image_ b) {return repeat(a,b,1);});\n  funcs.add(sizes, "mirror 0",  10, [](Image_ a, Image_ b) {return mirror(a,b);});\n  funcs.add(sizes, "mirror 1",  10, [](Image_ a, Image_ b) {return mirror(a,b,1);});\n\n\n  //Split\n  funcs.add("cut",       10, [](Image_ img) {return cut(img);});\n  funcs.add("splitCols", 10, [](Image_ img) {return splitCols(img);});\n  funcs.add("splitAll",     10, splitAll);\n  funcs.add("splitColumns", 10, splitColumns);\n  funcs.add("splitRows",    10, splitRows);\n  funcs.add("insideMarked", 10, insideMarked);\n  for (int id = 0; id < 4; id++)\n    funcs.add("gravity "+to_string(id), 10,\n\t      [id](Image_ img) {return gravity(img,id);});\n\n\n  //Join\n  for (int id = 0; id < 14; id++)\n    funcs.add("pickMax "+to_string(id), 10,\n\t      [id](vImage_ v) {return pickMax(v,id);});\n  for (int id = 0; id < 1; id++)\n    funcs.add("pickUnique "+to_string(id), 10,\n\t      [id](vImage_ v) {return pickUnique(v,id);});\n\n  funcs.add("composeGrowing", 10, composeGrowing);\n  funcs.add("stackLine", 10, stackLine);\n  for (int id = 0; id < 2; id++) //consider going to 4\n    funcs.add("myStack "+to_string(id), 10,\n\t      [id](vImage_ v) {return myStack(v,id);}); //\n\n\n  //Vector\n  for (int id = 0; id < 14; id++)\n    funcs.add("pickMaxes "+to_string(id), 10,\n\t      [id](vImage_ v) {return pickMaxes(v,id);});\n  for (int id = 0; id < 14; id++)\n    funcs.add("pickNotMaxes "+to_string(id), 10,\n\t      [id](vImage_ v) {return pickNotMaxes(v,id);});\n\n\n  static int said = 0;\n  if (!said) {\n    cout << "Function count: " << funcs.listed.size() << endl;\n    said = 1;\n  }\n\n\n  //funcs.add("smear",    [](Image_ a, Image_ b) {return smear(a,b,6);});\n\n  //funcs.add(insideMarked); //only do once at depth 0\n\n  // Smear diagonals?\n\n  // outerProducts\n\n  //for (int id = 0; id < 4; id++)\n  //  funcs.add([id](Image_ img) {return gravity(img, id);});\n\n  // Image makeBorder(Image_ img, int bcol = 1);\n  // Image makeBorder2(Image_ img, Image_ bord);\n  // Image greedyFillBlack(Image_ img, int N = 3);\n  // Image extend2(Image_ img, Image_ room);\n  // Image replaceTemplate(Image_ in, Image_ need_, Image_ marked_, int overlapping = 0, int rigids = 0);\n  // Image swapTemplate(Image_ in, Image_ a, Image_ b, int rigids = 0);\n\n  // funcs.add("heuristicCut", heuristicCut);\n\n  return funcs;\n}\n\nImage DAG::getImg(int ni) {\n  return tiny_node.getImg(ni);\n  //assert(tiny_node.getImg(ni) == node[ni].vimg[0]);\n  //return node[ni].state.vimg[0];\n  /*\n  //cout << nodei << endl;\n  assert(nodei >= 0 && nodei < (int)node.size());\n  assert(!node[nodei].isvec);\n  if (node[nodei].pfi == embed1fi) {\n    assert(funcs.f_list[embed1fi](node[nodei], tmp_node));\n    //assert(tmp_node.vimg == node[nodei].vimg);\n    return tmp_node.vimg[0];\n  } else {\n    assert(node[nodei].vimg.size());\n    return node[nodei].vimg[0];\n    }*/\n}\n\n\nint DAG::add(const State&nxt, bool force) { //force = false\n  hash_time.start();\n  ull h = nxt.hash();\n  hash_time.stop();\n  int nodes = tiny_node.size();\n  map_time.start();\n  auto [nodei,inserted] = hashi.insert(h,nodes);\n  //auto [it,inserted] = hashi.insert({h,nodes}); int nodei = it->second;\n  //assert(inserted == tiny_inserted);\n  //assert(nodei == tiny_nodei);\n\n  map_time.stop();\n\n  add_time.start();\n  if (inserted || force) {\n    bool ispiece = !nxt.isvec;\n    if (!nxt.isvec && target_size != point{-1,-1})\n      ispiece &= (nxt.vimg[0].p == point{0,0} && nxt.vimg[0].sz == target_size);\n    /*{\n      Node n;\n      n.state = nxt;\n      n.ispiece = ispiece;\n      n.freed = false;\n      node.push_back(n);\n      }*/\n    tiny_node.append(nxt, ispiece);\n  }\n  add_time.stop();\n  return nodei;\n}\n\n\nvoid DAG::build() {\n  build_f_time.start();\n\n  for (int curi = 0; curi < tiny_node.size(); curi++) {\n    int depth = tiny_node[curi].depth;\n    if (depth+1 > MAXDEPTH) continue;\n\n    //vector<pair<int,int>> child;\n    State nxt;\n    state_time.start();\n    State cur_state = tiny_node.getState(curi);\n    state_time.stop();\n    for (int fi : funcs.listed) {\n      nxt.depth = depth+funcs.cost[fi];\n      if (nxt.depth > MAXDEPTH) continue;\n      if (funcs.f_list[fi](cur_state, nxt)) {\n\tint newi = add(nxt);\n\t//child.emplace_back(fi, newi);\n\ttiny_node.addChild(curi, fi, newi);\n      } else {\n\ttiny_node.addChild(curi, fi, -1);\n\t//child.emplace_back(fi, -1);\n      }\n\n    }\n    //node[curi].child = child;\n  }\n\n  build_f_time.stop();\n}\n\nvoid DAG::initial(Image_ test_in, const vector<pair<Image,Image>>&train, vector<point> sizes, int ti) {\n  if (sizes.size() > 1)\n    target_size = sizes[1];\n  else\n    target_size = point{-1,-1};\n\n  Image in = ti < train.size() ? train[ti].first : test_in;\n\n  add(State({in}, false, 0), true);\n\n  //Output sizes\n  for (point sz : sizes)\n    add(State({core::empty(sz)}, false, 10), true);\n\n  // Outputs of other trains\n  for (int tj = 0; tj < train.size(); tj++)\n    add(State({ti != tj ? train[tj].second : core::empty(train[tj].second.sz)}, false, 10), true);\n\n  //add(State({greedyFillBlack2(in)}, false, 10), true);\n\n  //filterCol?\n\n  givens = tiny_node.size();\n}\n\n//time each function\nvoid DAG::benchmark() {\n  vector<pair<double,int>> v;\n  for (int fi : funcs.listed) {\n    double start_time = now();\n    State nxt;\n    for (int i = 0; i < tiny_node.size(); i++) {\n      funcs.f_list[fi](tiny_node.getState(i), nxt);\n    }\n    double elapsed = now()-start_time;\n    v.emplace_back(elapsed, fi);\n  }\n  sort(v.begin(), v.end());\n  for (auto [t,fi] : v)\n    printf("%.1f ms - %s\\n", t*1e3, funcs.getName(fi).c_str());\n}\n\nint DAG::applyFunc(int curi, int fi, const State&state) {\n\n  find_child_time.start();\n  //auto it = lower_bound(node[curi].child.begin(), node[curi].child.end(), make_pair(fi,-1));\n  int it2 = tiny_node.getChild(curi, fi);\n  find_child_time.stop();\n  if (it2 != TinyChildren::None) {//it != node[curi].child.end() && it->first == fi) {\n    //if (it2 != it->second) cout << it2 << \' \' << it->second << endl;\n    //assert(it2 == it->second);\n    return it2;//it->second;\n  }\n  //assert(it2 == -2);\n\n\n  State nxt;\n  nxt.depth = tiny_node[curi].depth+funcs.cost[fi];\n  //nxt.par = curi;\n\n  int newi = -1;\n\n  apply_f_time.start();\n  bool ok = funcs.f_list[fi](state, nxt);\n  apply_f_time.stop();\n\n  if (ok) {\n    //nxt.pfi = fi;\n    newi = add(nxt);\n  }\n\n  add_child_time.start();\n  tiny_node.addChild(curi, fi, newi);\n  //node[curi].child.emplace_back(fi, newi);\n  //sort(node[curi].child.begin(), node[curi].child.end());\n  add_child_time.stop();\n  return newi;\n}\n\nint DAG::applyFunc(int curi, int fi) {\n  state_time.start();\n  State state = tiny_node.getState(curi);\n  state_time.stop();\n  return applyFunc(curi, fi, state);\n}\n\nvoid DAG::applyFunc(string name, bool vec) {\n  int fi = funcs.findfi(name);\n\n  int start_nodes = tiny_node.size();\n  for (int curi = 0; curi < start_nodes; curi++) {\n    if (tiny_node[curi].isvec == vec) applyFunc(curi, fi);\n  }\n}\n\nvoid DAG::applyFuncs(vector<pair<string,int>> names, bool vec) {\n  vector<pair<int,int>> fis;\n  for (auto [name,id] : names) {\n    fis.emplace_back(funcs.findfi(name), id);\n  }\n\n  vector<int> parid(tiny_node.size(),-1);\n  for (int curi = 0; curi < tiny_node.size(); curi++) {\n    if (tiny_node[curi].isvec != vec) continue;\n    state_time.start();\n    State state = tiny_node.getState(curi);\n    state_time.stop();\n\n    for (int i = 0; i < fis.size(); i++) {\n      auto [fi,id] = fis[i];\n      if (id <= parid[curi]) continue;\n\n      if (applyFunc(curi, fi, state) == parid.size()) {\n\tparid.push_back(id);\n\tassert(parid.size() == tiny_node.size());\n      }\n    }\n  }\n}\n\n\n\n\n\nvoid DAG::buildBinary() {\n  int fis = *max_element(funcs.listed.begin(), funcs.listed.end())+1;\n  binary.assign(fis*fis, -1);\n  vector<State> state(fis);\n  vector<int> active(fis), memi(fis);\n  for (int fi : funcs.listed) {\n    int curi = tiny_node.getChild(0, fi);\n    if (curi >= 0) {\n      active[fi] = 1;\n      state[fi] = tiny_node.getState(curi);\n      memi[fi] = curi;\n    }\n  }\n  for (int fa : funcs.listed) {\n    if (!active[fa]) continue;\n    for (int fb : funcs.listed) {\n      if (!active[fb]) continue;\n\n      if (state[fa].isvec || state[fb].isvec) continue;\n      State nxt;\n      nxt.vimg = {align(state[fa].vimg[0], state[fb].vimg[0])};\n      nxt.depth = 2; //TODO\n      nxt.isvec = false;\n      binary[fa*fis+fb] = add(nxt);\n      /*if (binary[fa*fis+fb] != memi[fa]) {\n\tcout << binary[fa*fis+fb] << \' \' << memi[fa] << \' \' << tiny_node.size() << endl;\n      }\n      assert(binary[fa*fis+fb] == memi[fa]);*/\n    }\n  }\n}\n\n\n\nvector<DAG> brutePieces2(Image_ test_in, const vector<pair<Image,Image>>&train, vector<point> out_sizes) {\n  int print = 1;\n\n  vector<DAG> dag(train.size()+1);\n\n  int all_train_out_mask = 0, and_train_out_mask = ~0;\n  for (int ti = 0; ti < train.size(); ti++)\n    and_train_out_mask &= core::colMask(train[ti].second);\n\n  for (int ti = 0; ti <= train.size(); ti++) {\n    vector<point> sizes;\n    if (ti < train.size())\n      sizes.push_back(train[ti].first.sz);\n    else\n      sizes.push_back(test_in.sz);\n\n    if (out_sizes.size())\n      sizes.push_back(out_sizes[ti]);\n\n    dag[ti].funcs = initFuncs3(sizes);\n\n    dag[ti].initial(test_in, train, sizes, ti);\n\n    total_time.start();\n\n    double start_time = now();\n    dag[ti].build();\n    if (print) cout << now()-start_time << endl;\n    //dag[ti].buildBinary();\n    //if (print) cout << now()-start_time << endl;\n    dag[ti].applyFunc("composeGrowing", 1);\n    if (print) cout << now()-start_time << endl;\n\n    if (sizes.size() > 1) {\n      vector<pair<string,int>> toapply;\n      toapply.emplace_back("toOrigin",0);\n      for (int c = 1; c <= 5; c++)\n\tif (and_train_out_mask>>c&1)\n\t  toapply.emplace_back("colShape "+to_string(c),1);\n      toapply.emplace_back("embed 1",2);\n      dag[ti].applyFuncs(toapply, 0);\n      /*\n      dag[ti].applyFunc("toOrigin", 0);\n      if (print) cout << now()-start_time << endl;\n\n      int mask;\n      if (ti < train.size()) {\n\tmask = core::colMask(train[ti].second);\n\tall_train_out_mask |= mask;\n      } else {\n\tmask = all_train_out_mask;\n      }\n\n      for (int c = 1; c <= 5; c++)\n\tif (and_train_out_mask>>c&1)\n\tdag[ti].applyFunc("colShape "+to_string(c), 0);\n      if (print) cout << now()-start_time << endl;\n\n      if (ti < train.size())\n\tdeducePositions(dag[ti], train[ti].second);\n      if (print) cout << now()-start_time << endl;\n\n      dag[ti].applyFunc("embed 1", 0);\n      */\n      if (print) cout << now()-start_time << endl;\n\n      /*if (ti < train.size())\n\tdeducePositions(dag[ti], train[ti].second);\n\n\tif (print) cout << now()-start_time << endl;*/\n\n      total_time.stop();\n      total_time.print("Total time");\n      build_f_time.print("Build f time");\n      apply_f_time.print("Apply f time");\n      real_f_time .print("Real f time ");\n\n      add_time.print("Add time");\n      find_child_time.print("Find child");\n      add_child_time.print("Add child");\n      hash_time.print("Hash");\n      map_time.print("Map");\n\n      state_time.print("getState");\n      //exit(0);\n      /*FILE*fp = fopen("images.txt", "w");\n      for (Node&n : dag[ti].node) {\n\tfor (Image_ img : n.vimg) {\n\t  fprintf(fp, "%d %d %d %d\\n", img.x, img.y, img.w, img.h);\n\t  for (char c : img.mask)\n\t    fprintf(fp, "%c", \'0\'+c);\n\t  fprintf(fp, "\\n");\n\t}\n      }\n      exit(0);*/\n      //dag[ti].freeAll();\n    } else total_time.stop();\n    //dag[ti].benchmark();\n    //exit(0);\n\n    /*\n    for (Node&n : dag[ti].node) {\n      if (!n.isvec && n.pfi == dag[ti].embed1fi) {\n\tn.vimg.clear();\n\tn.vimg.shrink_to_fit();\n      }\n    }\n    */\n\n    /*if (ti < train.size()) {\n      for (Node&n : dag[ti].node) {\n\tif (n.par > -1 && (n.isvec || n.img[0].w > 30 || n.img[0].h > 30)) {// || n.img[0].p != point{0,0} || n.img[0].sz != given_sizes[ti][1])) {\n\t  n.img.clear();\n\t  n.img.shrink_to_fit();\n\t}\n      }\n      dag[ti].hashi.clear();\n      }*/\n  }\n\n\n  if (out_sizes.size() && print_nodes) {\n    cout << "Dag sizes: ";\n    for (int i = 0; i < dag.size(); i++)\n      cout << dag[i].tiny_node.size() << " ";\n    cout << endl;\n  }\n\n  return dag;\n}\n', 'vendor/icecuber/src/brute_size.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "image_functions2.hpp"\n#include "visu.hpp"\n\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "compose2.hpp"\n\nextern int MAXDEPTH;\n\nbool operator<(const point a, const point b) {\n  if (a.x != b.x) return a.x < b.x;\n  return a.y < b.y;\n}\n\npair<vector<int>,double> solveSingle(vector<vector<int>>&seeds, const vector<int>& target) {\n  int n = target.size()+1;\n  vector<int> ans(n,1);\n  pair<int,double> best = {-1,1e9};\n\n  auto add = [&](const vector<int>&szs, double loss) {\n    int oks = 0;\n    for (int ti = 0; ti < target.size(); ti++)\n      oks += (szs[ti] == target[ti]);\n    pair<int,double> cand = {oks, -loss-10};\n    int sz = szs.back();\n    if (sz >= 1 && sz <= 30 &&\n\tcand > best) {\n      best = cand;\n      ans = szs;\n    }\n  };\n\n  for (int w = 1; w <= 30; w++) {\n    add(vector<int>(n,w), w);\n  }\n\n  vector<int> szs(n);\n  for (int i = 0; i < seeds.size(); i++) {\n    double a = i+1;\n    for (int w = 1; w < 6; w++) {\n      for (int x = -3; x <= 3; x++) {\n\tfor (int k = 0; k < n; k++)\n\t  szs[k] = seeds[i][k]*w+x;\n\tadd(szs, a*w*(abs(x)+1));\n      }\n    }\n  }\n\n  return {ans,best.second};\n}\n\npoint solveSize(vector<vector<point>>&seeds, const vector<point>& target) {\n  point ans = {1,1};\n  pair<int,double> best = {-1,1e9};\n\n  auto add = [&](const vector<point>&szs, double loss) {\n    int oks = 0;\n    for (int ti = 0; ti < target.size(); ti++)\n      oks += (szs[ti] == target[ti]);\n    pair<int,double> cand = {oks, -loss};\n    point sz = szs.back();\n    if (sz.x >= 1 && sz.x <= 30 &&\n\tsz.y >= 1 && sz.y <= 30 &&\n\tcand > best) {\n      best = cand;\n      ans = sz;\n    }\n  };\n\n  int n = target.size()+1;\n  vector<point> szs(n);\n\n  for (int i = 0; i < seeds.size(); i++) {\n    double a = i+1;\n    for (int h = 1; h < 6; h++) {\n      for (int w = 1; w < 6; w++) {\n\tfor (int y = -3; y <= 3; y++) {\n\t  for (int x = -3; x <= 3; x++) {\n\t    for (int k = 0; k < n; k++)\n\t      szs[k] = {seeds[i][k].x*w+x, seeds[i][k].y*h+y};\n\t    add(szs, a*w*h*(abs(x)+1)*(abs(y)+1));\n\t  }\n\t}\n      }\n    }\n  }\n\n  if (1) {//best.first < target.size()) {\n    for (int i = 0; i < seeds.size(); i++) {\n      double a = i+1;\n      for (int j = 0; j < i; j++) {\n\tdouble b = j+1;\n\tfor (int d = 0; d < 3; d++) {\n\t  for (int k = 0; k < n; k++) {\n\t    szs[k] = seeds[i][k];\n\t    if (d == 0 || d == 2) szs[k].x = szs[k].x+seeds[j][k].x;\n\t    if (d == 1 || d == 2) szs[k].y = szs[k].y+seeds[j][k].y;\n\t  }\n\t  add(szs, a*b);\n\t}\n      }\n    }\n  }\n\n  {\n    vector<vector<int>> seedsx, seedsy;\n    for (vector<point>&seed : seeds) {\n      vector<int> seedx, seedy;\n      for (point p : seed) {\n\tseedx.push_back(p.x);\n\tseedy.push_back(p.y);\n      }\n      seedsx.push_back(seedx);\n      seedsy.push_back(seedy);\n    }\n    vector<int> targetx, targety;\n    for (point p : target) {\n      targetx.push_back(p.x);\n      targety.push_back(p.y);\n    }\n    auto [bestx, scorex] = solveSingle(seedsx, targetx);\n    auto [besty, scorey] = solveSingle(seedsy, targety);\n\n    vector<point> combined;\n    for (int i = 0; i < n; i++) {\n      combined.push_back({bestx[i], besty[i]});\n    }\n    double combloss = scorex*scorey;\n    add(combined, combloss);\n  }\n\n  //cout << best.first << \' \' << target.size() << \' \' << best.second << endl;\n  //assert(best.first == target.size());\n  return ans;\n}\n\nvector<point> bruteSize(Image_ test_in, vector<pair<Image,Image>> train) {\n  vector<point> out_sizes;\n  for (auto [in,out] : train) {\n    out_sizes.push_back(out.sz);\n  }\n  int cp = MAXDEPTH;\n  MAXDEPTH = min(MAXDEPTH, 30);\n  Pieces pieces;\n  {\n    vector<DAG> dags = brutePieces2(test_in, train, {});\n    pieces = makePieces2(dags, train, {});\n  }\n  int dags = pieces.dag.size();\n\n  MAXDEPTH = cp;\n\n  vector<point> target;\n  for (auto [in,out] : train) target.push_back(out.sz);\n\n  //cout << pieces.piece.size() << endl;\n  //cout << pieces.seen.size() << endl;\n  vector<vector<point>> seeds;\n  set<vector<point>> seen;\n  for (Piece3&p : pieces.piece) {\n    vector<point> sz;\n    int ok = 1;\n    int*ind = &pieces.mem[p.memi];\n    for (int ti = 0; ti <= train.size(); ti++) {\n      if (pieces.dag[ti].tiny_node[ind[ti]].isvec) ok = 0;\n      else {\n\tsz.push_back(pieces.dag[ti].getImg(ind[ti]).sz);\n      }\n    }\n    if (ok) {\n      if (seen.insert(sz).second)\n\tseeds.push_back(sz);\n    }\n  }\n\n  point ans = solveSize(seeds, target);\n  out_sizes.push_back(ans);\n  return out_sizes;\n}\n\n', 'vendor/icecuber/src/compose2.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "image_functions2.hpp"\n#include "visu.hpp"\n\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "compose2.hpp"\n\n#include "timer.hpp"\n\n/*\nstruct Candidate {\n  vector<Image> imgs;\n  double score;\n};\n*/\n\n\nextern int print_times;\n\n\nstruct mybitset {\n  vector<ull> data;\n  mybitset(int n) {\n    data.resize((n+63)/64);\n  }\n  int operator[](int i) {\n    return data[i>>6]>>(i&63)&1;\n  }\n  void set(int i, ull v) {\n    int bit = i&63;\n    data[i>>6] &= ~(1ull<<bit);\n    data[i>>6] |= (v<<bit);\n  }\n  ull hash() {\n    ull r = 1;\n    for (ull h : data) {\n      r = r*137139+h;\n    }\n    return r;\n  }\n};\n\n\n\nint popcount64c(ull x) {\n  const uint64_t m1  = 0x5555555555555555; //binary: 0101...\n  const uint64_t m2  = 0x3333333333333333; //binary: 00110011..\n  const uint64_t m4  = 0x0f0f0f0f0f0f0f0f; //binary:  4 zeros,  4 ones ...\n  const uint64_t m8  = 0x00ff00ff00ff00ff; //binary:  8 zeros,  8 ones ...\n  const uint64_t m16 = 0x0000ffff0000ffff; //binary: 16 zeros, 16 ones ...\n  const uint64_t m32 = 0x00000000ffffffff; //binary: 32 zeros, 32 ones\n  const uint64_t h01 = 0x0101010101010101; //the sum of 256 to the power of 0,1,2,3...\n  x -= (x >> 1) & m1;             //put count of each 2 bits into those 2 bits\n  x = (x & m2) + ((x >> 2) & m2); //put count of each 4 bits into those 4 bits\n  x = (x + (x >> 4)) & m4;        //put count of each 8 bits into those 8 bits\n  return (x * h01) >> 56;  //returns left 8 bits of x + (x<<8) + (x<<16) + (x<<24) + ...\n}\n\nint popcount64d(ull x) {\n  int pop = 0;\n  while (x) {\n    x &= x-1;\n    pop++;\n  }\n  return pop;\n}\n\n\n\nvector<Candidate> greedyCompose2(Pieces&pieces, vector<Image>&target, vector<point> out_sizes) {\n  if (pieces.piece.empty()) return {};\n\n  Timer greedy_fill_time;\n\n  {\n    int d = -1;\n    for (Piece3&p : pieces.piece) {\n      assert(p.depth >= d);\n      d = p.depth;\n    }\n  }\n\n  vector<Image> init;\n  vector<int> sz;\n  {\n    for (int i = 0; i < pieces.dag.size(); i++) {\n      if (i < target.size()) assert(out_sizes[i] == target[i].sz);\n      init.push_back(core::full(out_sizes[i], 10));\n      sz.push_back(init.back().mask.size());\n    }\n  }\n\n  vector<Candidate> rets;\n\n  int n = pieces.piece.size();\n\n  int M = 0;\n  for (int s : sz) M += s;\n\n  const int M64 = (M+63)/64;\n  vector<ull> bad_mem, active_mem;\n  //vector<mybitset> bad_mem, active_mem;\n  vector<int> img_ind, bad_ind, active_ind;\n\n  {\n    active_mem.reserve(n*M64);\n    bad_mem.reserve(n*M64);\n    TinyHashMap seen;\n    mybitset badi(M), blacki(M);\n    for (int i = 0; i < n; i++) {\n      int x = 0, y = 0;\n      for (int j = 0; j < sz.size(); j++) {\n\tint*ind = &pieces.mem[pieces.piece[i].memi];\n\tImage_ img = pieces.dag[j].getImg(ind[j]);\n\tconst vector<char>&p = img.mask;\n\tconst vector<char>&t = j < target.size() ? target[j].mask : init[j].mask;\n\tassert(p.size() == sz[j]);\n\tassert(t.size() == sz[j]);\n\tfor (int k = 0; k < sz[j]; k++) {\n\t  badi.set  (x++, (p[k] != t[k]));\n\t  blacki.set(y++, (p[k] == 0));\n\t}\n      }\n      img_ind.push_back(i);\n\n      active_ind.push_back(active_mem.size());\n      for (ull v : blacki.data)\n\tactive_mem.push_back(v);\n\n      bad_ind.push_back(bad_mem.size());\n      for (ull v : badi.data)\n\tbad_mem.push_back(v);\n    }\n  }\n\n  //cout << active_mem.size() << endl;\n  //cout << bad_mem.size() << endl;\n  //exit(0);\n\n  int max_piece_depth = 0;\n  for (int i = 0; i < n; i++)\n    max_piece_depth = max(max_piece_depth, pieces.piece[i].depth);\n\n  //mt19937 mrand(0);\n\n\n\n  auto greedyComposeCore = [&](mybitset&cur, const mybitset&careMask, const int piece_depth_thres, vImage&ret) {\n    vector<int> sparsej;\n    for (int j = 0; j < M64; j++) {\n      if (~cur.data[j] & careMask.data[j]) sparsej.push_back(j);\n    }\n\n    vector<ull> best_active(M64), tmp_active(M64);\n    int besti = -1;\n    pair<int,int> bestcnt = {0,0};\n    //for (int i : order) {\n    //  if (skip[i] || pieces.piece[img_ind[i]].depth > piece_depth_thres) continue;\n    for (int i = 0; i < img_ind.size(); i++) {\n      if (pieces.piece[img_ind[i]].depth > piece_depth_thres) continue;\n\n      for (int k : {0,1,2}) {\n\tconst ull*active_data = &active_mem[active_ind[i]];\n\n\tull flip = (k == 0 ? ~0 : 0);\n\tull full = (k == 2 ? ~0 : 0);\n\n\tconst ull*bad_data = &bad_mem[bad_ind[i]];\n\tint cnt = 0, covered = 0, ok = 1;\n\tfor (int j = 0; j < M64; j++) {\n\t  ull active = ((active_data[j]^flip) | full);\n\t  if (~cur.data[j] & bad_data[j] & active) {\n\t    ok = 0;\n\t    break;\n\t  }\n\t}\n\tif (!ok) continue;\n\tfor (int j : sparsej) {\n\t  ull active = ((active_data[j]^flip) | full);\n\t  cnt += popcount64d(active & ~cur.data[j] & careMask.data[j]);\n\t  //covered += popcount64d(bad.data[j] & cur.data[j] & careMask.data[j]);\n\t}\n\tif (ok && make_pair(cnt,-covered) > bestcnt) {\n\t  bestcnt = make_pair(cnt,-covered);\n\t  besti = i;\n\n\t  if (k == 0) {\n\t    for (int j = 0; j < M64; j++)\n\t      tmp_active[j] = ~active_data[j];\n\t  } else if (k == 1) {\n\t    for (int j = 0; j < M64; j++)\n\t      tmp_active[j] = active_data[j];\n\t  } else {\n\t    for (int j = 0; j < M64; j++)\n\t      tmp_active[j] = ~0;\n\t  }\n\t  best_active = tmp_active;\n\t}\n      }\n    }\n\n    if (besti == -1) return -1;\n\n    {\n      int i = img_ind[besti], x = 0;\n\n      int depth = pieces.piece[i].depth;\n\n      for (int l = 0; l < ret.size(); l++) {\n\tint*ind = &pieces.mem[pieces.piece[i].memi];\n\tconst vector<char>&mask = pieces.dag[l].getImg(ind[l]).mask;\n\tfor (int j = 0; j < sz[l]; j++) {\n\t  if ((best_active[x>>6]>>(x&63)&1) && ret[l].mask[j] == 10) {\n\t    //assert(cur[x-1] == 0);\n\t    ret[l].mask[j] = mask[j];\n\t  }\n\n\t  x++;\n\t}\n      }\n      for (int j = 0; j < M; j++) {\n\tif (best_active[j>>6]>>(j&63)&1) cur.set(j, 1);\n      }\n\n      return depth;\n    }\n  };\n\n\n\n\n\n\n\n\n\n  map<ull,Image> greedy_fill_mem;\n\n  int maxiters = 10;\n\n\n  //vector<int> skip(img_ind.size());\n  //vector<int> order;\n  //for (int i = 0; i < img_ind.size(); i++) order.push_back(i);\n\n  for (int pdt = max_piece_depth%10; pdt <= max_piece_depth; pdt += 10) {\n    int piece_depth_thres = pdt;\n\n    for (int it0 = 0; it0 < 10; it0++) {\n      for (int mask = 1; mask < min(1<<target.size(), 1<<5); mask++) {\n\tvector<int> maskv;\n\tfor (int j = 0; j < target.size(); j++)\n\t  if (mask>>j&1) maskv.push_back(j);\n\n\tint caremask;\n\tif (it0 < maskv.size()) {\n\t  caremask = 1<<maskv[it0];\n\t} else {\n\t  continue;\n\t  /*caremask = 1<<maskv[mrand()%maskv.size()];\n\t    for (int i = 0; i < n; i++)\n\t    skip[i] = mrand()%3;\n\t    random_shuffle(order.begin(), order.end());*/\n\t}\n\n\tmybitset cur(M), careMask(M);\n\t{\n\t  int base = 0;\n\t  for (int j = 0; j < sz.size(); j++) {\n\t    if (!(mask>>j&1))\n\t      for (int k = 0; k < sz[j]; k++)\n\t\tcur.set(base+k, 1);\n\t    if ((caremask>>j&1))\n\t      for (int k = 0; k < sz[j]; k++)\n\t\tcareMask.set(base+k, 1);\n\t    base += sz[j];\n\t  }\n\t}\n\n\n\tint cnt_pieces = 0;\n\tvector<int> piece_depths;\n\tint sum_depth = 0, max_depth = 0;\n\n\tvector<Image> ret = init;\n\tfor (int it = 0; it < maxiters; it++) {\n\n\t  int depth = greedyComposeCore(cur, careMask, piece_depth_thres, ret);\n\t  if (depth == -1) break;\n\t  piece_depths.push_back(depth);\n\t  cnt_pieces++;\n\t  sum_depth += depth;\n\t  max_depth = max(max_depth, depth);\n\n\n\t  {\n\t    greedy_fill_time.start();\n\t    vImage cp = ret;\n\t    int carei = 31-__builtin_clz(caremask);\n\t    assert(caremask == 1<<carei);\n\t    int ok = 1;\n\t    {\n\t      Image& img = cp[carei];\n\t      for (char&c : img.mask) if (c == 10) c = 0;\n\t      img = greedyFillBlack(img);\n\t      if (img != target[carei]) ok = 0;\n\t    }\n\t    if (ok) {\n\t      for (int i = 0; i < cp.size(); i++) {\n\t\tif (i == carei) continue;\n\t\tImage& img = cp[i];\n\t\tfor (char&c : img.mask) if (c == 10) c = 0;\n\t\tull h = hashImage(img);\n\t\tif (!greedy_fill_mem.count(h)) {\n\t\t  greedy_fill_mem[h] = greedyFillBlack(img);\n\t\t}\n\t\timg = greedy_fill_mem[h];\n\t\tif (img.w*img.h <= 0) ok = 0;\n\t      }\n\t      if (ok)\n\t\trets.emplace_back(cp, cnt_pieces+1, sum_depth, max_depth);\n\t    }\n\t    greedy_fill_time.stop();\n\t  }\n\t}\n\n\t/*for (Image&img : ret)\n\t  for (char&c : img.mask)\n\t  if (c == 10) c = 0;\n\t*/\n\n\trets.emplace_back(ret, cnt_pieces, sum_depth, max_depth);\n      }\n    }\n  }\n\n  if (print_times)\n    greedy_fill_time.print("Greedy fill time");\n\n  return rets;\n}\n\n\n\nvector<Candidate> composePieces2(Pieces&pieces, vector<pair<Image, Image>> train, vector<point> out_sizes) {\n  vector<Candidate> cands;\n\n  vector<Image> target;\n  for (auto [in,out] : train)\n    target.push_back(out);\n\n  /*\n  for (Piece3&p : pieces.piece) {\n    vector<Image> imgs;\n    //assert(p.ind.size() == pieces.dag.size());\n    int*ind = &pieces.mem[p.memi];\n    for (int i = 0; i < pieces.dag.size(); i++) {\n      //assert(p.ind[i] >= 0 && p.ind[i] < (int)pieces.dag[i].node.size());\n      imgs.push_back(pieces.dag[i].getImg(ind[i]));\n    }\n    cands.emplace_back(imgs, 1, p.depth, p.depth);\n  }\n  */\n\n  for (const Candidate&cand : greedyCompose2(pieces, target, out_sizes)) {\n    cands.push_back(cand);\n  }\n  return cands;\n}\n\n\nvector<Candidate> evaluateCands(const vector<Candidate>&cands, vector<pair<Image,Image>> train) {\n  vector<Candidate> ret;\n  for (const Candidate& cand : cands) {\n    vImage_ imgs = cand.imgs;\n    assert(cand.max_depth >= 0 && cand.max_depth < 100);\n    assert(cand.cnt_pieces >= 0 && cand.cnt_pieces < 100);\n    //cout << cand.max_depth << \' \' << cand.cnt_pieces << endl;\n    double prior = cand.max_depth+cand.cnt_pieces*1e-3;//cnt_pieces;\n\n    int goods = 0;\n    for (int i = 0; i < train.size(); i++) {\n      goods += (imgs[i] == train[i].second);\n    }\n    double score = goods-prior*0.01;\n\n    Image answer = imgs.back();\n    if (answer.w > 30 || answer.h > 30 || answer.w*answer.h == 0) goods = 0;\n    for (int i = 0; i < answer.h; i++)\n      for (int j = 0; j < answer.w; j++)\n\tif (answer(i,j) < 0 || answer(i,j) >= 10) goods = 0;\n\n    if (goods)\n      ret.emplace_back(imgs, score);\n  }\n  sort(ret.begin(), ret.end());\n  //printf("%.20f\\n\\n", ret[0].score);\n  return ret;\n}\n', 'vendor/icecuber/src/core_functions.cpp': '/*#include <vector>\n#include <cassert>\n#include <iostream>\n#include <tuple>\n#include <functional>*/\n#include "precompiled_stl.hpp"\nusing namespace std;\n\n#include "utils.hpp"\n#include "core_functions.hpp"\n\nnamespace core {\n  int colMask(Image_ img) {\n    int mask = 0;\n    for (int i = 0; i < img.h; i++)\n      for (int j = 0; j < img.w; j++)\n\tmask |= 1<<img(i,j);\n    return mask;\n  }\n  int countCols(Image_ img, int include0) {//include0 = 0\n    int mask = colMask(img);\n    if (!include0) mask = mask&~1;\n    return __builtin_popcount(mask);\n  }\n  int count(Image_ img) {\n    int ans = 0;\n    for (int i = 0; i < img.h; i++)\n      for (int j = 0; j < img.w; j++)\n\tans += img(i,j) > 0;\n    return ans;\n  }\n\n\n  Image full(point p, point sz, int filling) {//filling = 1\n    Image ret;\n    ret.p = p;\n    ret.sz = sz;\n    ret.mask.assign(ret.h*ret.w, filling);\n    return ret;\n  }\n\n  Image empty(point p, point sz) {\n    return full(p, sz, 0);\n  }\n\n  Image full(point sz, int filling) {//filling = 1\n    Image ret;\n    ret.p = {0,0};\n    ret.sz = sz;\n    ret.mask.assign(ret.h*ret.w, filling);\n    return ret;\n  }\n\n  Image empty(point sz) {\n    return full(point{0,0}, sz, 0);\n  }\n\n\n  bool isRectangle(Image_ a) {\n    return count(a) == a.w*a.h;\n  }\n\n  void countComponents_dfs(Image&img, int r, int c) {\n    img(r,c) = 0;\n    for (int nr = r-1; nr <= r+1; nr++)\n      for (int nc = c-1; nc <= c+1; nc++)\n\tif (nr >= 0 && nr < img.h && nc >= 0 && nc < img.w && img(nr,nc))\n\t  countComponents_dfs(img,nr,nc);\n  }\n\n  int countComponents(Image img) {\n    int ans = 0;\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tif (img(i,j)) {\n\t  countComponents_dfs(img,i,j);\n\t  /*function<void(int,int)> dfs = [&](int r, int c) {\n\t    if (r < 0 || r >= img.h || c < 0 || c >= img.w || !img(r,c)) return;\n\t    img(r,c) = 0;\n\t    for (int nr : {r-1,r,r+1})\n\t      for (int nc : {c-1,c,c+1})\n\t\tdfs(nr,nc);\n\t  };\n\t  dfs(i,j);*/\n\t  ans++;\n\t}\n      }\n    }\n    return ans;\n  }\n\n\n  char majorityCol(Image_ img, int include0) { //include0 = 0\n    int cnt[10] = {};\n    for (int i = 0; i < img.h; i++)\n      for (int j = 0; j < img.w; j++) {\n\tchar c = img(i,j);\n\tif (c >= 0 && c < 10)\n\t  cnt[c]++;\n      }\n    if (!include0) cnt[0] = 0;\n    int ret = 0;\n    int ma = cnt[ret];\n    for (int c = 1; c < 10; c++) {\n      if (cnt[c] > ma) {\n\tma = cnt[c];\n\tret = c;\n      }\n    }\n    return ret;\n  }\n  Image subImage(Image_ img, point p, point sz) {\n    assert(p.x >= 0 && p.y >= 0 && p.x+sz.x <= img.w && p.y+sz.y <= img.h && sz.x >= 0 && sz.y >= 0);\n    Image ret;\n    ret.p = p+img.p;\n    ret.sz = sz;\n    ret.mask.resize(ret.w*ret.h);\n    for (int i = 0; i < ret.h; i++)\n      for (int j = 0; j < ret.w; j++)\n\tret(i,j) = img(i+p.y, j+p.x);\n    return ret;\n  }\n\n  vector<pair<Image,int>> splitCols(Image_ img, int include0) { //include0 = 0\n    vector<pair<Image,int>> ret;\n    int mask = colMask(img);\n    for (char c = !include0; c < 10; c++) {\n      if (mask>>c&1) {\n\tImage s = img;\n\tfor (int i = 0; i < s.h; i++)\n\t  for (int j = 0; j < s.w; j++)\n\t    s(i,j) = s(i,j) == c;\n\tret.emplace_back(s, c);\n      }\n    }\n    return ret;\n  }\n};\n', 'vendor/icecuber/src/deduce_op.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n\n#include "visu.hpp"\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "compose2.hpp"\n\n#include "deduce_op.hpp"\n\npair<Image,Image> iOuterProductSI(Image_ img, int w, int h) {\n  if (img.w*img.h <= 0 || img.w%w || img.h%h) return {badImg,badImg};\n  Image big = core::full({img.w/w,img.h/h},-1);\n  Image small = core::full({w,h},-1);\n\n  for (int ii = 0; ii < img.h/h; ii++) {\n    for (int jj = 0; jj < img.w/w; jj++) {\n      int all0 = 1;\n      for (int i = 0; i < h; i++)\n\tfor (int j = 0; j < w; j++)\n\t  if (img(ii*h+i,jj*w+j)) all0 = 0;\n\n      big(ii,jj) = !all0;\n\n      if (!all0) {\n\tfor (int i = 0; i < h; i++) {\n\t  for (int j = 0; j < w; j++) {\n\t    char& a = small(i,j);\n\t    char b = img(ii*h+i,jj*w+j);\n\t    if (a != -1 && a != b) return {badImg,badImg};\n\t    a = b;\n\t  }\n\t}\n      }\n    }\n  }\n  return {big, small};\n}\n\n\npair<Image,Image> iOuterProductIS(Image_ img, int w, int h) {\n  if (img.w*img.h <= 0 || img.w%w || img.h%h) return {badImg,badImg};\n  Image big = core::full({img.w/w,img.h/h},-1);\n  Image small = core::full({w,h},-1);\n\n  for (int ii = 0; ii < img.h/h; ii++) {\n    for (int jj = 0; jj < img.w/w; jj++) {\n      int mask = 0;\n      for (int i = 0; i < h; i++)\n\tfor (int j = 0; j < w; j++)\n\t  mask |= 1<<img(ii*h+i,jj*w+j);\n\n      if (__builtin_popcount(mask&~1) > 1) return {badImg,badImg};\n      big(ii,jj) = 31-__builtin_clz(mask);\n      if (big(ii,jj)) {\n\tfor (int i = 0; i < h; i++) {\n\t  for (int j = 0; j < w; j++) {\n\t    char& a = small(i,j);\n\t    char b = img(ii*h+i,jj*w+j) > 0;\n\t    if (a != -1 && a != b) return {badImg,badImg};\n\t    a = b;\n\t  }\n\t}\n      }\n    }\n  }\n  return {big, small};\n}\n\n\ndeduceOuterProduct::deduceOuterProduct(vector<pair<Image,Image>> train) {\n\n  auto score = [](vector<pair<Image,Image>> pa, int fi) {\n    double ans = 0;\n    for (int k : {0,1}) {\n      vector<Image> imgs;\n      int allequal = 1;\n      for (int ti = 0; ti < pa.size(); ti++) {\n\tImage img = k ? pa[ti].second : pa[ti].first;\n\tif (img.w*img.h <= 0) return 1e9;\n\n\timgs.push_back(img);\n\tif (imgs[0] != imgs.back()) allequal = 0;\n      }\n      if (allequal && imgs.size() > 1) continue;\n\n      for (Image_ img : imgs) {\n\tint cols = __builtin_popcount(core::colMask(img)&~1);\n\tif (cols <= 1 && core::isRectangle(img)) {\n\t  ans += log(img.w+1)+log(img.h+1);\n\t} else if (cols <= 1) {\n\t  ans += log(2)*img.w*img.h;\n\t} else {\n\t  ans += log(10)*img.w*img.h;\n\t}\n      }\n      //ans *= 1-1e-5*1;//(fi*2-1);\n    }\n    return ans;\n  };\n\n\n  int minw = 1e9, minh = 1e9;\n  for (auto [in,out] : train) {\n    minw = min(minw, out.w);\n    minh = min(minh, out.h);\n  }\n\n  /*{\n    Image img = train[0].second;\n    auto [a,b] = iOuterProductSI(img,img.w,img.h);\n    assert(reconstruct(a,b) == img);\n    exit(0);\n    }*/\n\n  rec_funci = -1;\n  double best_score = 1e9;\n  for (int h = 1; h <= minh; h++) {\n    for (int w = 1; w <= minw; w++) {\n      for (int l : {0,1}) {\n\tfor (int k : {0,1}) {\n\t  auto f = k ? iOuterProductSI : iOuterProductIS;\n\t  vector<pair<Image,Image>> is;\n\t  int bad = 0;\n\t  for (auto [in,out] : train) {\n\t    int sw = w, sh = h;\n\t    if (l) {\n\t      if (out.w%w || out.h%h) bad = 1;\n\t      sw = out.w/w;\n\t      sh = out.h/h;\n\t    }\n\t    is.push_back(f(out, sw, sh));\n\t  }\n\t  double entropy = score(is,k);\n\t  if (entropy < best_score) {\n\t    best_score = entropy;\n\t    rec_funci = k;\n\t    train_targets = is;\n\t  }\n\t}\n      }\n    }\n  }\n  assert(rec_funci != -1);\n\n  for (int k : {0,1}) {\n    auto f = k ? iOuterProductSI : iOuterProductIS;\n    vector<double> best_at(train.size(), 1e9);\n    vector<pair<Image,Image>> best_single(train.size(), {badImg,badImg});\n    for (int ti = 0; ti < train.size(); ti++) {\n      Image target = train[ti].second;\n      for (int h = 1; h <= target.h; h++) {\n\tfor (int w = 1; w <= target.w; w++) {\n\t  auto is = f(target, w, h);\n\t  double entropy = score({is},k);\n\t  if (entropy < best_at[ti]) {\n\t    best_at[ti] = entropy;\n\t    best_single[ti] = is;\n\t  }\n\t}\n      }\n    }\n    double entropy = score(best_single,k);\n    if (entropy < best_score) {\n      best_score = entropy;\n      rec_funci = k;\n      train_targets = best_single;\n    }\n  }\n\n  assert(rec_funci != -1);\n  assert(train_targets.size() == train.size());\n  for (int ti = 0; ti < train.size(); ti++) {\n    Image a, b;\n    tie(a,b) = train_targets[ti];\n    //print(a);\n    //print(b);\n    //print(train[ti].second);\n    assert(reconstruct(a,b) == train[ti].second);\n    //cout << "OK" << endl;\n    }\n}\n\n\nImage deduceOuterProduct::reconstruct(Image_ a, Image_ b) {\n  auto f = rec_funci ? outerProductSI : outerProductIS;\n  return f(a,b);\n}\n\nextern int MAXDEPTH;\n\nvoid addDeduceOuterProduct(Pieces&pieces, vector<pair<Image,Image>> train, vector<Candidate>&cands) {\n  deduceOuterProduct deduce_op(train);\n\n  int interestings = 0;\n  for (auto [in,out] : deduce_op.train_targets) {\n    if (core::count(in) > 1 && core::count(out) > 1) interestings++;\n  }\n  if (interestings*2 < train.size()) return;\n\n  vImage a, b;\n  for (int k : {0,1}) {\n    vImage&cand = k ? b : a;\n    int best_match = -1;\n\n    auto add = [&](vImage_ vi) {\n      int matches = 0;\n      for (int i = 0; i < train.size(); i++) {\n\tImage_ target = k ? deduce_op.train_targets[i].second : deduce_op.train_targets[i].first;\n\tmatches += (vi[i] == target);\n      }\n      if (matches > best_match) {\n\tbest_match = matches;\n\tcand = vi;\n      }\n    };\n\n    for (int pi = 0; pi < pieces.mem.size(); pi += pieces.dag.size()) {\n      int*ind = &pieces.mem[pi];\n      //TODO: Use hashes to compare instead of full images\n      vImage imgs;\n      for (int i = 0; i <= train.size(); i++) {\n\tif (pieces.dag[i].tiny_node[ind[i]].isvec) continue;\n\tImage_ img = pieces.dag[i].getImg(ind[i]);\n\timgs.push_back(img);\n\timgs.back().p = point{0,0};\n      }\n      if (imgs.size() == train.size()+1)\n\tadd(imgs);\n    }\n    for (auto [x,y] : deduce_op.train_targets) {\n      add(vImage(train.size()+1, (k ? y : x)));\n    }\n  }\n\n  {\n    assert(a.size() == train.size()+1);\n    assert(b.size() == train.size()+1);\n\n    // TODO: Use correct depths\n    vImage imgs;\n    for (int i = 0; i <= train.size(); i++)\n      imgs.push_back(deduce_op.reconstruct(a[i],b[i]));\n    cands.emplace_back(imgs, 2, MAXDEPTH, MAXDEPTH*2);\n  }\n\n  /*\n  visu.next(to_string(si));\n  for (auto [in,out] : train)\n    visu.add(in,out);\n\n  visu.next(to_string(si)+"!");\n  for (auto [in,out] : deduce_op.train_targets) {\n    visu.add(in,out);\n    }*/\n}\n', 'vendor/icecuber/src/efficient.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "brute2.hpp"\n#include "visu.hpp"\n\npair<int,bool> TinyHashMap::insert(ull key, int value) {\n  if (table.size() <= data.size()*sparse_factor) {\n    table.resize(max((int)(table.size() * resize_step), minsz));\n    assert((table.size()&(table.size()-1)) == 0);\n    mask = table.size()-1;\n\n    fill(table.begin(), table.end(), -1);\n    for (int i = 0; i < data.size(); i++) {\n      int&head = table[data[i].key&mask];\n      data[i].next = head;\n      head = i;\n    }\n  }\n  int&head = table[key&mask];\n  int previ = head;\n  while (1) {\n    if (previ == -1) {\n      data.push_back({key, value, head});\n      head = data.size()-1;\n      return {value, true};\n    } else if (data[previ].key == key) {\n      return {data[previ].value, false};\n    }\n    previ = data[previ].next;\n  }\n}\n\n\n\n\nvoid TinyChildren::add(int fi, int to) {\n  assert(fi >= 0);\n  if (sz+1 == dense_thres) {\n    //Convert to dense\n    cap = max(sparse[sz-1].first,fi)+1; //Max fi\n    pair<int,int>*old = sparse;\n    dense = new int[cap];\n    fill_n(dense, cap, None);\n    dense[fi] = to;\n    for (int i = 0; i < sz; i++) {\n      auto [fi, to] = old[i];\n      assert(fi >= 0);\n      assert(fi < cap);\n      dense[fi] = to;\n    }\n    assert(old);\n    delete[]old;\n    sz = dense_thres;\n  }\n\n  if (sz < dense_thres) {\n    //Sparse\n    if (sz+1 > cap) {\n      cap = max((cap+1)*3/2,4);\n      pair<int,int>*old = sparse;\n      sparse = new pair<int,int>[cap];\n      copy_n(old, sz, sparse);\n      if (old) delete[]old;\n    }\n    {\n      int p = sz++;\n      while (p && sparse[p-1].first > fi) {\n\tsparse[p] = sparse[p-1];\n\tp--;\n      }\n      sparse[p] = {fi, to};\n    }\n  } else {\n    //Dense\n    if (cap <= fi) {\n      int oldcap = cap;\n      int*old = dense;\n      cap = (fi+1)*3/2;\n      dense = new int[cap];\n      fill_n(dense+oldcap, cap-oldcap, None);\n      copy_n(old, oldcap, dense);\n      assert(old);\n      delete[]old;\n    }\n    dense[fi] = to;\n  }\n}\nint TinyChildren::get(int fi) {\n  assert(fi >= 0);\n  if (sz < dense_thres) {\n    int low = 0, high = sz-1;\n    while (low <= high) {\n      int mid = (low+high)>>1;\n      int cfi = sparse[mid].first;\n      if (cfi == fi) return sparse[mid].second;\n      else if (cfi > fi) high = mid-1;\n      else low = mid+1;\n    }\n    return None;\n  } else {\n    if (fi >= cap) return None;\n    return dense[fi];\n  }\n}\nvoid TinyChildren::legacy(vector<pair<int,int>>&ret) {\n  if (sz < dense_thres) {\n    //Sparse\n    ret.resize(sz);\n    for (int i = 0; i < sz; i++)\n      ret[i] = sparse[i];\n  } else {\n    //Dense\n    ret.resize(0);\n    for (int i = 0; i < cap; i++) {\n      if (dense[i] != None)\n\tret.emplace_back(i, dense[i]);\n    }\n  }\n}\n\n\n\n\n\n\n\nTinyImage::TinyImage(Image_ img, TinyBank&bank) {\n  for (int c : {img.x, img.y, img.w, img.h}) {\n    assert(c >= -128 && c < 128);\n  }\n  x = img.x, y = img.y, w = img.w, h = img.h;\n\n  int freq[10] = {};\n  for (char c : img.mask) {\n    assert(c >= 0 && c < 10);\n    freq[c]++;\n  }\n\n  priority_queue<pair<int,int>> pq;\n  for (int d = 0; d < 10; d++) {\n    if (freq[d]) {\n      pq.emplace(-freq[d], -d);\n      //cout << d << ": " << freq[d] << endl;\n    }\n  }\n  while (pq.size() < 2) pq.emplace(0,0);\n\n  int nodes = pq.size()-1;\n  int pos = nodes-1;\n  auto convert = [](int a, int p) {\n    if (a <= 0) return -a;\n    else {\n      assert(9+a-p >= 10 && 9+a-p < 16);\n      return 9+a-p;\n    }\n  };\n  while (pq.size() > 1) {\n    auto [mfa,a] = pq.top(); pq.pop();\n    auto [mfb,b] = pq.top(); pq.pop();\n    tree[pos] = convert(a,pos) | convert(b,pos) << 4;\n    pq.emplace(mfa+mfb, pos--);\n  }\n  int code[10] = {}, codelen[10] = {};\n  int path[10] = {}, pathlen[10] = {};\n  for (int p = 0; p < nodes; p++) {\n    for (int k : {0,1}) {\n      int child = tree[p]>>k*4&15;\n      int newpath = path[p] | k<<pathlen[p];\n      if (child < 10) {\n\tcode[child] = newpath;\n\tcodelen[child] = pathlen[p]+1;\n      } else {\n\tchild += p-9;\n\tpath[child] = newpath;\n\tpathlen[child] = pathlen[p]+1;\n      }\n    }\n    //cout << p << ": " << (int)tree[p][0] << \' \' << (int)tree[p][1] << endl;\n  }\n  /*for (int d = 0; d < 10; d++) {\n    cout << d << ": " << bitset<4>(code[d]) << \' \' << codelen[d] << endl;\n    }*/\n\n  assert((bank.curi+align-1)/align < 1ll<<32);\n  memi = (bank.curi+align-1)/align;\n  sz = 0;\n  ll memstart = (ll)memi*align;\n  bank.alloc();\n  for (char c : img.mask) {\n    bank.set(memstart+ sz, code[c], codelen[c]);\n    sz += codelen[c];\n    //for (int i = 0; i < codelen[c]; i++)\n    //bank.set(memstart+ sz++, code[c]>>i&1);\n  }\n  /*for (int it = 0; it < 10; it++)\n    cout << (int)img.mask[it] << \' \';\n    cout << endl;\n    for (int it = 0; it < 20; it++)\n    cout << bank.get(memi+it);\n    cout << endl;*/\n  bank.curi = memstart+sz;\n}\n\n\nImage TinyImage::decompress(TinyBank&bank) {\n  Image ret;\n  ret.x = x, ret.y = y, ret.w = w, ret.h = h;\n  ret.mask.resize(ret.w*ret.h);\n  int treep = 0, maski = 0;\n  ll memstart = (ll)memi*align;\n  for (ll i = memstart; i < memstart+sz; i++) {\n    int bit = bank.get(i);\n    int child = tree[treep]>>bit*4&15;\n    if (child < 10) {\n      ret.mask[maski++] = child;\n      treep = 0;\n    } else\n      treep += child-9;\n  }\n  //cout << maski << \' \' << ret.w*ret.h << endl;\n  assert(maski == ret.w*ret.h);\n  assert(treep == 0);\n  return ret;\n}\n', 'vendor/icecuber/src/image_functions.cpp': '#include "precompiled_stl.hpp"\n/*#include <vector>\n#include <cassert>\n#include <queue>\n#include <tuple>\n#include <functional>*/\nusing namespace std;\n\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n\nImage Col(int id) {\n  assert(id >= 0 && id < 10);\n  return core::full({0,0}, {1,1}, id);\n}\nImage Pos(int dx, int dy) {\n  return core::full({dx,dy},{1,1});\n}\nImage Square(int id) {\n  assert(id >= 1);\n  return core::full({0,0}, {id,id});\n}\nImage Line(int orient, int id) {\n  assert(id >= 1);\n  int w = id, h = 1;\n  if (orient) swap(w,h);\n  return core::full({0,0}, {w,h});\n}\n\nImage getPos(Image_ img) {\n  return core::full(img.p, {1,1}, core::majorityCol(img));\n}\nImage getSize(Image_ img) {\n  return core::full({0,0}, img.sz, core::majorityCol(img));\n}\nImage hull(Image_ img) {\n  return core::full(img.p, img.sz, core::majorityCol(img));\n}\nImage toOrigin(Image img) {\n  img.p = {0,0};\n  return img;\n}\n\nImage getW(Image_ img, int id) {\n  return core::full({0,0}, {img.w,id ? img.w : 1}, core::majorityCol(img));\n}\nImage getH(Image_ img, int id) {\n  return core::full({0,0}, {id ? img.h : 1, img.h}, core::majorityCol(img));\n}\n\nImage hull0(Image_ img) {\n  return core::full(img.p, img.sz, 0);\n}\nImage getSize0(Image_ img) {\n  return core::full({0,0}, img.sz, 0);\n}\n\n\n\nImage Move(Image img, Image_ p) {\n  img.x += p.x;\n  img.y += p.y;\n  return img;\n}\n\nImage filterCol(Image_ img, Image_ palette) {\n  Image ret = img;\n  int palMask = core::colMask(palette);\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      if ((palMask>>img(i,j)&1) == 0)\n\tret(i,j) = 0;\n  return ret;\n}\n\nImage filterCol(Image_ img, int id) {\n  assert(id >= 0 && id < 10);\n  if (id == 0) return invert(img);\n  else return filterCol(img, Col(id));\n}\n\n\n\nImage broadcast(Image_ col, Image_ shape, int include0) { //include0 = 1\n  if (col.w*col.h == 0 || shape.w*shape.h == 0) return badImg;\n\n  if (shape.w%col.w == 0 && shape.h%col.h == 0) {\n    Image ret = shape;\n    int dh = shape.h/col.h, dw = shape.w/col.w;\n    for (int ii = 0; ii < col.h; ii++) {\n      for (int jj = 0; jj < col.w; jj++) {\n\tint c = col(ii,jj);\n\tfor (int i = ii*dh; i < ii*dh+dh; i++)\n\t  for (int j = jj*dw; j < jj*dw+dw; j++)\n\t    ret(i,j) = c;\n      }\n    }\n    return ret;\n  }\n\n  Image ret = shape;\n  double fh = col.h*1./shape.h, fw = col.w*1./shape.w;\n\n  const double eps = 1e-9;\n  double w0[10] = {};\n  for (int c : col.mask) w0[c] += 1e-6;\n\n  double tot = fh*fw;\n  double weight[10];\n  for (int i = 0; i < shape.h; i++) {\n    for (int j = 0; j < shape.w; j++) {\n      copy_n(w0, 10, weight);\n\n      double r0 = i*fh+eps, r1 = (i+1)*fh-eps;\n      double c0 = j*fw+eps, c1 = (j+1)*fw-eps;\n\n      int guess = !include0;\n      for (int y = r0; y < r1; y++) {\n\tdouble wy = min((double)y+1,r1)-max((double)y,r0);\n\tfor (int x = c0; x < c1; x++) {\n\t  double wx = min((double)x+1,c1)-max((double)x,c0);\n\t  char c = col(y,x);\n\t  weight[c] += wx*wy;\n\t  guess = c;\n\t}\n      }\n\n      if (weight[guess]*2 > tot) {\n\tret(i,j) = guess;\n\tcontinue;\n      }\n\n      int maj = !include0;\n      double w = weight[maj];\n      for (int c = 1; c < 10; c++) {\n\tif (weight[c] > w) maj = c, w = weight[c];\n      }\n      ret(i,j) = maj;\n      //point sz = {max(c1-c0, 1), max(r1-r0, 1)};\n      //ret(i,j) = core::majorityCol(core::subImage(col, {c0,r0}, sz), include0);\n    }\n  }\n  return ret;\n}\n\nImage colShape(Image_ col, Image_ shape) {\n  if (shape.w*shape.h == 0 || col.w*col.h == 0) return badImg;\n  Image ret = broadcast(col, getSize(shape));\n  ret.p = shape.p;\n  for (int i = 0; i < ret.h; i++)\n    for (int j = 0; j < ret.w; j++)\n      if (!shape(i,j)) ret(i,j) = 0;\n  return ret;\n}\nImage colShape(Image_ shape, int id) {\n  assert(id >= 0 && id < 10);\n  Image ret = shape;\n  for (char&c : ret.mask)\n    c = c ? id : 0;\n  return ret;\n}\n\n\n\nImage compress(Image_ img, Image_ bg) { // bg = Col(0)\n  int bgmask = core::colMask(bg);\n\n  int xmi = 1e9, xma = 0, ymi = 1e9, yma = 0;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if ((bgmask>>img(i,j)&1) == 0) {\n\txmi = min(xmi, j);\n\txma = max(xma, j);\n\tymi = min(ymi, i);\n\tyma = max(yma, i);\n      }\n    }\n  }\n  Image ret;\n  if (xmi == 1e9) {\n    ret.p = {0,0};\n    ret.sz = {0,0};\n    return ret;\n  }\n  ret.p = img.p + point{xmi, ymi};\n  ret.sz = {xma-xmi+1, yma-ymi+1};\n  ret.mask.resize(ret.h*ret.w);\n  for (int i = ymi; i <= yma; i++) {\n    for (int j = xmi; j <= xma; j++) {\n      ret(i-ymi,j-xmi) = img(i,j);\n    }\n  }\n  return ret;\n}\n\n\n\nImage embedSlow(Image_ img, Image_ shape) {\n  Image ret = core::empty(shape.p, shape.sz);\n  point d = shape.p-img.p;\n  for (int i = 0; i < ret.h; i++)\n    for (int j = 0; j < ret.w; j++)\n      ret(i,j) = img.safe(i+d.y, j+d.x);\n  return ret;\n}\n\nImage embed(Image_ img, Image_ shape) {\n  Image ret = core::empty(shape.p, shape.sz);\n  point d = shape.p-img.p;\n  int sx = max(0,-d.x);\n  int sy = max(0,-d.y);\n  int ex = min(ret.w,img.w-d.x);\n  int ey = min(ret.h,img.h-d.y);\n\n  int retw = ret.w, imgw = img.w, off = d.y*img.w+d.x;\n  for (int i = sy; i < ey; i++)\n    for (int j = sx; j < ex; j++)\n      ret.mask[i*retw+j] = img.mask[i*imgw+j+off];\n  //assert(ret == embedSlow(img,shape));\n  return ret;\n}\n\n\nImage compose(Image_ a, Image_ b, const function<int(int,int)>&f, int overlap_only) {\n  Image ret;\n  if (overlap_only == 1) {\n    ret.p = {max(a.p.x,b.p.x),\n\t     max(a.p.y,b.p.y)};\n    point ra = a.p+a.sz, rb = b.p+b.sz;\n    ret.sz = {min(ra.x,rb.x),min(ra.y,rb.y)};\n    ret.sz = ret.sz-ret.p;\n    if (ret.w <= 0 || ret.h <= 0) return badImg;\n  } else if (overlap_only == 0) {\n    ret.p = {min(a.p.x,b.p.x),\n\t     min(a.p.y,b.p.y)};\n    point ra = a.p+a.sz, rb = b.p+b.sz;\n    ret.sz = {max(ra.x,rb.x),max(ra.y,rb.y)};\n    ret.sz = ret.sz-ret.p;\n  } else if (overlap_only == 2) {\n    ret.p = a.p;\n    ret.sz = a.sz;\n  } else assert(0);\n  if (ret.w > MAXSIDE || ret.h > MAXSIDE || ret.w*ret.h > MAXAREA) return badImg;\n  ret.mask.assign(ret.w*ret.h, 0);\n  point da = ret.p-a.p;\n  point db = ret.p-b.p;\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      int ca = a.safe(i+da.y, j+da.x);\n      int cb = b.safe(i+db.y, j+db.x);\n      ret(i,j) = f(ca,cb);\n    }\n  }\n  return ret;\n}\n\n\nImage compose(Image_ a, Image_ b, int id) { //id = 0\n  if (id == 0) {\n    return compose(a, b, [](int a, int b) {return b ? b : a;}, 0); //a then b, inside either\n  } else if (id == 1) {\n    return compose(a, b, [](int a, int b) {return b ? b : a;}, 1); //a then b, inside both\n  } else if (id == 2) {\n    return compose(a, b, [](int a, int b) {return b ? a : 0;}, 1); //a masked by b\n  } else if (id == 3) {\n    return compose(a, b, [](int a, int b) {return b ? b : a;}, 2); //a then b, inside of a\n  } else if (id == 4) {\n    return compose(a, b, [](int a, int b) {return b ? 0 : a;}, 2); //a masked by inverse of b, inside of a\n  } else assert(id >= 0 && id < 5);\n  return badImg;\n}\n\nImage outerProductIS(Image_ a, Image_ b) {\n  if (a.w*b.w > MAXSIDE || a.h*b.h > MAXSIDE || a.w*b.w*a.h*b.h > MAXAREA) return badImg;\n  point rpos = {a.p.x*b.w+b.p.x,\n\t\ta.p.y*b.h+b.p.y};\n  Image ret = core::empty(rpos, {a.w*b.w, a.h*b.h});\n  for (int i = 0; i < a.h; i++)\n    for (int j = 0; j < a.w; j++)\n      for (int k = 0; k < b.h; k++)\n\tfor (int l = 0; l < b.w; l++)\n\t  ret(i*b.h+k, j*b.w+l) = a(i,j) * !!b(k,l);\n  return ret;\n}\nImage outerProductSI(Image_ a, Image_ b) {\n  if (a.w*b.w > MAXSIDE || a.h*b.h > MAXSIDE || a.w*b.w*a.h*b.h > MAXAREA) return badImg;\n  point rpos = {a.p.x*b.w+b.p.x,\n\t\ta.p.y*b.h+b.p.y};\n  Image ret = core::empty(rpos, {a.w*b.w, a.h*b.h});\n  for (int i = 0; i < a.h; i++)\n    for (int j = 0; j < a.w; j++)\n      for (int k = 0; k < b.h; k++)\n\tfor (int l = 0; l < b.w; l++)\n\t  ret(i*b.h+k, j*b.w+l) = (a(i,j)>0) * b(k,l);\n  return ret;\n}\n\n\n\nImage Fill(Image_ a) {\n  Image ret = core::full(a.p, a.sz, core::majorityCol(a));\n  vector<pair<int,int>> q;\n  for (int i = 0; i < a.h; i++)\n    for (int j = 0; j < a.w; j++)\n      if ((i == 0 || j == 0 || i == a.h-1 || j == a.w-1) && !a(i,j)) {\n\tq.emplace_back(i,j);\n\tret(i,j) = 0;\n      }\n  while (q.size()) {\n    auto [r,c] = q.back();\n    q.pop_back();\n    for (int d = 0; d < 4; d++) {\n      int nr = r+(d==2)-(d==3);\n      int nc = c+(d==0)-(d==1);\n      if (nr >= 0 && nr < a.h && nc >= 0 && nc < a.w && !a(nr,nc) && ret(nr,nc)) {\n\tq.emplace_back(nr,nc);\n\tret(nr,nc) = 0;\n      }\n    }\n  }\n  return ret;\n}\n\nImage interior(Image_ a) {\n  return compose(Fill(a), a, [](int x, int y) {return y ? 0 : x;}, 0);\n}\n\nImage border(Image_ a) {\n  Image ret = core::empty(a.p, a.sz);\n  vector<pair<int,int>> q;\n  for (int i = 0; i < a.h; i++)\n    for (int j = 0; j < a.w; j++)\n      if (i == 0 || j == 0 || i == a.h-1 || j == a.w-1) {\n\tif (!a(i,j))\n\t  q.emplace_back(i,j);\n\tret(i,j) = 1;\n      }\n  while (q.size()) {\n    auto [r,c] = q.back();\n    q.pop_back();\n#define DO(nr,nc) {if (!ret(nr,nc)) { ret(nr,nc) = 1; if (!a(nr,nc)) q.emplace_back(nr,nc); }}\n    if (r > 0) {\n      if (c > 0) DO(r-1,c-1);\n      DO(r-1,c);\n      if (c+1 < a.w) DO(r-1,c+1);\n    }\n    if (r+1 < a.h) {\n      if (c > 0) DO(r+1,c-1);\n      DO(r+1,c);\n      if (c+1 < a.w) DO(r+1,c+1);\n    }\n\n    if (c > 0) DO(r,c-1);\n    if (c+1 < a.w) DO(r,c+1);\n#undef DO\n    /*for (int nr : {r-1,r,r+1}) {\n      for (int nc : {c-1,c,c+1}) {\n\tif (nr >= 0 && nr < a.h && nc >= 0 && nc < a.w && !ret(nr,nc)) {\n    if (!a(nr,nc))\n\t  q.emplace_back(nr,nc);\n\t  ret(nr,nc) = 1;\n\t}\n      }\n      }*/\n  }\n    for (int i = 0; i < a.mask.size(); i++)\n      ret.mask[i] = ret.mask[i]*a.mask[i];\n  return ret;\n}\n\n\nImage alignx(Image_ a, Image_ b, int id) {\n  assert(id >= 0 && id < 5);\n  Image ret = a;\n  if (id == 0)      ret.x = b.x-a.w;\n  else if (id == 1) ret.x = b.x;\n  else if (id == 2) ret.x = b.x+(b.w-a.w)/2;\n  else if (id == 3) ret.x = b.x+b.w-a.w;\n  else if (id == 4) ret.x = b.x+b.w;\n  return ret;\n}\nImage aligny(Image_ a, Image_ b, int id) {\n  assert(id >= 0 && id < 5);\n  Image ret = a;\n  if (id == 0)      ret.y = b.y-a.h;\n  else if (id == 1) ret.y = b.y;\n  else if (id == 2) ret.y = b.y+(b.h-a.h)/2;\n  else if (id == 3) ret.y = b.y+b.h-a.h;\n  else if (id == 4) ret.y = b.y+b.h;\n  return ret;\n}\nImage align(Image_ a, Image_ b, int idx, int idy) {\n  assert(idx >= 0 && idx < 6);\n  assert(idy >= 0 && idy < 6);\n  Image ret = a;\n  if      (idx == 0) ret.x = b.x-a.w;\n  else if (idx == 1) ret.x = b.x;\n  else if (idx == 2) ret.x = b.x+(b.w-a.w)/2;\n  else if (idx == 3) ret.x = b.x+b.w-a.w;\n  else if (idx == 4) ret.x = b.x+b.w;\n\n  if      (idy == 0) ret.y = b.y-a.h;\n  else if (idy == 1) ret.y = b.y;\n  else if (idy == 2) ret.y = b.y+(b.h-a.h)/2;\n  else if (idy == 3) ret.y = b.y+b.h-a.h;\n  else if (idy == 4) ret.y = b.y+b.h;\n  return ret;\n}\n\n\n\nImage align(Image_ a, Image_ b) {\n  //Find most matching color and align a to b using it\n  Image ret = a;\n  int match_size = 0;\n  for (int c = 1; c < 10; c++) {\n    Image ca = compress(filterCol(a, c));\n    Image cb = compress(filterCol(b, c));\n    if (ca.mask == cb.mask) {\n      int cnt = core::count(ca);\n      if (cnt > match_size) {\n\tmatch_size = cnt;\n\tret.p = a.p+cb.p-ca.p;\n      }\n    }\n  }\n  if (match_size == 0) return badImg;\n  return ret;\n}\n\nImage replaceCols(Image_ base, Image_ cols) {\n  Image ret = base;\n  Image done = core::empty(base.p,base.sz);\n  point d = base.p-cols.p;\n  for (int i = 0; i < base.h; i++) {\n    for (int j = 0; j < base.w; j++) {\n      if (!done(i,j) && base(i,j)) {\n\tint acol = base(i,j);\n\tint cnt[10] = {};\n\tvector<pair<int,int>> path;\n\tfunction<void(int,int)> dfs = [&](int r, int c) {\n\t  if (r < 0 || r >= base.h || c < 0 || c >= base.w || base(r,c) != acol || done(r,c)) return;\n\t  cnt[cols.safe(r+d.y,c+d.x)]++;\n\t  path.emplace_back(r,c);\n\t  done(r,c) = 1;\n\t  for (int nr : {r-1,r,r+1})\n\t    for (int nc : {c-1,c,c+1})\n\t      dfs(nr,nc);\n\t};\n\tdfs(i,j);\n\tpair<int,int> maj = {0,0};\n\tfor (int c = 1; c < 10; c++) {\n\t  maj = max(maj, make_pair(cnt[c], -c));\n\t}\n\tfor (auto [r,c] : path)\n\t  ret(r,c) = -maj.second;\n      }\n    }\n  }\n  return ret;\n}\n\n//Exploit symmetries?\nImage center(Image_ img) {\n  point sz = {(img.w+1)%2+1,\n\t      (img.h+1)%2+1};\n  return core::full(img.p+(img.sz-sz)/2, sz);\n}\n\nImage transform(Image_ img, int A00, int A01, int A10, int A11) {\n  if (img.w*img.h == 0) return img;\n  Image c = center(img);\n  point off = point{1-c.w,1-c.h}+(img.p-c.p)*2;\n  auto t = [&](point p) {\n    p = p*2+off;\n    p = {A00*p.x+A01*p.y,\n\t A10*p.x+A11*p.y};\n    p = p-off;\n    p.x >>= 1;\n    p.y >>= 1;\n    return p;\n  };\n  point corner[4] = {t({0,0}),t({img.w-1,0}),t({0,img.h-1}),t({img.w-1,img.h-1})};\n  point a = corner[0], b = corner[0];\n  for (int i = 1; i < 4; i++) {\n    a.x = min(a.x, corner[i].x);\n    a.y = min(a.y, corner[i].y);\n    b.x = max(b.x, corner[i].x);\n    b.y = max(b.y, corner[i].y);\n  }\n  Image ret = core::empty(img.p, b-a+point{1,1});\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      point go = t({j,i})-a;\n      ret(go.y,go.x) = img(i,j);\n    }\n  }\n  return ret;\n}\n\nint mirrorHeuristic(Image_ img) {\n  //Meant to be used for mirroring, flip either x or y, depending on center of gravity\n  int cnt = 0, sumx = 0, sumy = 0;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (img(i,j)) {\n\tcnt++;\n\tsumx += j;\n\tsumy += i;\n      }\n    }\n  }\n  return abs(sumx*2-(img.w-1)*cnt) < abs(sumy*2-(img.h-1)*cnt);\n}\n\nImage rigid(Image_ img, int id) {\n  if (id == 0) return img;\n  else if (id == 1) return transform(img, 0, 1,-1, 0); //CCW\n  else if (id == 2) return transform(img,-1, 0, 0,-1); //180\n  else if (id == 3) return transform(img, 0,-1, 1, 0); //CW\n  else if (id == 4) return transform(img,-1, 0 ,0, 1); //flip x\n  else if (id == 5) return transform(img, 1, 0, 0,-1); //flip y\n  else if (id == 6) return transform(img, 0, 1, 1, 0); //swap xy\n  else if (id == 7) return transform(img, 0,-1,-1, 0); //swap other diagonal\n  else if (id == 8) return rigid(img, 4+mirrorHeuristic(img));\n  else assert(id >= 0 && id < 9);\n  return badImg;\n}\n\nImage invert(Image img) {\n  if (img.w*img.h == 0) return img;\n  int mask = core::colMask(img);\n  int col = 1;\n  while (col < 10 && (mask>>col&1) == 0) col++;\n  if (col == 10) col = 1;\n\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      img(i,j) = img(i,j) ? 0 : col;\n  return img;\n}\n\n\nImage interior2(Image_ a) {\n  return compose(a, invert(border(a)), 2);\n}\n\n\n\nImage count(Image_ img, int id, int outType) {\n  assert(id >= 0 && id < 7);\n  assert(outType >= 0 && outType < 3);\n  int num;\n  if (id == 0)      num = core::count(img);\n  else if (id == 1) num = core::countCols(img);\n  else if (id == 2) num = core::countComponents(img);\n  else if (id == 3) num = img.w;\n  else if (id == 4) num = img.h;\n  else if (id == 5) num = max(img.w,img.h);\n  else if (id == 6) num = min(img.w,img.h);\n  else assert(0);\n\n  point sz;\n  if (outType == 0) sz = {num,num};\n  else if (outType == 1) sz = {num,1};\n  else if (outType == 2) sz = {1,num};\n  else assert(0);\n\n  if (max(sz.x,sz.y) > MAXSIDE || sz.x*sz.y > MAXAREA) return badImg;\n  return core::full(sz, core::majorityCol(img));\n}\n\n\nImage myStack(Image_ a, Image b, int orient) {\n  assert(orient >= 0 && orient <= 3);\n  b.p = a.p;\n  if (orient == 0) { //Horizontal\n    b.x += a.w;\n  } else if (orient == 1) { //Vertical\n    b.y += a.h;\n  } else if (orient == 2) { //Diagonal\n    b.x += a.w;\n    b.y += a.h;\n  } else { //Other diagonal, bottom-left / top-right\n    Image c = a;\n    c.y += b.h;\n    b.x += a.w;\n    return compose(c,b);\n  }\n  return compose(a,b);\n}\n\n\nImage wrap(Image_ line, Image_ area) {\n  if (line.w*line.h == 0 || area.w*area.h == 0) return badImg;\n  Image ans = core::empty(area.sz);\n  for (int i = 0; i < line.h; i++) {\n    for (int j = 0; j < line.w; j++) {\n      int x = j, y = i;\n      x += y/area.h * line.w;\n      y %= area.h;\n\n      y += x/area.w * line.h;\n      x %= area.w;\n\n      if (x >= 0 && y >= 0 && x < ans.w && y < ans.h)\n\tans(y,x) = line(i,j);\n    }\n  }\n  return ans;\n}\n\nImage smear(Image_ base, Image_ room, int id) {\n  assert(id >= 0 && id < 7);\n  const int arr[] = {1,2,4,8,3,12,15};\n  int mask = arr[id];\n\n  point d = room.p-base.p;\n\n  Image ret = embed(base, hull(room));\n  if (mask&1) {\n    for (int i = 0; i < ret.h; i++) {\n      char c = 0;\n      for (int j = 0; j < ret.w; j++) {\n\tif (!room(i,j)) c = 0;\n\telse if (base.safe(i+d.y,j+d.x)) c = base(i+d.y,j+d.x);\n\tif (c) ret(i,j) = c;\n      }\n    }\n  }\n\n  if (mask>>1&1) {\n    for (int i = 0; i < ret.h; i++) {\n      char c = 0;\n      for (int j = ret.w-1; j >= 0; j--) {\n\tif (!room(i,j)) c = 0;\n\telse if (base.safe(i+d.y,j+d.x)) c = base(i+d.y,j+d.x);\n\tif (c) ret(i,j) = c;\n      }\n    }\n  }\n\n  if (mask>>2&1) {\n    for (int j = 0; j < ret.w; j++) {\n      char c = 0;\n      for (int i = 0; i < ret.h; i++) {\n\tif (!room(i,j)) c = 0;\n\telse if (base.safe(i+d.y,j+d.x)) c = base(i+d.y,j+d.x);\n\tif (c) ret(i,j) = c;\n      }\n    }\n  }\n\n  if (mask>>3&1) {\n    for (int j = 0; j < ret.w; j++) {\n      char c = 0;\n      for (int i = ret.h-1; i >= 0; i--) {\n\tif (!room(i,j)) c = 0;\n\telse if (base.safe(i+d.y,j+d.x)) c = base(i+d.y,j+d.x);\n\tif (c) ret(i,j) = c;\n      }\n    }\n  }\n\n  return ret;\n}\n\n/*\nImage smear(Image_ base, int id) {\n  return smear(base, hull(base), id);\n}\n*/\n\nImage extend(Image_ img, Image_ room) {\n  if (img.w*img.h == 0) return badImg;\n  Image ret = room;\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      point p = point{j,i}+room.p-img.p;\n      p.x = clamp(p.x, 0, img.w-1);\n      p.y = clamp(p.y, 0, img.h-1);\n      ret(i,j) = img(p.y,p.x);\n    }\n  }\n  return ret;\n}\n\n\n\n\n\n\n\n\nImage pickMax(vImage_ v, const function<int(Image_)>& f) {\n  if (v.empty()) return badImg;\n  int ma = f(v[0]), maxi = 0;\n  for (int i = 1; i < v.size(); i++) {\n    int score = f(v[i]);\n    if (score > ma) {\n      ma = score;\n      maxi = i;\n    }\n  }\n  return v[maxi];\n}\n\nint maxCriterion(Image_ img, int id) {\n  assert(id >= 0 && id < 14);\n  switch (id) {\n  case 0: return  core::count(img);\n  case 1: return -core::count(img);\n  case 2: return  img.w*img.h;\n  case 3: return -img.w*img.h;\n  case 4: return core::countCols(img);\n  case 5: return -img.p.y;\n  case 6: return  img.p.y;\n  case 7: return  core::countComponents(img);\n  case 8:\n    {\n      Image comp = compress(img);\n      return comp.w*comp.h-core::count(comp);\n    }\n  case 9:\n    {\n      Image comp = compress(img);\n      return -(comp.w*comp.h-core::count(comp));\n    }\n  case 10: return  core::count(interior(img));\n  case 11: return -core::count(interior(img));\n  case 12: return -img.p.x;\n  case 13: return  img.p.x;\n  }\n  return -1;\n}\n\n\nImage pickMax(vImage_ v, int id) {\n  return pickMax(v, [id](Image_ img) {return maxCriterion(img,id);});\n}\n\nvImage cut(Image_ img, Image_ a) {\n  vector<Image> ret;\n  Image done = core::empty(img.p,img.sz);\n  point d = img.p-a.p;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (!done(i,j) && !a.safe(i+d.y,j+d.x)) {\n\tImage toadd = core::empty(img.p,img.sz);\n\tfunction<void(int,int)> dfs = [&](int r, int c) {\n\t  if (r < 0 || r >= img.h || c < 0 || c >= img.w || a.safe(r+d.y,c+d.x) || done(r,c)) return;\n\t  toadd(r,c) = img(r,c)+1;\n\t  done(r,c) = 1;\n\t  for (int nr : {r-1,r,r+1})\n\t    for (int nc : {c-1,c,c+1})\n\t      dfs(nr,nc);\n\t};\n\tdfs(i,j);\n\ttoadd = compress(toadd);\n\tfor (int i = 0; i < toadd.h; i++) {\n\t  for (int j = 0; j < toadd.w; j++) {\n\t    toadd(i,j) = max(0, toadd(i,j)-1);\n\t  }\n\t}\n\tret.push_back(toadd);\n      }\n    }\n  }\n  return ret;\n}\n\n\nvImage splitCols(Image_ img, int include0) { //include0 = 0\n  vector<Image> ret;\n  int mask = core::colMask(img);\n  for (int c = !include0; c < 10; c++) {\n    if (mask>>c&1) {\n      Image s = img;\n      for (int i = 0; i < s.h; i++)\n\tfor (int j = 0; j < s.w; j++)\n\t  s(i,j) = (s(i,j) == c ? c : 0);\n      ret.push_back(s);\n    }\n  }\n  return ret;\n}\n\nImage compose(vImage_ imgs, int id) {\n  if (imgs.empty()) return badImg;\n  Image ret = imgs[0];\n  for (int i = 1; i < imgs.size(); i++)\n    ret = compose(ret, imgs[i], id);\n  return ret;\n}\n\nvoid getRegular(vector<int>&col) {\n  int colw = col.size();\n\n  for (int w = 1; w < colw; w++) {\n    int s = -1;\n    if (colw%(w+1) == w) { //No outer border\n      s = w;\n    } else if (colw%(w+1) == 1) { //Outer border\n      s = 0;\n    }\n    if (s != -1) {\n      int ok = 1;\n      for (int i = 0; i < colw; i++) {\n\tif (col[i] != (i%(w+1) == s)) {\n\t  ok = 0;\n\t  break;\n\t}\n      }\n      if (ok) {\n\treturn;\n      }\n    }\n  }\n  fill(col.begin(), col.end(), 0);\n}\n\nImage getRegular(Image_ img) {\n  //Look for regular grid division in single color\n  Image ret = img;\n  vector<int> col(img.w,1), row(img.h,1);\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (img(i,j) != img(i,0)) row[i] = 0;\n      if (img(i,j) != img(0,j)) col[j] = 0;\n    }\n  }\n  getRegular(col);\n  getRegular(row);\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      ret(i,j) = row[i] || col[j];\n    }\n  }\n  return ret;\n}\n\n\nImage cutPickMax(Image_ a, Image_ b, int id) {\n  return pickMax(cut(a,b), id);\n}\nImage regularCutPickMax(Image_ a, int id) {\n  Image b = getRegular(a);\n  return pickMax(cut(a,b), id);\n}\nImage splitPickMax(Image_ a, int id, int include0) { //include0 = 0\n  return pickMax(splitCols(a, include0), id);\n}\n\nImage cutCompose(Image_ a, Image_ b, int id) {\n  auto v = cut(a,b);\n  for (Image& img : v) img = toOrigin(img);\n  return compose(v, id);\n}\nImage regularCutCompose(Image_ a, int id) {\n  Image b = getRegular(a);\n  auto v = cut(a,b);\n  for (Image& img : v) img = toOrigin(img);\n  return compose(v, id);\n}\nImage splitCompose(Image_ a, int id, int include0) { //include0 = 0\n  auto v = splitCols(a, include0);\n  for (Image& img : v) img = toOrigin(compress(img));\n  return compose(v, id);\n}\n\nImage cutIndex(Image_ a, Image_ b, int ind) {\n  auto v = cut(a,b);\n  if (ind < 0 || ind >= (int)v.size()) return badImg;\n  return v[ind];\n}\n\n\nvImage pickMaxes(vImage_ v, function<int(Image_)> f, int invert = 0) {\n  int n = v.size();\n  if (!n) return {};\n  vector<int> score(n);\n  int ma = -1e9;\n  for (int i = 0; i < n; i++) {\n    score[i] = f(v[i]);\n    if (!i || score[i] > ma)\n      ma = score[i];\n  }\n  vector<Image> ret_imgs;\n  for (int i = 0; i < n; i++)\n    if ((score[i] == ma) ^ invert)\n      ret_imgs.push_back(v[i]);\n  return ret_imgs;\n}\n\nvImage pickMaxes(vImage_ v, int id) {\n  return pickMaxes(v, [id](Image_ img){return maxCriterion(img, id);}, 0);\n}\nvImage pickNotMaxes(vImage_ v, int id) {\n  return pickMaxes(v, [id](Image_ img){return maxCriterion(img, id);}, 1);\n}\n\nImage cutPickMaxes(Image_ a, Image_ b, int id) {\n  return compose(pickMaxes(cut(a,b), id), 0);\n}\nImage splitPickMaxes(Image_ a, int id) {\n  return compose(pickMaxes(splitCols(a), id), 0);\n}\n\n//Return a single color\n//Cut into at least 2 pieces\n//No nested pieces\n//Must touch at least 2 opposite sides\n//Smallest piece should be as big as possible\n\nImage heuristicCut(Image_ img) {\n  int ret = core::majorityCol(img, 1);\n  int ret_score = -1;\n\n  int mask = core::colMask(img);\n  Image done = core::empty(img.p,img.sz);\n  for (int col = 0; col < 10; col++) {\n    if ((mask>>col&1) == 0) continue;\n    fill(done.mask.begin(), done.mask.end(), 0);\n    function<void(int,int)> edgy = [&](int r, int c) {\n      if (r < 0 || r >= img.h || c < 0 || c >= img.w || img(r,c) != col || done(r,c)) return;\n      done(r,c) = 1;\n      for (int nr : {r-1,r,r+1})\n\tfor (int nc : {c-1,c,c+1})\n\t  edgy(nr,nc);\n    };\n    int top = 0, bot = 0, left = 0, right = 0;\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tif (img(i,j) == col) {\n\t  if (i == 0) top = 1;\n\t  if (j == 0) left = 1;\n\t  if (i == img.h-1) bot = 1;\n\t  if (j == img.w-1) right = 1;\n\t}\n\tif ((i == 0 || j == 0 || i == img.h-1 || j == img.w-1) && img(i,j) == col && !done(i,j)) {\n\t  edgy(i,j);\n\t}\n      }\n    }\n\n    if (!(top && bot || left && right)) continue;\n\n    int score = 1e9, components = 0, nocontained = 1;\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tint cnt = 0, contained = 1;\n\tif (!done(i,j) && img(i,j) != col) {\n\t  function<void(int,int)> dfs = [&](int r, int c) {\n\t    if (r < 0 || r >= img.h || c < 0 || c >= img.w) return;\n\t    if (img(r,c) == col) {\n\t      if (done(r,c)) contained = 0;\n\t      return;\n\t    }\n\t    if (done(r,c)) return;\n\t    cnt++;\n\t    done(r,c) = 1;\n\t    for (int nr : {r-1,r,r+1})\n\t      for (int nc : {c-1,c,c+1})\n\t\tdfs(nr,nc);\n\t  };\n\t  dfs(i,j);\n\t  components++;\n\t  score = min(score, cnt);\n\t  if (contained) nocontained = 0;\n\t}\n      }\n    }\n    if (components >= 2 && nocontained && score > ret_score) {\n      ret_score = score;\n      ret = col;\n    }\n  }\n  return filterCol(img,ret);\n}\n\nvImage cut(Image_ img) {\n  return cut(img, heuristicCut(img));\n}\n\nImage cutPickMax(Image_ a, int id) {\n  return cutPickMax(a, heuristicCut(a), id);\n}\nImage cutIndex(Image_ a, int ind) {\n  return cutIndex(a, heuristicCut(a), ind);\n}\nImage cutPickMaxes(Image_ a, int id) {\n  return cutPickMaxes(a, heuristicCut(a), id);\n}\n\n\n\nImage repeat(Image_ a, Image_ b, int pad) { //pad = 0\n  if (a.w*a.h <= 0 || b.w*b.h <= 0) return badImg;\n  Image ret;\n  ret.p = b.p;\n  ret.sz = b.sz;\n  ret.mask.resize(ret.w*ret.h,0);\n\n  const int W = a.w+pad, H = a.h+pad;\n  int ai  = ((b.y-a.y)%H+H)%H;\n  int aj0 = ((b.x-a.x)%W+W)%W;\n  for (int i = 0; i < ret.h; i++) {\n    int aj = aj0;\n    for (int j = 0; j < ret.w; j++) {\n      if (ai < a.h && aj < a.w)\n\tret(i,j) = a(ai,aj);\n      if (++aj == W) aj = 0;\n    }\n    if (++ai == H) ai = 0;\n  }\n  return ret;\n}\n\nImage mirror(Image_ a, Image_ b, int pad) { //pad = 0\n  if (a.w*a.h <= 0 || b.w*b.h <= 0) return badImg;\n  Image ret;\n  ret.p = b.p;\n  ret.sz = b.sz;\n  ret.mask.resize(ret.w*ret.h);\n  const int W = a.w+pad, H = a.h+pad;\n  const int W2 = W*2, H2 = H*2;\n  int ai  = ((b.y-a.y)%H2+H2)%H2;\n  int aj0 = ((b.x-a.x)%W2+W2)%W2;\n  for (int i = 0; i < ret.h; i++) {\n    int aj = aj0;\n    for (int j = 0; j < ret.w; j++) {\n      int x = -1, y = -1;\n      if (aj < a.w) x = aj;\n      else if (aj >= W && aj < W+a.w) x = W+a.w-1-aj;\n      if (ai < a.h) y = ai;\n      else if (ai >= H && ai < H+a.h) y = H+a.h-1-ai;\n      if (x != -1 && y != -1)\n\tret(i,j) = a(y,x);\n      if (++aj == W2) aj = 0;\n    }\n    if (++ai == H2) ai = 0;\n  }\n  return ret;\n}\n\nImage majCol(Image_ img) {\n  return Col(core::majorityCol(img));\n}\n', 'vendor/icecuber/src/image_functions2.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "spec.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n\n\nvImage splitAll(Image_ img) {\n  vector<Image> ret;\n  Image done = core::empty(img.p,img.sz);\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (!done(i,j)) {\n\tImage toadd = core::empty(img.p,img.sz);\n\tfunction<void(int,int,int)> dfs = [&](int r, int c, int col) {\n\t  if (r < 0 || r >= img.h || c < 0 || c >= img.w || img(r,c) != col || done(r,c)) return;\n\t  toadd(r,c) = img(r,c)+1;\n\t  done(r,c) = 1;\n\t  for (int d = 0; d < 4; d++)\n\t    dfs(r+(d==0)-(d==1),c+(d==2)-(d==3),col);\n\t};\n\tdfs(i,j,img(i,j));\n\ttoadd = compress(toadd);\n\tfor (int i = 0; i < toadd.h; i++) {\n\t  for (int j = 0; j < toadd.w; j++) {\n\t    toadd(i,j) = max(0, toadd(i,j)-1);\n\t  }\n\t}\n\t//Image i = interior(toadd);\n\t//if (!core::count(compose(img,i,2)))\n\tif (core::count(toadd))\n\t  ret.push_back(toadd);\n      }\n    }\n  }\n  return ret;\n}\n\n\nImage eraseCol(Image img, int col) {\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      if (img(i,j) == col) img(i,j) = 0;\n  return img;\n}\n\n\n// Looks for 4 corners\nvImage insideMarked(Image_ in) {\n  vector<Image> ret;\n  for (int i = 0; i+1 < in.h; i++) {\n    for (int j = 0; j+1 < in.w; j++) {\n      for (int h = 1; i+h+1 < in.h; h++) {\n\tfor (int w = 1; j+w+1 < in.w; w++) {\n\t  char col = in(i,j);\n\t  if (!col) continue;\n\t  int ok = 1;\n\t  for (int k = 0; k < 4; k++) {\n\t    int x = j+k%2*w, y = i+k/2*h;\n\t    for (int d = 0; d < 4; d++) {\n\t      if ((d != 3-k) == (in(y+d/2,x+d%2) != col)) {\n\t\tok = 0;\n\t\tgoto fail;\n\t      }\n\t    }\n\t  }\n\tfail:\n\t  if (ok) {\n\t    Image inside = invert(core::full(point{j+1,i+1}, point{w,h}));\n\t    ret.push_back(compose(inside,in,3));\n\t  }\n\t}\n      }\n    }\n  }\n  return ret;\n}\n\nImage makeBorder(Image_ img, int bcol = 1) {\n  Image ret = hull0(img);\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      if (img(i,j) == 0) {\n\tint ok = 0;\n\tfor (int ni : {i-1,i,i+1}) {\n\t  for (int nj : {j-1,j,j+1}) {\n\t    if (img.safe(ni,nj)) {\n\t      ok = 1;\n\t      break;\n\t    }\n\t  }\n\t}\n\tif (ok) {\n\t  ret(i,j) = bcol;\n\t}\n      }\n    }\n  }\n  return ret;\n}\n\n\nImage makeBorder2(Image_ img, int usemaj = 1) {\n  int bcol = 1;\n  if (usemaj) bcol = core::majorityCol(img);\n\n  point rsz = img.sz+point{2,2};\n  if (max(rsz.x,rsz.y) > MAXSIDE || rsz.x*rsz.y > MAXAREA) return badImg;\n  Image ret = core::full(img.p-point{1,1}, rsz, bcol);\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      ret(i+1,j+1) = img(i,j);\n  return ret;\n}\n\nImage makeBorder2(Image_ img, Image_ bord) {\n  int bcol = core::majorityCol(bord);\n  point rsz = img.sz+bord.sz+bord.sz;\n  if (max(rsz.x,rsz.y) > MAXSIDE || rsz.x*rsz.y > MAXAREA) return badImg;\n  Image ret = core::full(img.p-bord.sz, rsz, bcol);\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      ret(i+bord.h,j+bord.w) = img(i,j);\n  return ret;\n}\n\n\n//Delete black rows / cols\nImage compress2(Image_ img) {\n  vector<int> row(img.h), col(img.w);\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      if (img(i,j)) row[i] = col[j] = 1;\n  vector<int> rows, cols;\n  for (int i = 0; i < img.h; i++) if (row[i]) rows.push_back(i);\n  for (int j = 0; j < img.w; j++) if (col[j]) cols.push_back(j);\n  Image ret = core::empty(point{(int)cols.size(), (int)rows.size()});\n  for (int i = 0; i < ret.h; i++)\n    for (int j = 0; j < ret.w; j++)\n      ret(i,j) = img(rows[i],cols[j]);\n  return ret;\n}\n\n\n//Group single color rectangles\nImage compress3(Image_ img) {\n  if (img.w*img.h <= 0) return badImg;\n  vector<int> row(img.h), col(img.w);\n  row[0] = col[0] = 1;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (i && img(i,j) != img(i-1,j)) row[i] = 1;\n      if (j && img(i,j) != img(i,j-1)) col[j] = 1;\n    }\n  }\n  vector<int> rows, cols;\n  for (int i = 0; i < img.h; i++) if (row[i]) rows.push_back(i);\n  for (int j = 0; j < img.w; j++) if (col[j]) cols.push_back(j);\n  Image ret = core::empty(point{(int)cols.size(), (int)rows.size()});\n  for (int i = 0; i < ret.h; i++)\n    for (int j = 0; j < ret.w; j++)\n      ret(i,j) = img(rows[i],cols[j]);\n  return ret;\n}\n\n\n//TODO: return badImg if fail\n\nImage greedyFill(Image& ret, vector<pair<int,vector<int>>>&piece, Spec&done, int bw, int bh, int&donew) {\n  sort(piece.rbegin(), piece.rend());\n\n  const int dw = ret.w-bw+1, dh = ret.h-bh+1;\n  if (dw < 1 || dh < 1) return badImg;\n\n  vector<int> dones(dw*dh, -1);\n  priority_queue<tuple<int,int,int>> pq;\n  auto recalc = [&](int i, int j) {\n    int cnt = 0;\n    for (int y = 0; y < bh; y++)\n      for (int x = 0; x < bw; x++)\n\tcnt += done(i+y,j+x);\n    if (cnt != dones[i*dw+j]) {\n      dones[i*dw+j] = cnt;\n      pq.emplace(cnt,j,i);\n    }\n  };\n  for (int i = 0; i+bh <= ret.h; i++)\n    for (int j = 0; j+bw <= ret.w; j++)\n      recalc(i,j);\n\n  while (pq.size()) {\n    auto [ds,j,i] = pq.top();\n    pq.pop();\n    if (ds != dones[i*dw+j]) continue;\n    int found = 0;\n    for (auto [cnt,mask] : piece) {\n      int ok = 1;\n      for (int y = 0; y < bh; y++)\n\tfor (int x = 0; x < bw; x++)\n\t  if (done(i+y,j+x) && ret(i+y,j+x) != mask[y*bw+x])\n\t    ok = 0;\n      if (ok) {\n\tfor (int y = 0; y < bh; y++) {\n\t  for (int x = 0; x < bw; x++) {\n\t    if (!done(i+y,j+x)) {\n\t      done(i+y,j+x) = donew;\n\t      if (donew > 1) donew--;\n\t      ret(i+y,j+x) = mask[y*bw+x];\n\t    }\n\t  }\n\t}\n\tfor (int y = max(i-bh+1,0); y < min(i+bh, dh); y++)\n\t  for (int x = max(j-bw+1,0); x < min(j+bw, dw); x++)\n\t    recalc(y,x);\n\tfound = 1;\n\tbreak;\n      }\n    }\n    if (!found)\n      return badImg;//ret;\n  }\n  return ret;\n}\n\n\n\n\nImage greedyFillBlack(Image_ img, int N = 3) {\n  Image ret = core::empty(img.p, img.sz);\n  Spec done;\n  done.sz = img.sz;\n  done.mask.assign(done.w*done.h,0);\n\n  int donew = 1e6;\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      if (img(i,j)) {\n\tret(i,j) = img(i,j);\n\tdone(i,j) = donew;\n      }\n    }\n  }\n\n  map<vector<int>,int> piece_cnt;\n  vector<int> mask;\n  const int bw = N, bh = N;\n  for (int r = 0; r < 8; r++) {\n    Image rot = rigid(img,r);\n    for (int i = 0; i+bh <= rot.h; i++) {\n      for (int j = 0; j+bw <= rot.w; j++) {\n\tmask.reserve(bw*bh);\n\tmask.resize(0);\n\tint ok = 1;\n\tfor (int y = 0; y < bh; y++)\n\t  for (int x = 0; x < bw; x++) {\n\t    char c = rot(i+y,j+x);\n\t    mask.push_back(c);\n\t    if (!c) ok = 0;\n\t  }\n\tif (ok)\n\t  ++piece_cnt[mask];\n      }\n    }\n  }\n  vector<pair<int,vector<int>>> piece;\n  for (auto&[p,c] : piece_cnt)\n    piece.emplace_back(c,p);\n\n  return greedyFill(ret, piece, done, bw, bh, donew);\n}\n\nImage greedyFillBlack2(Image_ img, int N = 3) {\n  Image filled = greedyFillBlack(img, N);\n  return compose(filled, img, 4);\n}\n\nImage extend2(Image_ img, Image_ room) {\n  Image ret = core::empty(room.p, room.sz);\n  Spec done;\n  done.sz = room.sz;\n  done.mask.assign(done.w*done.h,0);\n\n  point d = room.p-img.p;\n  int donew = 1e6;\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      int x = j+d.x, y = i+d.y;\n      if (x >= 0 && y >= 0 && x < img.w && y < img.h) {\n\tret(i,j) = img(y,x);\n\tdone(i,j) = donew;\n      }\n    }\n  }\n\n  map<vector<int>,int> piece_cnt;\n  vector<int> mask;\n  const int bw = 3, bh = 3;\n  for (int r = 0; r < 8; r++) {\n    Image rot = rigid(img,r);\n    for (int i = 0; i+bh <= rot.h; i++) {\n      for (int j = 0; j+bw <= rot.w; j++) {\n\tmask.reserve(bw*bh);\n\tmask.resize(0);\n\tfor (int y = 0; y < bh; y++)\n\t  for (int x = 0; x < bw; x++)\n\t    mask.push_back(rot(i+y,j+x));\n\t++piece_cnt[mask];\n      }\n    }\n  }\n  vector<pair<int,vector<int>>> piece;\n  for (auto&[p,c] : piece_cnt)\n    piece.emplace_back(c,p);\n\n  return greedyFill(ret, piece, done, bw, bh, donew);\n}\n\n\n\nImage connect(Image_ img, int id) {\n  assert(id >= 0 && id < 3);\n  Image ret = core::empty(img.p, img.sz);\n\n  //Horizontal\n  if (id == 0 || id == 2) {\n    for (int i = 0; i < img.h; i++) {\n      int last = -1, lastc = -1;\n      for (int j = 0; j < img.w; j++) {\n\tif (img(i,j)) {\n\t  if (img(i,j) == lastc) {\n\t    for (int k = last+1; k < j; k++)\n\t      ret(i,k) = lastc;\n\t  }\n\t  lastc = img(i,j);\n\t  last = j;\n\t}\n      }\n    }\n  }\n\n  //Vertical\n  if (id == 1 || id == 2) {\n    for (int j = 0; j < img.w; j++) {\n      int last = -1, lastc = -1;\n      for (int i = 0; i < img.h; i++) {\n\tif (img(i,j)) {\n\t  if (img(i,j) == lastc) {\n\t    for (int k = last+1; k < i; k++)\n\t      ret(k,j) = lastc;\n\t  }\n\t  lastc = img(i,j);\n\t  last = i;\n\t}\n      }\n    }\n  }\n\n  return ret;\n}\n\nImage replaceTemplate(Image_ in, Image_ need_, Image_ marked_, int overlapping = 0, int rigids = 0) {\n  if (marked_.sz != need_.sz) return badImg;\n  if (need_.w*need_.h <= 0) return in;\n\n  const int rots = rigids ? 8 : 1;\n  vector<Image> needr(rots), markedr(rots);\n  for (int r = 0; r < rots; r++) {\n    needr[r] = rigid(need_,r);\n    markedr[r] = rigid(marked_,r);\n  }\n\n  Image ret = in;\n  for (int r = 0; r < rots; r++) {\n    Image_ need = needr[r];\n    Image_ marked = markedr[r];\n\n    for (int i = 0; i+need.h <= ret.h; i++) {\n      for (int j = 0; j+need.w <= ret.w; j++) {\n\tint ok = 1;\n\tfor (int y = 0; y < need.h; y++)\n\t  for (int x = 0; x < need.w; x++)\n\t    if ((overlapping ? in : ret)(i+y,j+x) != need(y,x)) ok = 0;\n\n\tif (overlapping == 2) {\n\t  for (int y = -1; y <= need.h; y++) {\n\t    for (int x = -1; x <= need.w; x++) {\n\t      if (x >= 0 && y >= 0 && x < need.w && y < need.h) continue;\n\n\t      char nn = need(clamp(y,0,need.h-1),\n\t\t\t     clamp(x,0,need.w-1));\n\t      if (nn && nn == in.safe(i+y,j+x)) ok = 0;\n\t    }\n\t  }\n\t}\n\n\tif (ok) {\n\t  for (int y = 0; y < need.h; y++)\n\t    for (int x = 0; x < need.w; x++)\n\t      ret(i+y,j+x) = marked(y,x);\n\t}\n      }\n    }\n  }\n  return ret;\n}\n\n\n\nImage swapTemplate(Image_ in, Image_ a, Image_ b, int rigids = 0) {\n  if (a.sz != b.sz) return badImg;\n  if (a.w*a.h <= 0) return in;\n\n  const int rots = rigids ? 8 : 1;\n  vector<Image> ar(rots), br(rots);\n  for (int r = 0; r < rots; r++) {\n    ar[r] = rigid(a,r);\n    br[r] = rigid(b,r);\n  }\n  Image done = hull0(in), ret = in;\n  for (int k : {0,1}) {\n    for (int r = 0; r < rots; r++) {\n      Image_ need = k ? ar[r] : br[r];\n      Image_ to   = k ? br[r] : ar[r];\n\n      for (int i = 0; i+need.h <= ret.h; i++) {\n\tfor (int j = 0; j+need.w <= ret.w; j++) {\n\n\t  int ok = 1;\n\t  for (int y = 0; y < need.h; y++)\n\t    for (int x = 0; x < need.w; x++)\n\t      if (done(i+y,j+x) || ret(i+y,j+x) != need(y,x)) ok = 0;\n\t  if (ok) {\n\t    for (int y = 0; y < need.h; y++) {\n\t      for (int x = 0; x < need.w; x++) {\n\t\tret(i+y,j+x) = to(y,x);\n\t\tdone(i+y,j+x) = 1;\n\t      }\n\t    }\n\t  }\n\t}\n\n      }\n    }\n  }\n  return ret;\n}\n\n\nImage spreadCols(Image img, int skipmaj = 0) {\n  int skipcol = -1;\n  if (skipmaj)\n    skipcol = core::majorityCol(img);\n\n  Image done = hull0(img);\n  queue<tuple<int,int,int>> q;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (img(i,j)) {\n\tif (img(i,j) != skipcol)\n\t  q.emplace(j,i,img(i,j));\n\tdone(i,j) = 1;\n      }\n    }\n  }\n  while (q.size()) {\n    auto [j,i,c] = q.front();\n    q.pop();\n    for (int d = 0; d < 4; d++) {\n      int ni = i+(d==0)-(d==1);\n      int nj = j+(d==2)-(d==3);\n      if (ni >= 0 && nj >= 0 && ni < img.h && nj < img.w && !done(ni,nj)) {\n\timg(ni,nj) = c;\n\tdone(ni,nj) = 1;\n\tq.emplace(nj,ni,c);\n      }\n    }\n  }\n  return img;\n}\n\n\nvImage splitColumns(Image_ img) {\n  if (img.w*img.h <= 0) return {};\n  vector<Image> ret(img.w);\n  for (int j = 0; j < img.w; j++) {\n    ret[j].p = {j,0};\n    ret[j].sz = {1,img.h};\n    ret[j].mask.resize(img.h);\n    for (int i = 0; i < img.h; i++)\n      ret[j].mask[i] = img(i,j);\n  }\n  return ret;\n}\nvImage splitRows(Image_ img) {\n  if (img.w*img.h <= 0) return {};\n  vector<Image> ret(img.h);\n  for (int i = 0; i < img.h; i++) {\n    ret[i].p = {0,i};\n    ret[i].sz = {img.w,1};\n    ret[i].mask.resize(img.w);\n    for (int j = 0; j < img.w; j++)\n      ret[i].mask[j] = img(i,j);\n  }\n  return ret;\n}\n\n\nImage half(Image_ img, int id) {\n  assert(id >= 0 && id < 4);\n  if (id == 0) {\n    return core::subImage(img, {0,0}, {img.w/2, img.h});\n  } else if (id == 1) {\n    return core::subImage(img, {img.w-img.w/2,0}, {img.w/2, img.h});\n  } else if (id == 2) {\n    return core::subImage(img, {0,0}, {img.w, img.h/2});\n  } else if (id == 3) {\n    return core::subImage(img, {0,img.h-img.h/2}, {img.w, img.h/2});\n  }\n  return badImg;\n}\n\n\nImage smear(Image_ img, int id) {\n  assert(id >= 0 && id < 15);\n  const pair<int,int> R = {1,0}, L = {-1,0}, D = {0,1}, U = {0,-1};\n  const pair<int,int> X = {1,1}, Y = {-1,-1}, Z = {1,-1}, W = {-1,1};\n  const vector<pair<int,int>> d[15] = {{R},{L},{D},{U},\n\t\t\t\t       {R,L},{D,U},\n\t\t\t\t       {R,L,D,U},\n\t\t\t\t       {X},{Y},{Z},{W},\n\t\t\t\t       {X,Y},{Z,W},\n\t\t\t\t       {X,Y,Z,W},\n\t\t\t\t       {R,L,D,U,X,Y,Z,W}};\n  const int w = img.w;\n  Image ret = img;\n\n\n  for (auto [dx,dy] : d[id]) {\n    int di = dy*w+dx;\n\n    for (int i = 0; i < ret.h; i++) {\n      int step = i == 0 || i == ret.h-1 ? 1 : max(ret.w-1,1);\n      for (int j = 0; j < ret.w; j += step) {\n\tif (i-dy < 0 || j-dx < 0 || i-dy >= img.h || j-dx >= img.w) {\n\t  int steps = MAXSIDE;\n\t  if (dx ==-1) steps = min(steps, j+1);\n\t  if (dx == 1) steps = min(steps, img.w-j);\n\t  if (dy ==-1) steps = min(steps, i+1);\n\t  if (dy == 1) steps = min(steps, img.h-i);\n\n\t  int ind = i*w+j;\n\t  int end_ind = ind+steps*di;\n\t  int c = 0;\n\t  for (; ind != end_ind; ind += di) {\n\t    if (img.mask[ind]) c = img.mask[ind];\n\t    if (c) ret.mask[ind] = c;\n\t  }\n\t}\n      }\n    }\n  }\n\n  return ret;\n}\n\n\nImage mirror2(Image_ a, Image_ line) {\n  Image ret;\n  if (line.w > line.h) {\n    ret = rigid(a,5);\n    ret.x = a.x;\n    ret.y = line.y*2+line.h-a.y-a.h;\n  } else {\n    ret = rigid(a,4);\n    ret.y = a.y;\n    ret.x = line.x*2+line.w-a.x-a.w;\n  }\n  return ret;\n}\n\nvImage gravity(Image_ in, int d) {\n  vImage pieces = splitAll(in);\n  Image room = hull0(in);\n  int dx = (d==0)-(d==1);\n  int dy = (d==2)-(d==3);\n\n  vImage ret;\n  Image out = room;\n  sort(pieces.begin(), pieces.end(), [dx,dy](Image_ a, Image_ b) {\n      return (a.x-b.x)*dx+(a.y-b.y)*dy > 0;});\n  for (Image p : pieces) {\n    while (1) {\n      p.x += dx;\n      p.y += dy;\n      for (int i = 0; i < p.h; i++) {\n\tfor (int j = 0; j < p.w; j++) {\n\t  if (p(i,j) == 0) continue;\n\t  int x = j+p.x-out.x;\n\t  int y = i+p.y-out.y;\n\t  if (x < 0 || y < 0 || x >= out.w || y >= out.h || out(y,x)) {\n\t    p.x -= dx;\n\t    p.y -= dy;\n\t    goto done;\n\t  }\n\t}\n      }\n    }\n  done:\n    ret.push_back(p);\n    out = compose(out, p, 3);\n  }\n  return ret;\n}\n\n\n\nImage myStack(vImage_ lens, int id) {\n  int n = lens.size();\n  if (!n) return badImg;\n  vector<pair<int,int>> order(n);\n  for (int i = 0; i < n; i++) {\n    order[i] = {lens[i].w*lens[i].h,i};\n  }\n  sort(order.begin(), order.end());\n\n  Image out = lens[order[0].second];\n  for (int i = 1; i < n; i++)\n    out = myStack(out,lens[order[i].second],id);\n  return out;\n}\n\nImage stackLine(vImage_ shapes) {\n  int n = shapes.size();\n  if (!n) return badImg;\n  else if (n == 1) return shapes[0];\n  vector<int> xs(n), ys(n);\n  for (int i = 0; i < n; i++) {\n    xs[i] = shapes[i].x;\n    ys[i] = shapes[i].y;\n  }\n  sort(xs.begin(), xs.end());\n  sort(ys.begin(), ys.end());\n  int xmin = 1e9, ymin = 1e9;\n  for (int i = 1; i < n; i++) {\n    xmin = min(xmin, xs[i]-xs[i-1]);\n    ymin = min(ymin, ys[i]-ys[i-1]);\n  }\n  int dx = 1, dy = 0;\n  if (xmin < ymin) dx = 0, dy = 1;\n\n  vector<pair<int,int>> order(n);\n  for (int i = 0; i < shapes.size(); i++) {\n    order[i] = {shapes[i].x*dx+shapes[i].y*dy,i};\n  }\n  sort(order.begin(), order.end());\n\n  Image out = shapes[order[0].second];\n  for (int i = 1; i < n; i++)\n    out = myStack(out,shapes[order[i].second],dy);\n  return out;\n}\n\n\nImage composeGrowingSlow(vImage_ imgs) {\n  int n = imgs.size();\n  if (!n) return badImg;\n\n  vector<pair<int,int>> order(n);\n  for (int i = 0; i < n; i++) {\n    order[i] = {core::count(imgs[i]), i};\n  }\n  sort(order.rbegin(), order.rend());\n\n  Image ret = imgs[order[0].second];\n  for (int i = 1; i < n; i++)\n    ret = compose(ret, imgs[order[i].second], 0);\n  return ret;\n}\n\n\nImage composeGrowing(vImage_ imgs) {\n  int n = imgs.size();\n  if (!n) return badImg;\n  if (n == 1) return imgs[0];\n\n  int minx = 1e9, miny = 1e9, maxx = -1e9, maxy = -1e9;\n  for (Image_ img : imgs) {\n    minx = min(minx, img.x);\n    miny = min(miny, img.y);\n    maxx = max(maxx, img.x+img.w);\n    maxy = max(maxy, img.y+img.h);\n  }\n\n  point rsz = {maxx-minx, maxy-miny};\n  if (max(rsz.x, rsz.y) > MAXSIDE || rsz.x*rsz.y > MAXAREA || rsz.x <= 0 || rsz.y <= 0)\n    return badImg;\n\n  vector<pair<int,int>> order(n);\n  for (int i = 0; i < n; i++) {\n    order[i] = {core::count(imgs[i]), i};\n  }\n  sort(order.rbegin(), order.rend());\n\n  Image ret = core::empty(point{minx, miny}, rsz);\n  for (auto [cnt,imgi] : order) {\n    Image_ img = imgs[imgi];\n    int dx = img.x-ret.x, dy = img.y-ret.y;\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tif (img(i,j))\n\t  ret(i+dy,j+dx) = img(i,j);\n      }\n    }\n  }\n  //assert(ret == composeGrowingSlow(imgs));\n  return ret;\n}\n\n\nImage pickUnique(vImage_ imgs, int id) {\n  assert(id == 0);\n\n  int n = imgs.size();\n  if (!n) return badImg;\n\n  //Pick the one with the unique color\n  vector<int> mask(n);\n  vector<int> cnt(10);\n  for (int i = 0; i < n; i++) {\n    mask[i] = core::colMask(imgs[i]);\n    for (int c = 0; c < 10; c++) {\n      if (mask[i]>>c&1) cnt[c]++;\n    }\n  }\n  int reti = -1;\n  for (int i = 0; i < n; i++) {\n    for (int c = 0; c < 10; c++) {\n      if (mask[i]>>c&1) {\n\tif (cnt[c] == 1) {\n\t  if (reti == -1) reti = i;\n\t  else return badImg;\n\t}\n      }\n    }\n  }\n  if (reti == -1) return badImg;\n  return imgs[reti];\n}\n', 'vendor/icecuber/src/main.cpp': '#include "runner.hpp"\n#include <stdio.h>\n#include <stdlib.h>\n\n#include <unistd.h>\n\nint main(int argc, char**argv) {\n  int only_sid = -1;\n  if (argc >= 2) {\n    only_sid = atoi(argv[1]);\n    printf("Running only task # %d\\n", only_sid);\n  }\n  int maxdepth = -1;\n  if (argc >= 3) {\n    maxdepth = atoi(argv[2]);\n    printf("Using max depth %d\\n", maxdepth);\n  }\n  run(only_sid, maxdepth);\n}\n', 'vendor/icecuber/src/normalize.cpp': '#include "precompiled_stl.hpp"\n/*#include <cassert>\n#include <functional>\n#include <vector>\n#include <iostream>\n#include <map>\n#include <cmath>*/\nusing namespace std;\n\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "visu.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n\nvector<double> shapeFeatures(Image_ img, int col) {\n  double fill_cnt = core::count(img);\n  double comp_cnt = core::countComponents(img);\n  double int_cnt = core::count(interior(img));\n  vector<double> r =  {fill_cnt, 1./(fill_cnt+1e-3),\n\t\t       comp_cnt, 1./(comp_cnt+1e-3),\n\t\t       int_cnt, 1./(int_cnt+1e-3)};\n  for (int c = 0; c < 10; c++)\n    r.push_back((c == col)+2);\n\n  point center2 = img.p*2+img.sz-point{1,1};\n  for (int a = 0; a < 2; a++) {\n    for (int b = 0; b < 2; b++) {\n      for (int c = 0; c < 2; c++) {\n\tpoint dir;\n\tif (c) dir = point{a*2-1, b*2-1}*2;\n\telse dir = point{(a*2-1)*b, (a*2-1)*!b}*3;\n\tint ma = -50;\n\tfor (int i = 0; i < img.h; i++)\n\t  for (int j = 0; j < img.w; j++)\n\t    if (img(i,j)) ma = max(ma, (point{i*2,j*2}-center2)*dir);\n\tr.push_back(ma+1000);\n      }\n    }\n  }\n  return r;\n}\n\nstruct UniquePicker {\n  vector<int> feature_dim;\n  int save;\n  UniquePicker(const vector<Image>&ins, int save_) {\n    save = save_ | 1;\n\n    int nfeats = -1;\n    vector<vector<vector<double>>> feat; // inp, col, feat\n    for (Image_ in : ins) {\n      vector<vector<double>> features;\n      vector<pair<Image, int>> split = core::splitCols(in);\n      for (auto&[sh,col] : split) {\n\tfeatures.push_back(shapeFeatures(sh,col));\n\tnfeats = features.back().size();\n      }\n      feat.push_back(features);\n    }\n\n    int nins = ins.size();\n    vector<vector<int>> done(nins); // inp, cole\n    vector<int> cols_left(nins);\n    for (int i = 0; i < nins; i++) {\n      done[i].assign(feat[i].size(), 0);\n      cols_left[i] = feat[i].size();\n    }\n\n    if (nfeats == -1) return;\n\n    while (1) {\n      vector<double> score(nfeats, 1e3); //feat\n      vector<vector<int>> picki(nins); //inp, feat -> col\n\n      int found = 0;\n      for (int inpi = 0; inpi < nins; inpi++) {\n\tif (!cols_left[inpi]) continue;\n\tfound = 1;\n\n\tauto&f = feat[inpi];\n\tint cols = f.size();\n\tfor (int fi = 0; fi < nfeats; fi++) {\n\t  pair<double,int> best = {-1,-1};\n\t  for (int i = 0; i < cols; i++) {\n\t    if (done[inpi][i]) continue;\n\t    double worst = 1e3;\n\t    for (int j = 0; j < cols; j++) {\n\t      if (done[inpi][j] || i == j) continue;\n\t      worst = min(worst, diff(f[i][fi], f[j][fi]));\n\t    }\n\t    best = max(best, make_pair(worst, i));\n\t  }\n\t  assert(best.second != -1);\n\t  score[fi] = min(score[fi], best.first);\n\t  picki[inpi].push_back(best.second);\n\t}\n      }\n      if (!found) break;\n\n      pair<double,int> best = {-1,-1};\n      for (int fi = 0; fi < nfeats; fi++) {\n\tbest = max(best, make_pair(score[fi], fi));\n\t//cout << score[fi] << \'-\' << fi << "  ";\n      }\n      //cout << endl;\n      //cout << best.first << \'-\' << best.second << endl;\n      int pickf = best.second;\n      assert(pickf != -1);\n      feature_dim.push_back(pickf);\n      for (int inpi = 0; inpi < nins; inpi++) {\n\tif (!cols_left[inpi]) continue;\n\tauto&col = picki[inpi];\n\tdone[inpi][col[pickf]] = 1;\n\tcols_left[inpi]--;\n      }\n    }\n\n    //for (int i : feature_dim) cout << i << endl;\n    //cout << endl;\n    //exit(0);\n  }\n\n  double diff(double a, double b) const {\n    return a/(b+1e-5);\n  }\n\n  int getUnique(const vector<vector<double>>&features, const vector<int>&done, int fi) const {\n    pair<double,int> best = {-1,-1};\n    int n = features.size();\n    for (int i = 0; i < n; i++) {\n      if (done[i]) continue;\n      double score = 0;\n      for (int j = 0; j < features[i].size(); j++) {\n\tif (fi != -1) j = fi;\n\tdouble fscore = 1e3;\n\tfor (int k = 0; k < n; k++) {\n\t  if (k == i || done[k]) continue;\n\t  fscore = min(fscore, diff(features[i][j], features[k][j]));\n\t}\n\tscore = max(score, fscore);\n\tif (fi != -1) break;\n      }\n      best = max(best, make_pair(score, i));\n      //cout << score << \' \';\n    }\n    //cout << endl;\n    //exit(0);\n    assert(best.first != -1);\n    return best.second;\n  }\n\n  void getMap(Image_ in, int cols[10]) const {\n    int j = 0, done = 0;\n    for (int i = 0; i < 10; i++)\n      if (save>>i&1) {\n\tdone |= 1<<i;\n\tcols[i] = j++;\n      }\n\n    vector<vector<double>> features;\n    vector<pair<Image, int>> split = core::splitCols(in);\n    vector<int> splitcol;\n    for (auto&[sh,col] : split) {\n      if (save>>col&1) continue;\n      features.push_back(shapeFeatures(sh, col));\n      splitcol.push_back(col);\n    }\n\n    vector<int> order, fdone(features.size());\n    for (int it = 0; it < features.size(); it++) {\n      int fi = it < feature_dim.size() ? feature_dim[it] : -1;\n      int i = getUnique(features, fdone, fi);\n      fdone[i] = 1;\n      order.push_back(i);\n    }\n    for (int&i : order) i = splitcol[i];\n\n    for (int i : order) {\n      if (done>>i&1) continue;\n      cols[i] = j++;\n      done |= 1<<i;\n    }\n    for (int i = 0; i < 10; i++) {\n      if ((done>>i&1) == 0) {\n\tcols[i] = j++;\n\tdone |= 1<<i;\n      }\n    }\n    assert(done == (1<<10)-1);\n  }\n};\n\n\n\n\n\n\nImage remapCols(Image_ img, int cols[10]) {\n  Image r = img;\n  for (int i = 0; i < r.h; i++)\n    for (int j = 0; j < r.w; j++)\n      r(i,j) = cols[img(i,j)];\n  return r;\n}\n\n\n\nvoid remapCols(const vector<pair<Image,Image>>&train, vector<Simplifier>&sims) {\n  int orin = 0, orout = 0, andin = ~0, andout = ~0;\n  for (auto [in,out] : train) {\n    int maskin = core::colMask(in);\n    int maskout = core::colMask(out);\n    orin |= maskin;\n    andin &= maskin;\n    orout |= maskout;\n    andout &= maskout;\n  }\n\n  int save = andout & ~orin;\n  if (orout == andout) save = andout;\n  save |= andin&~orout;\n\n  vector<Image> train_ins;\n  for (auto&[in,out] : train) train_ins.push_back(in);\n  UniquePicker up(train_ins, save | (orout&~orin));\n\n  Simplifier ret;\n\n  ret.in = [up](Image_ in) {\n    int cols[10];\n    up.getMap(in, cols);\n    return remapCols(in, cols);\n  };\n  ret.out = [up](Image_ in, Image_ out) {\n    int cols[10];\n    up.getMap(in, cols);\n    return remapCols(out, cols);\n  };\n  ret.rec = [up](Image_ in, Image_ out) {\n    int cols[10];\n    up.getMap(in, cols);\n    int icols[10];\n    for (int i = 0; i < 10; i++)\n      icols[cols[i]] = i;\n    return remapCols(out, icols);\n  };\n\n  sims.push_back(ret);\n}\n\n\nImage listCols(Image_ img, int extra) {\n  int mask = core::colMask(img);\n  int w = __builtin_popcount(mask);\n  Image ret = core::full({w,2}, 9);\n  int j = 0;\n  for (int i = 0; i < 10; i++)\n    if ((mask>>i&1) && !(extra>>i&1))\n      ret(0,j++) = i;\n  j = 0;\n  for (int i = 0; i < 10; i++)\n    if (extra>>i&1)\n      ret(1,j++) = i;\n  return ret;\n}\n\n\n\nvoid normalizeCols(vector<Sample>&sample) {\n  Visu visu;\n\n  int count = 0;\n  for (Sample&s : sample) {\n    auto train = s.train;\n    vector<Simplifier> sims;\n    remapCols(train, sims);\n    if (sims.size()) {\n      Simplifier&sim = sims[0];\n      for (auto&[in,out] : train) {\n\tImage cp = out;\n\tout = sim.out(in, out);\n\tassert(cp == sim.rec(in, out));\n\tin = sim.in(in);\n      }\n\n      count++;\n      visu.next(s.id);\n      for (auto [in,out] : s.train)\n\tvisu.add(in,out);\n      visu.next(s.id);\n      for (auto [in,out] : train) {\n\tvisu.add(in,out);\n      }\n    }\n    continue;\n    vector<pair<int,int>> masks;\n    int orin = 0, orout = 0, andin = ~0, andout = ~0;\n    for (auto [in,out] : s.train) {\n      masks.emplace_back(core::colMask(in), core::colMask(out));\n      orin |= masks.back().first;\n      andin &= masks.back().first;\n      orout |= masks.back().second;\n      andout &= masks.back().second;\n    }\n    int eq = core::countCols(s.train[0].first);\n\n    int meaningCols = checkAll(s.train, [&](pair<Image,Image> p) {\n\treturn core::countCols(p.first) == eq;});\n    //if (__builtin_popcount(eq&~1) != 1) continue;\n\n    if (meaningCols) {\n      count++;\n      visu.next(s.id);\n      for (auto [in,out] : s.train)\n\tvisu.add(in,out);\n      visu.next(s.id);\n      for (auto [in,out] : s.train) {\n\t//visu.add(in,core::embed(core::filterCol(out,10), out.sz));\n\tvisu.add(listCols(in, andin&~orout),listCols(out, andout&~orin));\n      }\n    }\n  }\n  cout << count << " tasks" << endl;\n}\n\n\n\n\n\n\n\n\n\n\n\n\n//Needs rigid:\n//0, 2, 12, 24, 52, (63), 66, 76, (91), 96, 130, (134), 138, 176, (182)\n\nstruct OrientationPicker {\n  int feature_dim;\n  OrientationPicker(const vector<Image>&ins) {\n    feature_dim = 0;\n    double best_score = 1.5;\n\n    {\n      double score = 1e3;\n      for (Image_ in : ins) {\n\tscore = min(score, max(in.w*1./in.h, in.h*1./in.w));\n      }\n      if (score > best_score) {\n\tfeature_dim = 1;\n\tbest_score = score;\n      }\n    }\n\n    for (int c = 0; c < 10; c++) {\n      double score = 1e3;\n      for (Image_ in : ins) {\n\tauto [p,q] = inertia(in, c);\n\tdouble s = max(p*1./(q+1.), q*1./(p+1.));\n\tscore = min(score, sqrt(s));\n      }\n      if (score > best_score) {\n\tfeature_dim = 2+c;\n\tbest_score = score;\n      }\n    }\n\n    //TODO: improve heuristics for when to use\n    {\n      double scard = 1e3, sdiag = 1e3;\n      for (Image_ in : ins) {\n\tdouble x[10] = {}, y[10] = {}, cnt[10] = {};\n\tcolorMeans(in, x, y, cnt);\n\n\tdouble sx = 1e3, sy = 1e3, sxy = 1e3, syx = 1e3;\n\tint found = 0;\n\tfor (int a = 1; a < 10; a++) {\n\t  for (int b = 1; b < a; b++) {\n\t    if (cnt[a] && cnt[b]) {\n\t      double dx = x[a]-x[b];\n\t      double dy = y[a]-y[b];\n\t      sx = min(sx, abs(dx)/(abs(dy)+1e-1));\n\t      sy = min(sy, abs(dy)/(abs(dx)+1e-1));\n\t      sxy = min(sxy, abs(dx+dy)/(abs(dx-dy)+1e-1));\n\t      syx = min(syx, abs(dx-dy)/(abs(dx+dy)+1e-1));\n\t      found = 1;\n\t      goto outside;\n\t    }\n\t  }\n\t}\n      outside:\n\tif (!found) scard = sdiag = -1;\n\tscard = min(scard, max(sx, sy));\n\tsdiag = min(sdiag, max(sxy, syx));\n      }\n      scard /= 5., sdiag /= 5.;\n      if (scard > best_score) {\n\tbest_score = scard;\n\tfeature_dim = 12;\n      }\n      if (sdiag > best_score) {\n\tbest_score = sdiag;\n\tfeature_dim = 13;\n      }\n    }\n  }\n\n\n\n  pair<double,double> inertia(Image_ img, char c) const {\n    double x = 0, y = 0, xx = 0, yy = 0, cnt = 0;\n    for (int i = 0; i < img.h; i++)\n      for (int j = 0; j < img.w; j++)\n\tif (img(i,j) == c) {\n\t  x += j, xx += j*j;\n\t  y += i, yy += i*i;\n\t  cnt++;\n\t}\n    if (cnt < 4) return {0.,0.};\n    return {(xx*cnt-x*x), (yy*cnt-y*y)};\n  }\n\n  void colorMeans(Image_ in, double*x, double*y, double*cnt) const {\n    for (int i = 0; i < in.h; i++)\n      for (int j = 0; j < in.w; j++) {\n\tchar c = in(i,j);\n\tx[c] += j;\n\ty[c] += i;\n\tcnt[c]++;\n      }\n\n    for (int c = 0; c < 10; c++) {\n      if (cnt[c]) {\n\tx[c] /= cnt[c];\n\ty[c] /= cnt[c];\n      }\n    }\n  }\n\n  int getRigid(Image_ in) const {\n    if (feature_dim == 0) return 0;\n    else if (feature_dim == 1) return in.h > in.w ? 6 : 0;\n    else if (feature_dim >= 2 && feature_dim < 12) {\n      auto [p,q] = inertia(in, feature_dim-2);\n      return p > q ? 6 : 0;\n    } else if (feature_dim == 12 || feature_dim == 13) {\n\n      double x[10] = {}, y[10] = {}, cnt[10] = {};\n      colorMeans(in, x, y, cnt);\n\n      for (int a = 1; a < 10; a++)\n\tfor (int b = 1; b < a; b++) {\n\t  if (cnt[a] && cnt[b]) {\n\t    double dx = x[a]-x[b];\n\t    double dy = y[a]-y[b];\n\t    if (feature_dim == 12) {\n\t      if (abs(dx) > abs(dy))\n\t\treturn dx < 0 ? 4 : 0;\n\t      else\n\t\treturn dy < 0 ? 7 : 6;\n\t    } else {\n\t      if (abs(dx+dy) > abs(dx-dy))\n\t\treturn dx+dy < 0 ? 7 : 0;\n\t      else\n\t\treturn dx-dy < 0 ? 4 : 5;\n\t    }\n\t  }\n\t}\n      return 0;\n    }\n    else assert(0);\n\n\n    //Criteria:\n    //Size should be horizontal\n    //Colored lines should be vertical\n    //2 colors should be top-left and bottom-right diagonally\n    //2 colors should be left and right horizontally\n\n    //Preserve:\n    //Things touching one side / corner\n    //Colored lines in output\n\n  }\n};\n\nvoid normalizeRigid(const vector<pair<Image,Image>>&train, vector<Simplifier>&sims) {\n  Simplifier ret;\n\n  vector<Image> train_ins;\n  for (auto&[in,out] : train) train_ins.push_back(in);\n  OrientationPicker up(train_ins);\n\n  ret.in = [up](Image_ in) {\n    int rid = up.getRigid(in);\n    return toOrigin(rigid(in, rid));\n  };\n  ret.out = [up](Image_ in, Image_ out) {\n    int rid = up.getRigid(in);\n    return toOrigin(rigid(out, rid));\n  };\n  ret.rec = [up](Image_ in, Image_ out) {\n    int rid = up.getRigid(in);\n    int inv[] = {0,3,2,1,4,5,6,7};\n    return toOrigin(rigid(out, inv[rid]));\n  };\n\n  sims.push_back(ret);\n}\n\n\n\npair<Image,Image> Simplifier::operator()(Image_ a, Image_ b) {\n  return {in(a), out(a,b)};\n}\n\nSimplifier normalizeCols(const vector<pair<Image,Image>>&train) {\n  vector<Simplifier> sims;\n  remapCols(train, sims);\n  return sims[0];\n}\n\nSimplifier normalizeDummy(const vector<pair<Image,Image>>&train) {\n  Simplifier ret;\n  ret.in = [](Image_ in) { return in; };\n  ret.out = [](Image_ in, Image_ out) { return out; };\n  ret.rec = [](Image_ in, Image_ out) { return out; };\n  return ret;\n}\n', 'vendor/icecuber/src/pieces.cpp': '#include "precompiled_stl.hpp"\n#include <chrono>\nusing namespace std;\n#include "utils.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "image_functions2.hpp"\n\n#include "visu.hpp"\n\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "timer.hpp"\n/*\nstruct Piece2 {\n  vector<int> ind;\n  int depth;\n};\n\nstruct Pieces {\n  vector<DAG> dag;\n  vector<Piece2> piece;\n  set<vector<int>> seen;\n};\n*/\n\nextern int print_nodes;\n\null hashVec(const vector<int>&vec) {\n  ull r = 1;\n  for (int v : vec) {\n    r = r*1069388789821391921+v;\n  }\n  return r;\n}\n\nPieces makePieces2(vector<DAG>&dag, vector<pair<Image,Image>> train, vector<point> out_sizes) {\n  Timer set_time, piece_time, child_time;\n\n  Pieces pieces;\n\n  vector<int>&mem = pieces.mem;\n  vector<int> depth_mem;\n\n  int dags = dag.size();\n\n  TinyHashMap seen;\n  //set<vector<int>> seen;\n  vector<queue<int>> q;\n\n  auto add = [&](int d, const vector<int>&v) {\n    assert(v.size() == dags);\n\n    set_time.start();\n    auto [memi, inserted] = seen.insert(hashVec(v), (int)mem.size());\n    set_time.stop();\n\n    if (inserted) {\n      for (int i : v) {\n\tmem.push_back(i);\n      }\n      depth_mem.push_back(d);\n    }\n\n    if (inserted || depth_mem[memi/dags] > d) {\n      depth_mem[memi/dags] = d;\n      while (q.size() <= d) q.push_back({});\n      q[d].push(memi);\n    }\n  };\n  for (int i = 0; i < dag[0].givens; i++) {\n    vector<int> v(dags,i);\n    add(dag[0].tiny_node[i].depth, v);\n  }\n\n  /*\n  for (int fi2 = 0; fi2 < dag[0].binary.size(); fi2++) {\n    vector<int> v;\n    v.reserve(dags);\n    int ok = 1, depth = -1;\n    for (DAG&d : dag) {\n      int ni = d.binary[fi2];\n      if (ni == -1) {\n\tok = 0;\n\tbreak;\n      }\n      assert(ni >= 0 && ni < d.tiny_node.size());\n      depth = max(depth, (int)d.tiny_node[ni].depth);\n      v.push_back(ni);\n    }\n    if (ok) {\n      pushQueue(depth, mem.size());\n      add(v, false);\n    }\n  }\n  */\n\n  vector<vector<pair<int,int>>> slow_child(train.size()+1);\n  vector<pair<int,vector<int>>> newi_list;\n\n  piece_time.start();\n  for (int depth = 0; depth < q.size(); depth++) {\n    while (q[depth].size()) {\n      int memi = q[depth].front();\n      q[depth].pop();\n      if (depth > depth_mem[memi/dags]) continue;\n      assert(depth == depth_mem[memi/dags]);\n\n      vector<int> ind(mem.begin()+memi, mem.begin()+memi+dags);\n      {\n\tint ok = 1, maxdepth = -1;\n\tfor (int i = 0; i < dags; i++) {\n\t  maxdepth = max(maxdepth, (int)dag[i].tiny_node[ind[i]].depth);\n\t  ok &= dag[i].tiny_node[ind[i]].ispiece;\n\t}\n\tif (ok && maxdepth >= depth) {\n\t  Piece3 p;\n\t  p.memi = memi;\n\t  p.depth = depth;\n\t  pieces.piece.push_back(p);\n\t}\n\tif (maxdepth < depth) continue;\n      }\n\n      newi_list.clear();\n\n      child_time.start();\n      {\n\tfor (int i = 0; i <= train.size(); i++)\n\t  dag[i].tiny_node[ind[i]].child.legacy(slow_child[i]);\n\tvector<int> newi(dags), ci(dags); //current index into child[]\n\tint fi = 0;\n\twhile (1) {\n\tnext_iteration:\n\t  for (int i = 0; i <= train.size(); i++) {\n\t    auto&child = slow_child[i];//dag[i].node[ind[i]].child;\n\t    while (ci[i] < child.size() &&\n\t\t   child[ci[i]].first < fi) ci[i]++;\n\t    if (ci[i] == child.size()) goto finish;\n\n\t    int next_available_fi = child[ci[i]].first;\n\t    if (next_available_fi > fi) {\n\t      fi = next_available_fi;\n\t      goto next_iteration;\n\n\t    } else {\n\n\t      newi[i] = child[ci[i]].second;\n\n\t      if (newi[i] == -1) {\n\t\tfi++;\n\t\tgoto next_iteration;\n\t      }\n\t    }\n\t  }\n\t  newi_list.emplace_back(fi, newi);\n\t  fi++;\n\t}\n      finish:;\n      }\n      child_time.stop();\n\n      for (auto&[fi, newi] : newi_list) {\n\tif (0) {\n\t  int i = train.size();\n\t  //auto&child = dag[i].node[ind[i]].child;\n\t  //auto it = lower_bound(child.begin(), child.end(), make_pair(fi, -1));\n\t  //if (it == child.end() || it->first != fi) {\n\t  int to = dag[i].tiny_node.getChild(ind[i], fi);\n\t  if (to == TinyChildren::None) {\n\t    string name = dag[i].funcs.getName(fi);\n\t    if (name.substr(0,4) == "Move") {\n\t      newi[i] = dag[i].applyFunc(ind[i], fi);\n\t      if (newi[i] != -1 && out_sizes.size())\n\t\tdag[i].applyFunc(newi[i], dag[i].funcs.findfi("embed 1"));\n\t    } else continue;\n\t  } else {\n\t    newi[i] = to; //it->second\n\t  }\n\t  if (newi[i] == -1) continue;\n\t}\n\n\tint new_depth = -1;\n\tfor (int i = 0; i < dags; i++) {\n\t  new_depth = max(new_depth, (int)dag[i].tiny_node[newi[i]].depth);\n\t}\n\n\tint cost = dag[0].funcs.cost[fi];\n\n\tif (new_depth >= depth+cost) {\n\t  add(depth+cost, newi);\n\t}\n      }\n    }\n  }\n  piece_time.stop();\n  piece_time.print("Piece loop time");\n  set_time.print("Set time");\n  child_time.print("Child looking time");\n\n\n  auto lookFor = [&](vector<string> name_list) {\n    vector<Image> look_imgs;\n    vector<int> look_v;\n\n    int di = 0;\n    for (DAG&d : dag) {\n      int p = 0;\n      for (string name : name_list) {\n\tint fi = d.funcs.findfi(name);\n\n\t//auto&child = d.node[p].child;\n\t//auto it = lower_bound(child.begin(), child.end(), make_pair(fi,-1));\n\t//assert(it != child.end());\n\tint ret = d.tiny_node.getChild(p, fi);\n\tassert(ret >= 0);\n\t  //auto [fi_, ret] = *it;\n\tif (0) {//fi != fi_) {\n\t  cout << p << \' \' << di << " / " << train.size() << endl;\n\t  /*for (auto [fi,nxti] : child) {\n\t    string name = d.funcs.getName(fi);\n\t    if (name.substr(0,4) == "Move") {\n\t      cout << name << \' \';\n\t    }\n\t  }\n\t  cout << endl;*/\n\t}\n\t//assert(fi == fi_);\n\tp = ret;\n\tassert(p != -1);\n      }\n      look_v.push_back(p);\n      look_imgs.push_back(d.getImg(p));\n      /*if (n.isvec || n.img[0].sz != given_sizes[di][1]) {\n\tcout << "Bad" << endl;\n\t}*/\n      di++;\n    }\n    if (!seen.insert(hashVec(look_v),0).second) cout << "Found indices" << endl;\n    //exit(0);\n  };\n\n  /*if (out_sizes.size()) {\n    lookFor({"compress", "toOrigin"});\n    }*/\n    /*\n    lookFor({"repeat 0 1", "colShape 2", "Move -1 -1", "embed 1"});\n    lookFor({"repeat 0 1", "colShape 2", "Move 1 1", "embed 1"});\n    lookFor({"repeat 0 1", "smear 4", "colShape 1"});\n    }*/\n  /*lookFor({"filterCol 1", "interior", "colShape 2"});\n    lookFor({"cut", "pickNotMaxes 11", "composeGrowing", "colShape 3", "embed 1"});\n    lookFor({"filterCol 5", "makeBorder", "colShape 1", "embed 1"});\n    lookFor({});\n  */\n\n  if (out_sizes.size() && print_nodes) {\n    int nodes = 0;\n    for (DAG&d : dag) nodes += d.tiny_node.size();\n    cout << "Nodes:  " << nodes << endl;\n    cout << "Pieces: " << pieces.piece.size() << endl;\n  }\n  pieces.dag = move(dag);\n  //pieces.seen = move(seen);\n\n  for (Piece3&p : pieces.piece) {\n    for (int i = 0; i < pieces.dag.size(); i++) {\n      int*ind = &mem[p.memi];\n      assert(ind[i] >= 0 && ind[i] < pieces.dag[i].tiny_node.size());\n    }\n  }\n\n  return pieces;\n}\n', 'vendor/icecuber/src/read.cpp': '/*#include <vector>\n#include <regex>\n#include <cassert>\n#include <iostream>*/\n#include "precompiled_stl.hpp"\n#include <experimental/filesystem>\n\nusing namespace std;\n#include "utils.hpp"\n#include "read.hpp"\n\nSample::Sample(string filename) {\n  vector<Image> train_input, train_output;\n  vector<Image> test_input, test_output;\n\n  regex re(".*/([a-z0-9]{8}).json");\n  smatch match;\n  assert(std::regex_search(filename, match, re) && match.size() == 2);\n  id = match.str(1);\n\n  fp = fopen(filename.c_str(), "r");\n  assert(fp);\n  expect(\'{\');\n  while (1) {\n    string train_test = getQuote();\n    expect(\'[\');\n    while (1) {\n      expect(\'{\');\n      while (1) {\n\tstring input_output = getQuote();\n\tImage img = readImage();\n\tif (train_test == "train" && input_output == "input") {\n\t  train_input.push_back(img);\n\t} else if (train_test == "train" && input_output == "output") {\n\t  train_output.push_back(img);\n\t} else if (train_test == "test" && input_output == "input") {\n\t  test_input.push_back(img);\n\t} else if (train_test == "test" && input_output == "output") {\n\t  test_output.push_back(img);\n\t} else {\n\t  cerr << train_test << \' \' << input_output << endl;\n\t  assert(!"Unexpected tag");\n\t}\n\tif (end(\'}\')) break;\n      }\n      if (end(\']\')) break;\n    }\n    if (end(\'}\')) break;\n  }\n  assert(mygetc() == -1);\n  assert(feof(fp));\n  fclose(fp);\n\n  assert(train_input.size() == train_output.size());\n  for (int i = 0; i < (int)train_input.size(); i++) {\n    train.emplace_back(train_input[i], train_output[i]);\n  }\n  if (test_input.size() == test_output.size()) {\n    for (int i = 0; i < (int)test_input.size(); i++) {\n      test.emplace_back(test_input[i], test_output[i]);\n    }\n  } else {\n    assert(test_output.empty());\n    for (int i = 0; i < (int)test_input.size(); i++) {\n      test.emplace_back(test_input[i], badImg);\n    }\n  }\n}\n\nvector<Sample> Sample::split() {\n  vector<Sample> ret;\n  for (int i = 0; i < test.size(); i++) {\n    ret.push_back(*this);\n    Sample&s = ret.back();\n    tie(s.test_in, s.test_out) = test[i];\n    s.test = {{s.test_in, s.test_out}};\n    s.id_ind = i;\n  }\n  return ret;\n}\n\nchar Sample::mygetc() {\n  char c = fgetc(fp);\n  while (c == \' \' || c == \'\\n\') c = fgetc(fp);\n  return c;\n}\nint Sample::end(char endc) {\n  char c = mygetc();\n  if (c == \',\') {\n    return 0;\n  } else {\n    if (c != endc)\n      cerr << "|"<<c<<"|" << " != " << "|"<<endc<<"|" << endl;\n    assert(c == endc);\n    return 1;\n  }\n}\nvoid Sample::expect(char c) {\n  char got = mygetc();\n  if (got != c)\n    cerr << "|"<<got<<"|" << " != " << "|"<<c<<"|" << endl;\n  assert(got == c);\n}\nstring Sample::getQuote() {\n  expect(\'\\"\');\n  char str[200];\n  assert(fscanf(fp, "%[^\\"]", str));\n  expect(\'\\"\');\n  expect(\':\');\n  return string(str);\n}\nvector<int> Sample::readRow() {\n  vector<int> ret;\n  expect(\'[\');\n  while (1) {\n    int v;\n    assert(fscanf(fp, "%d", &v) == 1);\n    ret.push_back(v);\n    if (end(\']\')) break;\n  }\n  return ret;\n}\nImage Sample::readImage() {\n  Image ret;\n  ret.p = {0,0};\n  expect(\'[\');\n  vector<int> widths;\n  while (1) {\n    vector<int> row = readRow();\n    widths.push_back(row.size());\n    for (int col : row)\n      ret.mask.push_back(col);\n    if (end(\']\')) break;\n  }\n  ret.h = widths.size();\n  assert(ret.h >= 1 && ret.h <= 30);\n  ret.w = widths[0];\n  for (int i = 0; i < ret.h; i++)\n    assert(widths[i] == ret.w);\n  assert(ret.w >= 1 && ret.w <= 30);\n  return ret;\n}\n\n\nvector<Sample> readAll(string path, int maxn) { //maxn = -1\n  const string base_path = "./dataset/";\n  assert(experimental::filesystem::exists(base_path+path));\n\n  vector<string> files;\n  for (auto magic_file_type : experimental::filesystem::directory_iterator(base_path+path)) {\n    string name = magic_file_type.path().u8string();\n    if (name.size() >= 5 && name.substr(name.size()-5,5) == ".json")\n      files.push_back(name);\n  }\n  sort(files.begin(), files.end());\n  if (maxn >= 0 && maxn < files.size()) files.resize(maxn);\n  vector<Sample> sample;\n  for (string sid : files) {\n    //sample.push_back(Sample(sid));\n    for (Sample s : Sample(sid).split()) {\n      //if (maxn < 0 || sample.size() < maxn)\n      sample.push_back(s);\n    }\n  }\n  return sample;\n}\n\n\nWriter::Writer(string filename) { //filename = "submission_part.csv"\n  filename = filename;\n  fp = fopen(filename.c_str(), "w");\n  assert(fp);\n  fprintf(fp, "output_id,output\\n");\n}\n\nvoid Writer::operator()(const Sample& s, vector<Image> imgs) {\n  //int cnt = seen[s.id]++;\n  //assert(cnt == s.id_ind);\n  fprintf(fp, "%s_%d,", s.id.c_str(), s.id_ind);\n  //TODO: is empty output allowed? Looks like no!\n  if (imgs.empty()) imgs = {dummyImg};\n  assert(imgs.size() <= 3);\n  int notfirst = 0;\n  for (Image_ img : imgs) {\n    assert(img.p == point({0,0}));\n    assert(img.w >= 1 && img.w <= 30 && img.h >= 1 && img.h <= 30);\n    if (notfirst++) fprintf(fp, " ");\n    fprintf(fp, "|");\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tint c = img(i,j);\n\tassert(c >= 0 && c <= 9);\n\tfprintf(fp, "%d", c);\n      }\n      fprintf(fp, "|");\n    }\n  }\n  fprintf(fp, "\\n");\n}\n\nWriter::~Writer() {\n  fclose(fp);\n}\n\n\nvoid writeAnswersWithScores(const Sample&s, string fn, vector<Image> imgs, vector<double> scores) {\n  FILE*fp = fopen(fn.c_str(), "w");\n  assert(fp);\n  fprintf(fp, "%s_%d\\n", s.id.c_str(), s.id_ind);\n  assert(imgs.size() == scores.size());\n  if (imgs.empty()) imgs = {dummyImg}, scores = {-1};\n  assert(imgs.size() <= 3);\n\n  for (int i = 0; i < imgs.size(); i++) {\n    Image_ img = imgs[i];\n    double score = scores[i];\n    assert(img.p == point({0,0}));\n    assert(img.w >= 1 && img.w <= 30 && img.h >= 1 && img.h <= 30);\n    fprintf(fp, "|");\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tint c = img(i,j);\n\tassert(c >= 0 && c <= 9);\n\tfprintf(fp, "%d", c);\n      }\n      fprintf(fp, "|");\n    }\n    fprintf(fp, " %.20f", score);\n    fprintf(fp, "\\n");\n  }\n  fclose(fp);\n}\n', 'vendor/icecuber/src/runner.cpp': '// Adapted from Johan S. Wind\'s generic DSL runner, MIT license; see ../LICENSE\n// and ../PROVENANCE.md. Historical task-specific/evaluation entry points are\n// excluded. This runner never reads a test output or uses a task identifier.\n#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "compose2.hpp"\n#include "deduce_op.hpp"\n#include "brute_size.hpp"\n\nint MAXDEPTH = -1;\nint MAXSIDE = 100, MAXAREA = 40*40, MAXPIXELS = 40*40*5;\nint print_times = 0, print_mem = 0, print_nodes = 0;\n\nstatic bool validAnswer(Image_ img) {\n  if (img.w < 1 || img.w > 30 || img.h < 1 || img.h > 30) return false;\n  if (img.mask.size() != img.w*img.h) return false;\n  for (char c : img.mask) if (c < 0 || c > 9) return false;\n  return true;\n}\n\nvoid run(int only_sid, int depth) {\n  assert(depth >= 1 && depth <= 3);\n  MAXDEPTH = depth*10;\n  vector<Sample> samples = readAll("test", -1);\n  assert(only_sid >= 0 && only_sid < samples.size());\n  const Sample& s = samples[only_sid];\n  Simplifier sim = normalizeCols(s.train);\n  vector<pair<Image,Image>> train;\n  for (auto& [in,out] : s.train) train.push_back(sim(in,out));\n  // Only the test INPUT enters the search. No dummy output is required.\n  Image test_in = sim.in(s.test_in);\n\n  int maxarea = 1;\n  for (auto& [in,out] : s.train)\n    maxarea = max({maxarea, in.w*in.h, out.w*out.h});\n  MAXAREA = maxarea*2;\n  MAXPIXELS = MAXAREA*5;\n\n  vector<point> out_sizes = bruteSize(test_in, train);\n  vector<DAG> dags = brutePieces2(test_in, train, out_sizes);\n  Pieces pieces = makePieces2(dags, train, out_sizes);\n  for (DAG& d : pieces.dag) {\n    d.hashi.clear();\n    for (TinyNode& n : d.tiny_node.node) n.child.clear();\n  }\n  vector<Candidate> cands = composePieces2(pieces, train, out_sizes);\n  addDeduceOuterProduct(pieces, train, cands);\n  cands = evaluateCands(cands, train);\n  sort(cands.begin(), cands.end());\n\n  const string filename = "output/answer_"+to_string(only_sid)+"_"+to_string(depth)+".csv";\n  FILE* fp = fopen(filename.c_str(), "w");\n  assert(fp);\n  fprintf(fp, "%s_%d\\n", s.id.c_str(), s.id_ind);\n  set<ull> seen;\n  int emitted = 0;\n  for (const Candidate& cand : cands) {\n    Image answer = sim.rec(s.test_in, cand.imgs.back());\n    if (!validAnswer(answer) || !seen.insert(hashImage(answer)).second) continue;\n    // Export actual matches on ORIGINAL demonstrations, rather than infer them\n    // from the upstream normalized-score-minus-complexity expression.\n    int matches = 0;\n    for (int i = 0; i < s.train.size(); i++) {\n      Image reconstructed = sim.rec(s.train[i].first, cand.imgs[i]);\n      if (reconstructed.sz == s.train[i].second.sz &&\n          reconstructed.mask == s.train[i].second.mask) matches++;\n    }\n    fprintf(fp, "|");\n    for (int r = 0; r < answer.h; r++) {\n      for (int c = 0; c < answer.w; c++) fprintf(fp, "%d", answer(r,c));\n      fprintf(fp, "|");\n    }\n    fprintf(fp, " %.20f %d\\n", cand.score, matches);\n    fflush(fp);\n    if (++emitted >= 3) break;\n  }\n  fclose(fp);\n}\n', 'vendor/icecuber/src/visu.cpp': '/*#include "stdio.h"\n#include <vector>\n#include <string>\n#include <tuple>\n#include <cassert>\n#include <functional>*/\n#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "visu.hpp"\n\nVisu::Visu() {\n  fp = fopen("visu.txt", "w");\n}\nVisu::~Visu() {\n  fclose(fp);\n}\nvoid Visu::next(string s) {\n  fprintf(fp, "Task %s\\n", s.c_str());\n}\nvoid Visu::add(Image in, Image out) {\n  fprintf(fp, "Pair\\n");\n  for (Image_ img : {in,out}) {\n    fprintf(fp, "Image %d %d\\n", img.w, img.h);\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tint col = img(i,j);\n\tfprintf(fp, "%d", col);\n      }\n      fprintf(fp, "\\n");\n    }\n  }\n}\n\n\nvoid plot(const vector<vector<int>>&inp, const char*filename) { //filename = out.ppm\n  int h = inp.size();\n  int w = inp[0].size();\n\n  int tw = 512/max(w,h);\n  int bw = max(1, 10/max(w,h));\n\n  int W = (tw+bw)*w+bw, H = (tw+bw)*h+bw;\n  FILE*fp = fopen(filename, "w");\n  fprintf(fp, "P6\\n%d %d\\n255\\n", W, H);\n  vector<int> cols = {0x000000, 0x0074D9, 0xFF4136, 0x2ECC40, 0xFFDC00, 0xAAAAAA, 0xF012BE, 0xFF851B, 0x7FDBFF, 0x870C25};\n\n  vector<unsigned char> output(W*H*3, 0x60);\n  for (int i = 0; i < h; i++) {\n    for (int j = 0; j < w; j++) {\n      for (int k = 0; k < tw; k++) {\n\tfor (int l = 0; l < tw; l++) {\n\t  for (int c = 0; c < 3; c++) {\n\t    output[((i*(tw+bw)+bw+k)*W+\n\t\t    (j*(tw+bw)+bw+l))*3 + c] = cols[inp[i][j]] >> (2-c)*8 & 255;\n\t  }\n\t}\n      }\n    }\n  }\n  fwrite(output.data(), 1, W*H*3, fp);\n  fclose(fp);\n}\n\nvoid print(Image img) {\n  printf("[%d %d %d %d]\\n", img.p.x, img.p.y, img.w, img.h);\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      int col = img(i,j);\n      if (col)\n\tprintf("%d", col);\n      else printf(".");\n    }\n    printf("\\n");\n  }\n}\n'}
for relative, contents in PACKAGE_FILES.items():
    destination = RUNTIME_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents, encoding="utf-8")
print("Embedded source files written:", len(PACKAGE_FILES))


In [ ]:
# You may set ARC2_CHALLENGES to an explicit local challenges file.
# Automatic discovery accepts only the competition's test-challenges filenames.
explicit_path = os.environ.get("ARC2_CHALLENGES")
if explicit_path:
    CHALLENGES = Path(explicit_path)
    if not CHALLENGES.is_file():
        raise FileNotFoundError(CHALLENGES)
else:
    roots = [Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2"),
             Path("/kaggle/input/arc-prize-2026-arc-agi-2")]
    names = ("arc-agi_test_challenges.json", "arc_agi_test_challenges.json")
    candidates = sorted({(root / name).resolve() for root in roots for name in names if (root / name).is_file()})
    if not candidates and Path("/kaggle/input").is_dir():
        candidates = sorted({p.resolve() for name in names for p in Path("/kaggle/input").rglob(name)
                             if "arc-prize-2026-arc-agi-2" in p.parts})
    if len(candidates) != 1:
        raise RuntimeError("Expected one ARC-AGI-2 test-challenges file; found " + str(candidates) +
                           ". Attach the competition data or set ARC2_CHALLENGES explicitly.")
    CHALLENGES = candidates[0]

OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
SUBMISSION = OUTPUT_ROOT / "submission.json"
DIAGNOSTICS = OUTPUT_ROOT / "arc2_diagnostics.json"
SECONDS_PER_TASK = float(os.environ.get("ARC2_SECONDS_PER_TASK", "8.0"))
TOTAL_SECONDS = 11 * 60 * 60
MAX_STATES = int(os.environ.get("ARC2_MAX_STATES", "4000"))
BEAM_WIDTH = int(os.environ.get("ARC2_BEAM_WIDTH", "32"))
MAX_DEPTH = int(os.environ.get("ARC2_MAX_DEPTH", "3"))
BACKEND = os.environ.get("ARC2_BACKEND", "hybrid")  # "python" disables C++ search.
CPP_FRACTION = float(os.environ.get("ARC2_CPP_FRACTION", "0.6"))
CPP_MEMORY_MB = int(os.environ.get("ARC2_CPP_MEMORY_MB", "768"))
COMPILE_SECONDS = 180.0
print("Challenges:", CHALLENGES)
print("Total wall budget:", TOTAL_SECONDS, "seconds")


In [ ]:
# Prepare the optional C++ backend once, before any per-task time budget.
# The full source and upstream MIT license are embedded above; no download occurs.
sys.path.insert(0, str(RUNTIME_ROOT))
from arc2.runner import prepare_requested_backend, atomic_json, challenge_for_solver, fallback_attempts
challenges = json.loads(CHALLENGES.read_text(encoding="utf-8"))
initial = {task_id: fallback_attempts(challenge_for_solver(task)) for task_id, task in challenges.items()}
atomic_json(SUBMISSION, initial)
BACKEND_PREPARATION = prepare_requested_backend(BACKEND, COMPILE_SECONDS)
EFFECTIVE_BACKEND = BACKEND_PREPARATION["effective"]
print("Backend preparation:", BACKEND_PREPARATION)
command = [sys.executable, "-m", "arc2", "solve",
           "--challenges", str(CHALLENGES), "--output", str(SUBMISSION),
           "--diagnostics", str(DIAGNOSTICS),
           "--seconds-per-task", str(SECONDS_PER_TASK),
           "--total-seconds", str(TOTAL_SECONDS),
           "--max-states", str(MAX_STATES), "--beam-width", str(BEAM_WIDTH),
           "--max-depth", str(MAX_DEPTH), "--backend", EFFECTIVE_BACKEND,
           "--cpp-fraction", str(CPP_FRACTION), "--cpp-memory-mb", str(CPP_MEMORY_MB),
           "--compile-seconds", str(COMPILE_SECONDS)]
subprocess.run(command, cwd=RUNTIME_ROOT, check=True)
from arc2.runner import validate_submission
challenges = json.loads(CHALLENGES.read_text(encoding="utf-8"))
submission = json.loads(SUBMISSION.read_text(encoding="utf-8"))
assert validate_submission(submission, challenges), "Submission schema validation failed"
diagnostics = json.loads(DIAGNOSTICS.read_text(encoding="utf-8"))
diagnostics["notebook_backend_preparation"] = BACKEND_PREPARATION
atomic_json(DIAGNOSTICS, diagnostics)
print("Ready:", SUBMISSION, "tasks:", len(submission), "test outputs:", sum(map(len, submission.values())))
